# FactIndic — The whole project in one notebook (01–06)

**Run on:** Kaggle (Settings → Accelerator **GPU T4 x2**, Internet **On**, Secret `HF_TOKEN`) or Colab (T4).
**Inputs:** the `factindic_inputs` dataset (H1 results and training-pool scores; data only) · **Rough T4 time:** stops by itself after 4.5 h of work, then runs the analysis

Use **Save Version → Save & Run All (Commit)**; the page can be closed. Part 1 builds the data (notebook 01), Part 2 tests every step on 4 articles and prints `SMOKE TEST: …` (about 40 min after the start), Part 3 runs 01's scores and 02–06. Failed jobs are retried and skipped, optional ones are dropped if time runs short, and the analysis always runs.

Every step writes JSONL incrementally and skips finished work, so after a disconnect just *Run All* again
(on Kaggle, attach this notebook's previous output as an input first).

## Setup

In [ ]:
import os
SRC = "/tmp/factindic_src"
os.makedirs(f"{SRC}/factindic/pipeline", exist_ok=True)

In [ ]:
%%writefile /tmp/factindic_src/factindic/requirements.txt
# Main environment for notebooks 01-07 (Python 3.11+; checked against Kaggle/Colab Python 3.13).
# transformers / huggingface_hub are the versions proven in the H1 run. torch, numpy, pandas, pyarrow,
# scipy, scikit-learn, statsmodels, matplotlib and protobuf come from the Kaggle/Colab image on purpose:
# replacing them under a running kernel breaks compiled packages.
transformers==5.24.0
huggingface_hub==1.33.0
tokenizers==0.23.3
safetensors==0.8.0
accelerate==1.16.0
peft==0.22.0
datasets==4.8.5
bitsandbytes==0.50.2
sentencepiece==0.2.3
indic-nlp-library==0.92
indic-transliteration==2.3.82
rouge-score==0.1.2
rapidfuzz==3.14.6
wordfreq==3.1.1
nltk==3.9.1
tabulate==0.9.0

In [ ]:
%%writefile /tmp/factindic_src/factindic/requirements-it2.txt
# IndicTrans2 helper environment (Python 3.11). Full lock, installed with --no-deps into its own venv.
# transformers 4.53.2 is the version pinned upstream by AI4Bharat/IndicTrans2; resolved as of 2026-10-01.
alabaster==1.0.0
babel==2.18.0
certifi==2026.7.22
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
colorama==0.4.6
cython==3.3.0
docutils==0.22.4
filelock==4.0.8
fsspec==2026.9.0
hf-xet==1.6.0
huggingface-hub==0.36.2
idna==3.20
imagesize==2.0.1
indic-nlp-library-itt==0.1.1
indictranstoolkit==1.1.1
jinja2==3.1.6
joblib==1.6.0
lxml==6.1.3
markupsafe==3.0.3
morfessor==2.0.6
mpmath==1.3.0
networkx==3.6.1
numpy==2.2.6
nvidia-cublas-cu12==12.4.5.8
nvidia-cuda-cupti-cu12==12.4.127
nvidia-cuda-nvrtc-cu12==12.4.127
nvidia-cuda-runtime-cu12==12.4.127
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.2.1.3
nvidia-curand-cu12==10.3.5.147
nvidia-cusolver-cu12==11.6.1.9
nvidia-cusparse-cu12==12.3.1.170
nvidia-cusparselt-cu12==0.6.2
nvidia-nccl-cu12==2.21.5
nvidia-nvjitlink-cu12==12.4.127
nvidia-nvtx-cu12==12.4.127
packaging==26.3
pandas==2.2.3
portalocker==4.4.0
pygments==2.21.0
python-dateutil==2.9.0.post0
pytz==2026.4
pyyaml==6.0.3
regex==2026.9.29
requests==2.34.2
roman-numerals==4.1.0
sacrebleu==2.6.0
sacremoses==0.2.0
safetensors==0.8.0
sentencepiece==0.2.1
six==1.17.0
snowballstemmer==3.1.1
sphinx==9.0.4
sphinx-argparse==0.6.1
sphinx-rtd-theme==3.1.0
sphinxcontrib-applehelp==2.0.0
sphinxcontrib-devhelp==2.0.0
sphinxcontrib-htmlhelp==2.1.0
sphinxcontrib-jquery==4.1
sphinxcontrib-jsmath==1.0.1
sphinxcontrib-qthelp==2.0.0
sphinxcontrib-serializinghtml==2.0.0
sympy==1.13.1
tabulate==0.10.0
tokenizers==0.21.4
torch==2.6.0
tqdm==4.70.1
transformers==4.53.2
triton==3.2.0
typing-extensions==4.16.0
tzdata==2026.4
urllib3==2.8.0
setuptools==80.9.0

In [ ]:
%%writefile /tmp/factindic_src/factindic/h1_pilot_ids.txt
040111_tonyblair_india
040316_iraq_survey
040408_iraq_hostages_us
040625_iraq_reaction
050129_china_taiwan
050718_bjp_nationalexe
050726_chappell_opener
051202_rti_amendment
060412_rajkumar_died
060927_prabhakaran_talks
070203_entertainment_films
070209_airstrike_iraq
071029_mukesh_profile
081219_zardari_kabul_as
120802_testicles_dope_js
121001_twenty20_westindies_newzealand_va
121108_international_australia_arm
130304_raghuraj_kunda_ms
130328_nelson_mandela_sick_rd
130822_syria_force_france_fma
131018_ayodhya_vhp_rt
131103_rare_solar_eclipse_rns
131201_karan_with_salman_sk
140205_sex_education_pakistan_aa
140313_aviation_disasters_tk
140410_action_against_manvendra_singh_rns
140423_tamilnadu_election_freebies_rd
140901_sonam_kapoor_sb
140924_us_fight_isis_pk
150204_russian_child_heart_transplant_tk
150218_doctors_antibiotics_ps
150618_narendra_modi_app_an
160228_budget_no_relief_tax_pm
160331_t20_cricket_india_westindies_social_aj
160821_flood_up_mp_dil
india-38782852
india-40986964
india-41592998
india-41770828
india-42121748
india-42136012
india-42393578
india-42601260
india-43123803
india-43561148
india-43650852
india-43765542
india-44572963
india-44766351
india-44949199
india-46678047
india-46906065
india-47366348
india-47648333
india-48305397
india-49099600
india-49142752
india-49161316
india-49673185
india-50529269
india-50828527
india-50923324
india-51266469
india-51527168
india-51885327
india-52576552
india-52605183
india-52953689
india-54580895
india-55486014
india-55599108
india-56493823
india-56672496
india-56943102
india-56951006
india-57334201
international-40903051
international-41145646
international-41376647
international-41479820
international-42656648
international-42688768
international-43848089
international-44097773
international-44385907
international-44528611
international-45019913
international-45437537
international-46326885
international-46778408
international-48952337
international-51493052
international-51701209
international-54705728
international-54813156
international-55101527
international-56146088
international-57067109
media-50010748
sport-40099255

In [ ]:
!pip install -q -r /tmp/factindic_src/factindic/requirements.txt

## Configuration and shared helpers

Paths, model list, sizes, checkpointed JSONL files, multi-GPU workers, bootstrap and regression tests.

### `factindic/__init__.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/__init__.py
"""FactIndic: factuality-aware summarization for Hindi, Gujarati, Hinglish and Gujlish."""
__version__ = "1.0.0"

### `factindic/config.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/config.py
"""Paths, model inventory, budgets, and session setup for Kaggle / Colab T4 runs."""
from __future__ import annotations

import glob
import json
import os
import random
import shutil
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
IS_LOCAL = not IS_KAGGLE and not IN_COLAB      # a laptop with one 8 GB GPU that has native bf16 (RTX 40/50)
REPO = Path(__file__).resolve().parent.parent


def _root() -> Path:
    if os.environ.get("FACTINDIC_ROOT"):
        return Path(os.environ["FACTINDIC_ROOT"])
    if IS_KAGGLE:
        return Path("/kaggle/working/factindic")
    if IN_COLAB:
        return Path("/content/drive/MyDrive/factindic")
    return REPO / "runs" / "factindic"


ROOT = _root()
DIRS = {k: ROOT / k for k in
        ["data", "outputs", "verify", "eval", "figures", "annotation", "models", "cache", "logs"]}
H1_DIR = ROOT.parent / "factindic_h1"
DATA_VERSION = 3                          # bump when the data files of notebook 01 must be rebuilt
VERSION_FILE = DIRS["data"] / "VERSION"


def data_version(root=None):
    f = Path(root) / "data" / "VERSION" if root else VERSION_FILE
    try:
        return int(f.read_text())
    except (OSError, ValueError):
        return 0


def h1_pilot_ids():
    return set(Path(__file__).with_name("h1_pilot_ids.txt").read_text().split())

LANGS = {
    "hi": dict(name="Hindi", script="Devanagari", xlsum="hindi", ilsum="Hindi", it2="hin_Deva",
               block=(0x0900, 0x097F)),
    "gu": dict(name="Gujarati", script="Gujarati", xlsum="gujarati", ilsum="Gujarati", it2="guj_Gujr",
               block=(0x0A80, 0x0AFF)),
}
VARIANTS = ["native", "script_mixed", "roman"]
SEEDS = [42, 108, 2026]

MODELS = {
    # generators
    "gemma": "google/gemma-4-E2B-it",           # backbone chosen by the H1 gate
    "llama": "meta-llama/Llama-3.2-3B-Instruct",
    "sarvam": "sarvamai/sarvam-1",
    "indicbartss": "ai4bharat/IndicBARTSS",
    "mt5": "google/mt5-base",
    # stage 1
    "indicner": "ai4bharat/IndicNER",
    "latin_ner": "Babelscape/wikineural-multilingual-ner",
    # stage 4
    "it2": "ai4bharat/indictrans2-indic-en-dist-200M",
    "nllb": "facebook/nllb-200-distilled-600M",
    "qwen": "Qwen/Qwen2.5-1.5B-Instruct",
    "minicheck": "lytang/MiniCheck-Flan-T5-Large",
    "mdeberta": "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    # held-out evaluators (never used for training weights or reranking)
    "factcg": "yaxili96/FactCG-DeBERTa-v3-Large",
    "indicbert": "ai4bharat/IndicBERTv2-MLM-only",
}

# Default budgets sized for a free T4 (Kaggle: 30 GPU-h/week; T4 x2 is used when present).
CFG = dict(
    seed=42,
    n_test_per_lang=150,          # controlled parallel testbed (300 articles; cut from 250 to fit the GPU budget)
    n_dev_per_lang=50,            # dev split for tuning (100 articles)
    # articles per language actually run (balanced over the entity-density tertiles); None = all
    n_use=dict(testbed=60, dev=24, natural_cm=40),
    n_natural_cm_per_lang=100,    # ILSUM natural code-mixed subset
    # spec: CMI >= 8. Too few clean articles reach it, so each language uses the highest step with enough of them
    cmi_natural=[8.0, 6.0, 5.0, 4.0, 3.0],
    cmi_max_run=5,                # longer runs of Latin words are captions, tweets or headlines, not mixing
    max_article_words=350,
    n_weighted_per_lang=4000,     # sanitized training pool (spec §9.3); seq2seq baselines train on all of it
    n_score_per_lang=2000,        # pool subset that gets mFACT faithfulness scores (Gemma SFT samples from it)
    n_sft_per_lang=600,           # subset of the weighted pool used for Gemma LoRA SFT (one T4 session)
    w_min=0.30,
    sft=dict(max_len=1024, micro_bs=2, accum=8, epochs=1, lr=2e-4, r=16, alpha=32, dropout=0.05),
    # Llama-3.2 spends ~10 tokens per Gujarati word: 44 of 100 H1 summaries needed more than 160 tokens
    gen=dict(batch_size=8, max_new_tokens=160, max_new_tokens_by_model={"llama": 320}, latency_probe_n=15),
    mt_backend="it2",             # "it2" (IndicTrans2, isolated environment) or "nllb"; never mixed silently
    cad=dict(alpha_base=[0.2, 0.4, 0.6], alpha_anchor=[0.6, 0.9, 1.2], beta=[0.0, 0.2, 0.4],
             eta=0.1, static_alpha=0.5, tune_docs=20),
    n_decoding_ablation_per_lang=40,    # Exp 4a-4d run on 80 articles x 3 variants
    K=5, sample_T=0.7, sample_top_p=0.9,
    n_cove=60,                     # CoVe self-rewrite baseline (Exp 5b) on a subset
    tau_ver=0.5, lambda_len=0.2,
    n_annot_summaries=60, n_annot_dev=20,
    n_naturalness_per_lang=50,
    bootstrap=1000,
    n_s2s_train=4000, s2s_epochs=2,      # training pairs and epochs of the two seq2seq baselines
    n_tune=4,                            # decoding settings tried on dev
)
if IS_LOCAL:
    CFG["sft"].update(micro_bs=1, accum=16)          # same effective batch of 16
    CFG["gen"]["batch_size_by_model"] = {"llama": 4}
CFG.update(json.loads(os.environ.get("FACTINDIC_CFG", "{}")))       # overrides for the smoke test

SYSTEMS = {
    # name: (description, experiment id)
    "indicbartss": ("IndicBARTSS fine-tuned, beam 4", "Exp 1a"),
    "mt5": ("mT5-base fine-tuned, beam 4", "Exp 1b"),
    "gemma_A": ("Gemma-4 zero-shot, Cond A standard prompt", "Exp 2b"),
    "gemma_B1": ("Gemma-4 zero-shot, Cond B1 + anchor checklist", "Exp 3a"),
    "gemma_B2": ("Gemma-4 zero-shot, Cond B2 + 2-step CoD", "Exp 3b"),
    "gemma_B3": ("Gemma-4 zero-shot, Cond B3 + matrix-verb hint", "Exp 3c"),
    "gemma_sft": ("Gemma-4 LoRA-SFT (faithfulness-weighted), greedy", "Exp 2a"),
    "llama_A": ("Llama-3.2-3B zero-shot, Cond A", "Exp 2c"),
    "dec_greedy": ("Stage-2 best + plain greedy", "Exp 4-ref"),
    "dec_static_cad": ("Static CAD", "Exp 4a"),
    "dec_adacad": ("AdaCAD (token-level JSD alpha)", "Exp 4b"),
    "dec_dola": ("DoLa (dynamic JSD premature layer)", "Exp 4c"),
    "dec_anchor_cad_dola": ("Anchor-aware dynamic CAD + DoLa", "Exp 4d"),
    "drop_only": ("Greedy + drop every unverified sentence", "Exp 5a"),
    "cove": ("Chain-of-Verification self-rewrite", "Exp 5b"),
    "factindic": ("Full pipeline: Stage 1-3 + K=5 router rerank + prune", "Exp 6"),
}


def ensure_dirs():
    for d in DIRS.values():
        d.mkdir(parents=True, exist_ok=True)


def restore_inputs():
    """Kaggle: merge outputs of earlier notebooks attached as inputs into /kaggle/working."""
    if not IS_KAGGLE:
        return []
    restored = []

    def copy(src, dst):              # keep whichever file has more finished work
        if not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst):
            shutil.copy2(src, dst)
        return dst

    for name in ("factindic", "factindic_h1"):
        found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        for src in sorted(found, key=data_version, reverse=True):
            # only real run folders (never a copy of the code package, which is also named factindic)
            is_run = any(os.path.isdir(os.path.join(src, d)) for d in ("data", "outputs", "eval")) or \
                os.path.exists(os.path.join(src, "pilot.jsonl"))
            if os.path.isdir(src) and is_run:
                if name == "factindic" and data_version(src) < data_version():
                    print(f"ignored (older data version): {src}")
                    continue
                shutil.copytree(src, ROOT.parent / name, dirs_exist_ok=True, copy_function=copy)
                restored.append(src)
    return restored


def hf_login():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:                          # `hf auth login` on a laptop
        from huggingface_hub import get_token
        tok = get_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        from huggingface_hub import login
        login(token=tok)
    else:
        print("WARNING: HF_TOKEN not set; gated models will fail.")
    return bool(tok)


def safe_attention():
    """On the RTX 5050 (torch 2.8, CUDA 12.8) PyTorch's memory-efficient attention kernel silently gives wrong
    results for Gemma-4 when the sequence length is 32k + 1 (513, 673, 1089: logits off by 10-30). The math
    kernel agrees with eager attention at every length."""
    if IS_LOCAL:
        import torch
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_cudnn_sdp(False)


def max_new_tokens(model_key):
    g = CFG["gen"]
    return g["max_new_tokens_by_model"].get(model_key, g["max_new_tokens"])


def seed_everything(seed=42):
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass


def setup(need_gpu=True):
    if IN_COLAB and not os.environ.get("FACTINDIC_ROOT"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.environ["FACTINDIC_ROOT"] = str(ROOT)   # parallel worker processes inherit the same root
    restored = restore_inputs()
    ensure_dirs()
    h1 = Path(__file__).with_name("h1_results")          # the finished H1 run, for its figures
    if not (H1_DIR / "h1_tests.json").exists() and h1.exists():
        shutil.copytree(h1, H1_DIR, dirs_exist_ok=True)
    hf_login()
    seed_everything(CFG["seed"])
    n_gpu = 0
    try:
        import torch
        n_gpu = torch.cuda.device_count()
        if need_gpu:
            assert n_gpu > 0, "No GPU visible (Kaggle: GPU T4 x2; Colab: T4; laptop: python run_local.py doctor)."
        # asking for the name opens a CUDA context in this process, which an 8 GB card cannot spare
        names = [f"{n_gpu} visible"] if IS_LOCAL else [torch.cuda.get_device_name(i) for i in range(n_gpu)]
    except ImportError:
        names = []
    info = dict(root=str(ROOT), restored=restored, gpus=names)
    print(json.dumps(info, indent=2))
    return info

### `factindic/io.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/io.py
"""JSONL checkpointing, resumable job helpers, GPU hygiene, timing."""
from __future__ import annotations

import gc
import hashlib
import json
import time
from contextlib import contextmanager
from pathlib import Path


def jl_read(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # truncated last line after a disconnect
    return rows


def jl_read_glob(path_pattern):
    """Read `name.jsonl` plus shard files `name.shard*.jsonl` written by parallel workers."""
    p = Path(path_pattern)
    rows = jl_read(p)
    for sh in sorted(p.parent.glob(p.stem + ".shard*.jsonl")):
        rows += jl_read(sh)
    return rows


def jl_write(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp.replace(path)


class JsonlAppender:
    """Append rows with flush; `done` holds keys already present so reruns skip finished work."""

    def __init__(self, path, key_fn, shard=None):
        base = Path(path)
        base.parent.mkdir(parents=True, exist_ok=True)
        # parallel workers each append to their own shard file; `done` spans base + all shards
        self.path = base if shard is None else base.with_name(f"{base.stem}.shard{shard}.jsonl")
        self.key_fn = key_fn
        self.done = {key_fn(r) for r in jl_read_glob(base)}
        self._f = open(self.path, "a", encoding="utf-8")

    def add(self, row):
        self._f.write(json.dumps(row, ensure_ascii=False) + "\n")
        self._f.flush()
        self.done.add(self.key_fn(row))

    def close(self):
        self._f.close()

    def __enter__(self):
        return self

    def __exit__(self, *a):
        self.close()


def json_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)


def json_load(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def md5(*parts):
    return hashlib.md5("\x1f".join(map(str, parts)).encode("utf-8")).hexdigest()


def free_gpu():
    """Call after `del model` in the caller (a callee cannot drop the caller's references)."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    except ImportError:
        pass


@contextmanager
def cuda_timer(out: dict, key="seconds"):
    import torch
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t = time.perf_counter()
    yield
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    out[key] = time.perf_counter() - t


def peak_vram_gb():
    import torch
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def shard(items, i, n):
    return [x for j, x in enumerate(items) if j % n == i]


def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

### `factindic/parallel.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/parallel.py
"""Run a sharded job on every visible GPU (Kaggle T4 x2 → 2 processes), else in-process.

A job is a function `f(shard: int, n_shards: int, **kwargs)` that writes its own shard file through
`JsonlAppender(..., shard=shard)`. Free GPU memory in the notebook kernel before launching.
"""
from __future__ import annotations

import importlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path


def _resolve(path):
    mod, fn = path.split(":")
    return getattr(importlib.import_module(mod), fn)


def run(func_path, n_gpus=None, **kwargs):
    import torch
    n = n_gpus or max(1, torch.cuda.device_count())
    if n == 1:
        return _resolve(func_path)(shard=0, n_shards=1, **kwargs)
    from .config import DIRS
    DIRS["logs"].mkdir(parents=True, exist_ok=True)
    tag = func_path.replace(":", ".") + f".{int(time.time())}"
    job = DIRS["logs"] / f"{tag}.json"
    job.write_text(json.dumps(kwargs, ensure_ascii=False, default=str))
    pkg_root = str(Path(__file__).resolve().parent.parent)
    procs, logs = [], []
    for i in range(n):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i),
                   PYTHONPATH=pkg_root + os.pathsep + os.environ.get("PYTHONPATH", ""))
        log = DIRS["logs"] / f"{tag}.gpu{i}.log"
        logs.append(log)
        procs.append(subprocess.Popen([sys.executable, "-u", "-m", "factindic.parallel", func_path,
                                       str(job), str(i), str(n)], env=env,
                                      stdout=open(log, "w"), stderr=subprocess.STDOUT))
    print(f"launched {n} workers for {func_path}; logs: {DIRS['logs']}/{tag}.gpu*.log")
    last, shown = 0.0, {}
    while any(p.poll() is None for p in procs):
        time.sleep(15)
        if time.time() - last > 120:
            last = time.time()
            for i, log in enumerate(logs):
                tail = log.read_text(errors="ignore").strip().splitlines()[-1:] if log.exists() else []
                if tail and tail != shown.get(i):
                    shown[i] = tail
                    print(f"  gpu{i}: {tail[0][:160]}")
    bad = [i for i, p in enumerate(procs) if p.returncode != 0]
    if bad:
        for i in bad:
            print(f"--- gpu{i} log tail ---\n" + "\n".join(logs[i].read_text(errors='ignore').splitlines()[-40:]))
        raise RuntimeError(f"workers {bad} failed for {func_path}")
    return None


if __name__ == "__main__":
    func_path, job, i, n = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
    kwargs = json.loads(Path(job).read_text())
    from factindic.config import safe_attention
    safe_attention()
    _resolve(func_path)(shard=i, n_shards=n, **kwargs)

### `factindic/stats.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/stats.py
"""Paired bootstrap CIs, fixed-effects regressions with article-clustered SEs, agreement stats."""
from __future__ import annotations

import numpy as np
import pandas as pd


def paired_bootstrap(a, b, B=1000, seed=42):
    """Mean of (a − b) over paired units, 95% percentile CI, two-sided bootstrap p."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b))
    d = a[ok] - b[ok]
    if len(d) < 5:
        return dict(n=int(len(d)), diff=float("nan"), lo=float("nan"), hi=float("nan"), p=float("nan"))
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), (B, len(d)))].mean(1)
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return dict(n=int(len(d)), diff=float(d.mean()), lo=float(np.percentile(boots, 2.5)),
                hi=float(np.percentile(boots, 97.5)), p=float(min(1.0, p)))


def bootstrap_mean(x, B=1000, seed=42):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return dict(mean=float("nan"), lo=float("nan"), hi=float("nan"), n=0)
    rng = np.random.default_rng(seed)
    boots = x[rng.integers(0, len(x), (B, len(x)))].mean(1)
    return dict(mean=float(x.mean()), lo=float(np.percentile(boots, 2.5)),
                hi=float(np.percentile(boots, 97.5)), n=int(len(x)))


def fe_ols(df, formula, term, cluster="uid"):
    """OLS with fixed effects in the formula; SEs clustered by article. Returns β, SE, p (1- and 2-sided)."""
    import statsmodels.formula.api as smf
    try:
        cols = [c for c in df.columns if c in formula]
        d = df.dropna(subset=cols)
        m = smf.ols(formula, data=d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d[cluster])[0]})
        b, p2 = float(m.params[term]), float(m.pvalues[term])
        return dict(beta=b, se=float(m.bse[term]), p_two=p2, p_one_pos=p2 / 2 if b > 0 else 1 - p2 / 2,
                    p_one_neg=p2 / 2 if b < 0 else 1 - p2 / 2, n=int(m.nobs))
    except Exception as e:
        return dict(error=repr(e)[:200])


def cohen_kappa(a, b):
    from sklearn.metrics import cohen_kappa_score
    a, b = list(a), list(b)
    return float(cohen_kappa_score(a, b)) if len(set(a) | set(b)) > 1 else float("nan")


def macro_f1(y, yhat):
    from sklearn.metrics import f1_score
    return float(f1_score(y, yhat, average="macro"))

## Text processing: Hindi, Gujarati, Romanized and code-mixed

Cleaning, sentence splitting, Romanization and script mixing, word-level language ID and CMI, the number/date/entity normalizer with the anchor set A_core, and named entities.

### `factindic/text.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/text.py
"""Unicode-safe text helpers. Python's \\W matches Indic vowel signs, so punctuation is explicit."""
from __future__ import annotations

import html
import re
import unicodedata

from .config import LANGS

PUNCT = set("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~") | set("।॥“”‘’…–—₹«»·•")
INDIC_RE = re.compile(r"[ऀ-ॿ઀-૿]")
LATIN_RE = re.compile(r"[A-Za-z]")


def strip_punct(tok):
    i, j = 0, len(tok)
    while i < j and tok[i] in PUNCT:
        i += 1
    while j > i and tok[j - 1] in PUNCT:
        j -= 1
    return tok[:i], tok[i:j], tok[j:]


def words(text):
    return [w for w in text.split() if strip_punct(w)[1]]


def W(text):
    return len(words(text))


_NORMALIZERS = {}


def clean(text, lang):
    """NFC + indic_nlp normalization (nukta/vowel canonicalization) + whitespace collapse."""
    if lang not in _NORMALIZERS:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
        _NORMALIZERS[lang] = IndicNormalizerFactory().get_normalizer(lang)
    text = unicodedata.normalize("NFC", text or "")
    text = _NORMALIZERS[lang].normalize(text)
    return re.sub(r"\s+", " ", text).strip()


_BBC_FOOTER = re.compile(
    r"(\(बीबीसी (हिन्दी|हिंदी) के एंड्रॉएड ऐप.*$)|(\(બીબીસી ગુજરાતીની.*$)|"
    r"(आप हमें फ़ेसबुक.*$)|(તમે અમને ફેસબુક.*$)|(બીબીસી ગુજરાતીને.*$)", re.S)


def strip_boilerplate(text):
    return _BBC_FOOTER.sub("", text).strip()


_CODE = re.compile(r"/\*\s*\.jw-|\.jw-reset|comscore_jw|jwplayer\(|\bvar [A-Za-z_]\w*\s*=|function\s*\w*\s*\(|"
                   r"document\.(?:get|create|write)|googletag|\$\([\"'#.]")
_MARKUP = re.compile(r"<!--.*?-->|</?[A-Za-z][^<>]{0,300}>", re.S)
_LINK = re.compile(r"(?:https?://|www\.|pic\.twitter\.com/|t\.co/)[^\s\u0900-\u0AFF]*")


def strip_web(text):
    """Remove what ILSUM's crawler kept of the web page: player and gallery scripts, tags, HTML
    entities, links, and image credits. Scripts run from the first code word to the last '}' or ');'."""
    t = html.unescape(html.unescape(text or ""))
    m = _CODE.search(t)
    if m:
        end = max(t.rfind("}") + 1, t.rfind(");") + 2, m.end())
        t = t[:m.start()] + " " + t[end:]
    t = _LINK.sub(" ", _MARKUP.sub(" ", t))
    t = re.sub(r"Image\s*Source\s*:?", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def truncate_words(text, n):
    return " ".join(text.split()[:n])


# Letter names that spell initials (एस.पी., એલ.સી.બી) and abbreviations whose full stop ends no sentence.
INITIALS = set("""ए बी सी डी ई एफ जी एच आई जे के एल एम एन ओ पी क्यू आर एस टी यू वी डब्ल्यू एक्स वाई जेड
એ બી સી ડી ઈ એફ જી એચ આઈ આઇ જે કે એલ એમ એન ઓ પી ક્યુ ક્યૂ આર એસ ટી યુ યૂ વી ડબલ્યુ ડબલ્યૂ એક્સ વાય ઝેડ""".split())
_WORDLIKE = set("ए ई ओ जी आई એ ઈ ઓ જી".split())          # also ordinary words, so they may end a sentence
_ABBR = set("डॉ डा रु रू प्रो पं मो ડૉ ડો ડા રૂ રુ પ્રો મો Dr Mr Mrs Ms Prof Rs St vs".split())
_INDIC = "\u0900-\u0963\u0971-\u0A7F\u0A80-\u0AFF"          # no dandas
_GLUED = re.compile(rf"(?<=[{_INDIC}][.।!?])(?=[{_INDIC}A-Za-z\"'“‘(])")


def _abbrev_end(part):
    """True if the final '.' of `part` belongs to an abbreviation, initials or a list number."""
    if not part.endswith("."):
        return False
    w = part.split()[-1].lstrip("\"'“‘(")[:-1]
    if w in _ABBR or re.fullmatch(r"\d{1,2}", part[:-1].strip()):
        return True
    segs = w.split(".")
    return segs[-1] in INITIALS and (len(segs) > 1 or w not in _WORDLIKE)


def split_sentences(text):
    """Splits after . । ! ? — also when the next word is glued on (છે.પાણી), and never inside
    abbreviations. Only chooses boundaries; the characters are not changed."""
    out, join = [], ""
    for part in re.split(r"(?<=[।.!?])\s+", (text or "").strip()):
        pieces, start = [], 0
        for m in _GLUED.finditer(part):
            if not _abbrev_end(part[start:m.start()]):
                pieces.append(part[start:m.start()])
                start = m.start()
        pieces.append(part[start:])
        for i, p in enumerate(pieces):
            p = f"{join} {p}".strip() if join and i == 0 else p.strip()
            join = ""
            if i == len(pieces) - 1 and p and _abbrev_end(p):
                join = p
            elif p:
                out.append(p)
    if join:
        out.append(join)
    return out


def script_fraction(text, lang):
    lo, hi = LANGS[lang]["block"]
    letters = [c for c in text if c.isalpha() or unicodedata.category(c).startswith("M")]
    if not letters:
        return 0.0
    return sum(lo <= ord(c) <= hi for c in letters) / len(letters)


def chunk_words(text, size, stride):
    w = text.split()
    if len(w) <= size:
        return [" ".join(w)]
    out = [" ".join(w[i:i + size]) for i in range(0, max(1, len(w) - size + stride), stride)]
    return [c for c in out if c]


def chunk_sentences(sents, max_words):
    """Greedy sentence packing (FactCG / MiniCheck style chunking)."""
    chunks, cur, n = [], [], 0
    for s in sents:
        k = W(s)
        if cur and n + k > max_words:
            chunks.append(" ".join(cur))
            cur, n = [], 0
        cur.append(s)
        n += k
    if cur:
        chunks.append(" ".join(cur))
    return chunks or [""]

### `factindic/translit.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/translit.py
"""Romanization, script-mixing, cross-script mapping, and back-transliteration.

IndicXlit needs fairseq, which does not install on current Kaggle/Colab Python. Forward Romanization
here is rule-based ITRANS with schwa deletion (casual Hinglish/Gujlish style). Back-transliteration
uses IndicXlit when importable, otherwise the most frequent native spelling of that Roman form in the
training pool (data/roman_vocab.json), and an ITRANS reverse for unseen words.
"""
from __future__ import annotations

import re
import unicodedata
from functools import lru_cache

from .text import INDIC_RE, LATIN_RE, strip_punct

_CONS = "bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ"
GU_SUFFIX = sorted(["માં", "થી", "ને", "નો", "ની", "નું", "ના", "નાં", "એ"], key=len, reverse=True)


def _scheme(lang):
    from indic_transliteration import sanscript
    return {"hi": sanscript.DEVANAGARI, "gu": sanscript.GUJARATI}[lang]


_V, _C = "aAiIuUeEoO", "bcdfghjklmnpqrstvwxyzDTNLSJKGR"
_MEDIAL = re.compile(rf"(?<=[{_V}])([{_C}]h?)a(?=[{_C}]h?[{_V}])")
# ITRANS has no casual form for these: chandra vowels stayed in native script, nukta letters became ".D", "K"
_PRE = [(re.compile("[ऑ]"), "ओ"), (re.compile("[ॉॊ]"), "ो"), (re.compile("[ऍ]"), "ए"), (re.compile("[ॅ]"), "े"),
        (re.compile("[ઑ]"), "ઓ"), (re.compile("[ૉ]"), "ો"), (re.compile("[ઍ]"), "એ"), (re.compile("[ૅ]"), "ે"),
        (re.compile("([कखगडढयरन])़"), r"\1"), (re.compile("([કખગડઢ])઼"), r"\1")]


def _drop_medial_schwa(s):
    """VCaCV → VCCV, right to left (sabase → sabse, musalamAn → musalmAn)."""
    while True:
        hits = list(_MEDIAL.finditer(s))
        if not hits:
            return s
        m = hits[-1]
        s = s[:m.end() - 1] + s[m.end():]


def _casual(s):
    for a, b in [("R^i", "ri"), ("R^I", "ri"), ("L^i", "li"), (".N", "n"), ("~N", "n"), ("~n", "n"),
                 (".h", ""), (".a", ""), ("{}", ""), (".c", ""), ("^", ""), ("~", "n")]:
        s = s.replace(a, b)
    s = s.replace("|", ".")
    # final schwa goes (long A survives), except after a cluster ending in r/y/v (rashtra, rajya)
    if len(s) > 2 and s.endswith("a") and s[-2] in _CONS and not (s[-2] in "ryv" and s[-3] in _CONS):
        s = s[:-1]
    s = _drop_medial_schwa(s)
    s = s.replace("M", "n").replace("H", "h").replace("x", "ksh").lower()
    for v in "aiueo":
        s = re.sub(v + "{2,}", v, s)
    return s


@lru_cache(maxsize=500_000)
def _romanize_word(core, lang):
    """Romanizes each part of एस.डी. or पति-पत्नी alone, so schwa deletion sees its word ends."""
    parts = re.split(r"([^\u0900-\u0AFF\u200c\u200d]+)", core)
    if len(parts) > 1:
        return "".join(_romanize_word(p, lang) if INDIC_RE.search(p) else p for p in parts)
    from indic_transliteration import sanscript
    core = unicodedata.normalize("NFC", core)        # NFC keeps nukta as a separate mark (क़ → क + ़)
    for pat, rep in _PRE:
        core = pat.sub(rep, core)
    out = _casual(sanscript.transliterate(core, _scheme(lang), sanscript.ITRANS))
    return re.sub("[ऀ-૿]", "", out)


def romanize(text, lang):
    out = []
    for tok in text.split():
        pre, core, post = strip_punct(tok)
        if INDIC_RE.search(core):
            core = _romanize_word(core, lang)
        out.append((pre + core + post).replace("।", ".").replace("॥", "."))
    return " ".join(out)


def gu_to_deva(s):
    """Gujarati U+0A80–U+0AFF sits exactly 0x180 above Devanagari U+0900–U+097F."""
    return "".join(chr(ord(c) - 0x180) if 0x0A80 <= ord(c) <= 0x0AFF else c for c in s)


def deva_to_gu(s):
    return "".join(chr(ord(c) + 0x180) if 0x0900 <= ord(c) <= 0x097F else c for c in s)


def trim_gu(w):
    for suf in GU_SUFFIX:
        if w.endswith(suf) and len(w) - len(suf) >= 2:
            return w[: -len(suf)]
    return w


# --------------------------------------------------------------------------- script-mixed variant
LEX_RAW = {
    "hi": {"सरकार": "government", "प्रदूषण": "pollution", "स्तर": "level", "चुनाव": "election",
           "अस्पताल": "hospital", "मंत्री": "minister", "प्रधानमंत्री": "prime minister",
           "मुख्यमंत्री": "chief minister", "अदालत": "court", "न्यायालय": "court",
           "विश्वविद्यालय": "university", "छात्र": "student", "छात्रों": "students", "शिक्षा": "education",
           "स्वास्थ्य": "health", "बाज़ार": "market", "बाजार": "market", "कंपनी": "company",
           "रिपोर्ट": "report", "फ़ैसला": "decision", "फैसला": "decision", "फ़ैसले": "decision",
           "फैसले": "decision", "सुरक्षा": "security", "कानून": "law", "क़ानून": "law", "नीति": "policy",
           "अर्थव्यवस्था": "economy", "विकास": "development", "परियोजना": "project", "समस्या": "problem",
           "जानकारी": "information", "मामला": "case", "मामले": "case", "जांच": "investigation",
           "जाँच": "investigation", "घटना": "incident", "दुर्घटना": "accident", "मौत": "death",
           "देश": "country", "शहर": "city", "गांव": "village", "गाँव": "village", "बिजली": "electricity",
           "सड़क": "road", "किसान": "farmer", "किसानों": "farmers", "युद्ध": "war", "सेना": "army",
           "हमला": "attack", "हमले": "attack", "विरोध": "protest",
           "बयान": "statement", "अधिकारी": "officer", "अधिकारियों": "officers", "कर्मचारी": "employee",
           "कर्मचारियों": "employees", "नौकरी": "job", "वैज्ञानिक": "scientist", "शोध": "research",
           "अध्ययन": "study", "तकनीक": "technology", "फ़िल्म": "film", "फिल्म": "film",
           "खिलाड़ी": "player", "मैच": "match", "टीम": "team", "जलवायु": "climate",
           "तापमान": "temperature", "बारिश": "rain", "बाढ़": "flood", "वायरस": "virus",
           "वैक्सीन": "vaccine", "मरीज़": "patient", "मरीज": "patient", "मरीज़ों": "patients",
           "मरीजों": "patients", "डॉक्टर": "doctor", "इलाज": "treatment", "संक्रमण": "infection",
           "आंकड़े": "data", "आंकड़ों": "data", "सदस्य": "member", "संसद": "parliament",
           "पार्टी": "party", "नेता": "leader", "उम्मीदवार": "candidate", "वोट": "vote",
           "मतदान": "voting", "पुलिस": "police", "लगातार": "continuously"},
    "gu": {"સરકાર": "government", "પ્રદૂષણ": "pollution", "સ્તર": "level", "ચૂંટણી": "election",
           "હોસ્પિટલ": "hospital", "મંત્રી": "minister", "વડાપ્રધાન": "prime minister",
           "મુખ્યમંત્રી": "chief minister", "કોર્ટ": "court", "અદાલત": "court",
           "યુનિવર્સિટી": "university", "વિદ્યાર્થી": "student", "વિદ્યાર્થીઓ": "students",
           "શિક્ષણ": "education", "આરોગ્ય": "health", "બજાર": "market", "કંપની": "company",
           "રિપોર્ટ": "report", "નિર્ણય": "decision", "સુરક્ષા": "security", "કાયદો": "law",
           "કાયદા": "law", "નીતિ": "policy", "અર્થતંત્ર": "economy", "વિકાસ": "development",
           "પ્રોજેક્ટ": "project", "સમસ્યા": "problem", "માહિતી": "information", "કેસ": "case",
           "તપાસ": "investigation", "ઘટના": "incident", "અકસ્માત": "accident", "મૃત્યુ": "death",
           "દેશ": "country", "શહેર": "city", "ગામ": "village", "વીજળી": "electricity",
           "રસ્તો": "road", "ખેડૂત": "farmer", "ખેડૂતો": "farmers", "યુદ્ધ": "war", "સેના": "army",
           "હુમલો": "attack", "વિરોધ": "protest", "નિવેદન": "statement",
           "અધિકારી": "officer", "અધિકારીઓ": "officers", "કર્મચારી": "employee",
           "કર્મચારીઓ": "employees", "નોકરી": "job", "વૈજ્ઞાનિક": "scientist", "સંશોધન": "research",
           "અભ્યાસ": "study", "ટેકનોલોજી": "technology", "ફિલ્મ": "film", "ખેલાડી": "player",
           "મેચ": "match", "ટીમ": "team", "તાપમાન": "temperature", "વરસાદ": "rain", "પૂર": "flood",
           "વાયરસ": "virus", "રસી": "vaccine", "દર્દી": "patient", "દર્દીઓ": "patients",
           "ડૉક્ટર": "doctor", "સારવાર": "treatment", "ચેપ": "infection", "આંકડા": "data",
           "સભ્ય": "member", "સંસદ": "parliament", "પાર્ટી": "party", "નેતા": "leader",
           "ઉમેદવાર": "candidate", "મતદાન": "voting", "પોલીસ": "police", "સતત": "continuously"},
}
_LEX = {}


def lexicon(lang):
    if lang not in _LEX:
        from .text import clean
        _LEX[lang] = {clean(k, lang): v for k, v in LEX_RAW[lang].items()}
    return _LEX[lang]


def script_mix(text, lang, entities=()):
    """Embed Latin-script English content words inside the native matrix frame; keep Gujarati
    case enclitics as separate tokens ("pollution નું"). Words inside named entities stay as they are
    (विकास दुबे, સુરક્ષા દળ), so A_core does not change. Returns (text, embedded_english_terms)."""
    lex, out, emb = lexicon(lang), [], []
    toks = text.split()
    cores = [strip_punct(t)[1] for t in toks]
    keep = set()
    for e in entities:
        ew = [strip_punct(w)[1] for w in e.split()]
        for i in range(len(cores) - len(ew) + 1):
            if cores[i:i + len(ew)] == ew:
                keep.update(range(i, i + len(ew)))
    for i, tok in enumerate(toks):
        pre, core, post = strip_punct(tok)
        if i in keep:
            out.append(tok)
            continue
        rep = lex.get(core)
        if rep is None and lang == "gu":
            for suf in GU_SUFFIX:
                if core.endswith(suf) and core[: -len(suf)] in lex:
                    rep = lex[core[: -len(suf)]] + " " + suf
                    emb.append(lex[core[: -len(suf)]])
                    break
        elif rep is not None:
            emb.append(rep)
        out.append(pre + rep + post if rep else tok)
    return " ".join(out), sorted(set(emb))


# --------------------------------------------------------------------------- back-transliteration
_XLIT = {}


def _xlit_engine(lang):
    if lang in _XLIT:
        return _XLIT[lang]
    try:
        from ai4bharat.transliteration import XlitEngine
        _XLIT[lang] = XlitEngine(lang, beam_width=4, src_script_type="roman")
    except Exception:
        _XLIT[lang] = None
    return _XLIT[lang]


def back_transliterate_word(w, lang):
    eng = _xlit_engine(lang)
    if eng is not None:
        try:
            res = eng.translit_word(w, topk=1)
            cand = res[lang][0] if isinstance(res, dict) else res[0]
            if cand:
                return cand
        except Exception:
            pass
    from .lid import roman_vocab
    hit = roman_vocab(lang).get(w.lower())
    if hit:
        return hit[0]
    from indic_transliteration import sanscript
    s = w.lower()
    if s and s[-1] in _CONS.lower():
        s = s + "a"                     # restore the inherent vowel dropped by schwa deletion
    return sanscript.transliterate(s, sanscript.ITRANS, _scheme(lang))


def back_transliterate(text, lang, tags=None):
    """Romanized Indic tokens → native script; English tokens (per LID) are kept in Latin."""
    from .lid import tag_tokens
    toks = text.split()
    tags = tags or tag_tokens(text, lang)
    out = []
    for tok, tag in zip(toks, tags):
        pre, core, post = strip_punct(tok)
        if tag.endswith("_Latn") and not tag.startswith("eng") and LATIN_RE.search(core):
            core = back_transliterate_word(core, lang)
        out.append(pre + core + post)
    return " ".join(out)


def xlit_backend():
    return "IndicXlit" if _xlit_engine("hi") is not None else "pool-vocabulary + ITRANS-reverse"

### `factindic/lid.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/lid.py
"""Word-level language ID and the Code-Mixing Index (Gambäck & Das).

AI4Bharat IndicLID needs fairseq + large fastText assets; this lightweight tagger uses script for
native tokens and an English frequency lexicon (wordfreq) plus a romanized-Indic function-word list
for Latin tokens. Tags: hin_Deva, guj_Gujr, hin_Latn, guj_Latn, eng_Latn, univ (digits/punct).
"""
from __future__ import annotations

from .text import INDIC_RE, LATIN_RE, strip_punct

ROMAN_INDIC_STOP = {
    "hi": set("hai hain ka ki ke ko se mein me main aur ya bhi nahi nahin tha thi the ho raha rahi rahe "
              "kar karna kiya gaya gayi wajah liye par pe jo jab tab kya kyun ek do teen log sab koi "
              "apne apna unke unki uska uski yeh ye woh vo hum tum aap".split()),
    "gu": set("chhe che chhu hatu hati hata ane na ni nu no ne ma thi pan ke e aa te tame ame hu "
              "thai thay rahyu rahi karvu karyu kar ek be tran loko badha koi lidhe mate sathe "
              "upar niche satat".split()),
}

_ZIPF = None


def _zipf(w):
    global _ZIPF
    if _ZIPF is None:
        try:
            from wordfreq import zipf_frequency
            _ZIPF = zipf_frequency
        except ImportError:
            _ZIPF = False
    return _ZIPF(w, "en") if _ZIPF else 0.0


_VOCAB = {}


def roman_vocab(lang):
    """{roman form: [most frequent native spelling, zipf]} built by notebook 01 from the training pool."""
    if lang not in _VOCAB:
        from .config import DIRS
        from .io import json_load
        _VOCAB.update(json_load(DIRS["data"] / "roman_vocab.json", {"hi": {}, "gu": {}}))
    return _VOCAB[lang]


def tag_word(tok, lang, romanized=False):
    core = strip_punct(tok)[1]
    if not core or core.replace(".", "").replace(",", "").isdigit():
        return "univ"
    if INDIC_RE.search(core):
        return {"hi": "hin_Deva", "gu": "guj_Gujr"}[lang]
    if LATIN_RE.search(core):
        w = core.lower()
        if w in ROMAN_INDIC_STOP[lang]:
            return f"{'hin' if lang == 'hi' else 'guj'}_Latn"
        if romanized:                    # में → "men": in Romanized text the Indic reading wins when it is at least as common
            hit = roman_vocab(lang).get(w)
            if hit and hit[1] + 1.0 >= _zipf(w):
                return f"{'hin' if lang == 'hi' else 'guj'}_Latn"
        return "eng_Latn" if _zipf(w) >= 3.0 else f"{'hin' if lang == 'hi' else 'guj'}_Latn"
    return "univ"


def tag_tokens(text, lang):
    toks = text.split()
    latin = sum(bool(LATIN_RE.search(t)) for t in toks)
    indic = sum(bool(INDIC_RE.search(t)) for t in toks)
    return [tag_word(t, lang, romanized=latin > indic) for t in toks]


def mixed_cmi(text, lang, max_run):
    """CMI without runs of more than max_run Latin-script words: a caption, tweet or headline in English
    is a separate passage, not mixing inside a sentence. Digits and punctuation do not end a run."""
    toks, keep, run = text.split(), [], []
    for t in toks + ["\u0915"]:
        if LATIN_RE.search(t) and not INDIC_RE.search(t):
            run.append(t)
        elif INDIC_RE.search(t):
            keep += (run if len(run) <= max_run else []) + [t]
            run = []
    return cmi(" ".join(keep[:-1]), lang)


def cmi(text, lang, tags=None):
    """CMI = 100·(1 − max_L w(L) / (N − u)) with languages {Indic (either script), English}."""
    tags = tags or tag_tokens(text, lang)
    n = len(tags)
    u = sum(t == "univ" for t in tags)
    if n <= u:
        return 0.0
    eng = sum(t == "eng_Latn" for t in tags)
    ind = n - u - eng
    return 100.0 * (1 - max(eng, ind) / (n - u))

### `factindic/normalizer.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/normalizer.py
"""Deterministic Numeral, Date, and Cross-Script Entity Normalizer (A-5) and the Core Anchor Set.

A_core(D) = {Norm(E_D), Norm(N_D), Norm(T_D)} is represented as a set of canonical strings:
    "num:150000", "date:08-15", "date:2019-08", "ent:<vowel-collapsed ascii key>".
Embedded English terms X_D are tracked separately so A_core stays invariant across variants.
"""
from __future__ import annotations

import re
import unicodedata

from .text import INDIC_RE, strip_punct
from .translit import GU_SUFFIX, gu_to_deva, trim_gu

DIGITS = str.maketrans("०१२३४५६७८९૦૧૨૩૪૫૬૭૮૯", "01234567890123456789")
_nfc = lambda s: unicodedata.normalize("NFC", s)

MULT = {_nfc(k): v for k, v in {
    "सौ": 1e2, "સો": 1e2, "hundred": 1e2, "हज़ार": 1e3, "हजार": 1e3, "હજાર": 1e3, "hazar": 1e3,
    "hazaar": 1e3, "thousand": 1e3, "लाख": 1e5, "લાખ": 1e5, "lakh": 1e5, "lakhs": 1e5, "lac": 1e5,
    "करोड़": 1e7, "करोड": 1e7, "કરોડ": 1e7, "crore": 1e7, "crores": 1e7, "karod": 1e7,
    "million": 1e6, "मिलियन": 1e6, "મિલિયન": 1e6, "billion": 1e9, "अरब": 1e9, "અબજ": 1e9,
    "बिलियन": 1e9, "બિલિયન": 1e9}.items()}
WORDNUM = {_nfc(k): v for k, v in {
    "एक": 1, "दो": 2, "तीन": 3, "चार": 4, "पांच": 5, "पाँच": 5, "छह": 6, "सात": 7, "आठ": 8, "नौ": 9,
    "दस": 10, "बीस": 20, "पचास": 50, "डेढ़": 1.5, "ढाई": 2.5, "सवा": 1.25,
    "એક": 1, "બે": 2, "ત્રણ": 3, "ચાર": 4, "પાંચ": 5, "છ": 6, "સાત": 7, "આઠ": 8, "નવ": 9,
    "દસ": 10, "વીસ": 20, "પચાસ": 50, "દોઢ": 1.5, "અઢી": 2.5, "સવા": 1.25,
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
    "nine": 9, "ten": 10, "dedh": 1.5, "dhai": 2.5}.items()}
MONTHS = {_nfc(k): v for k, v in {
    "जनवरी": 1, "फ़रवरी": 2, "फरवरी": 2, "मार्च": 3, "अप्रैल": 4, "मई": 5, "जून": 6, "जुलाई": 7,
    "अगस्त": 8, "सितंबर": 9, "सितम्बर": 9, "अक्टूबर": 10, "अक्तूबर": 10, "नवंबर": 11, "नवम्बर": 11,
    "दिसंबर": 12, "दिसम्बर": 12,
    "જાન્યુઆરી": 1, "ફેબ્રુઆરી": 2, "માર્ચ": 3, "એપ્રિલ": 4, "મે": 5, "જૂન": 6, "જુલાઈ": 7,
    "ઑગસ્ટ": 8, "ઓગસ્ટ": 8, "સપ્ટેમ્બર": 9, "ઑક્ટોબર": 10, "ઓક્ટોબર": 10, "નવેમ્બર": 11,
    "ડિસેમ્બર": 12,
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6, "july": 7,
    "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "sept": 9,
    "oct": 10, "nov": 11, "dec": 12}.items()}


def _lookup(table, w):
    w = _nfc(w).lower()
    if w in table:
        return table[w]
    for suf in GU_SUFFIX + ["ों", "के", "की", "का"]:
        if w.endswith(suf) and w[: -len(suf)] in table:
            return table[w[: -len(suf)]]
    return None


def _fmt(v):
    return ("%.4f" % v).rstrip("0").rstrip(".")


def _tokens(text):
    toks = [strip_punct(t)[1] for t in (text or "").translate(DIGITS).split()]
    return [t for t in toks if t]


_NUM_RE = re.compile(r"[₹$]?(\d[\d,]*(?:\.\d+)?)")


def numbers(text):
    """Canonical numeric values; bare एक/એક ('a/one') counts only before a multiplier."""
    toks, vals, i = _tokens(text), set(), 0
    while i < len(toks):
        w, v = toks[i], None
        m = _NUM_RE.match(w)
        nxt = _lookup(MULT, toks[i + 1]) if i + 1 < len(toks) else None
        if m:
            try:
                v = float(m.group(1).replace(",", ""))
            except ValueError:
                v = None
        elif _lookup(WORDNUM, w) is not None and nxt is not None:
            v = _lookup(WORDNUM, w)
        if v is not None:
            if nxt is not None:
                v *= nxt
                i += 1
            vals.add(_fmt(v))
        i += 1
    return vals


def dates(text):
    """'15 अगस्त' → 08-15; 'अगस्त 2019' → 2019-08; '15 August 2019' → 2019-08-15."""
    toks, out = _tokens(text), set()
    for i, w in enumerate(toks):
        mo = _lookup(MONTHS, w)
        if mo is None:
            continue
        day = year = None
        if i > 0 and re.fullmatch(r"\d{1,2}", toks[i - 1]) and 1 <= int(toks[i - 1]) <= 31:
            day = int(toks[i - 1])
        if i + 1 < len(toks):
            m = re.match(r"(\d{4})", toks[i + 1])
            if m and 1900 <= int(m.group(1)) <= 2100:
                year = int(m.group(1))
        if day and year:
            out.add(f"{year:04d}-{mo:02d}-{day:02d}")
        elif day:
            out.add(f"{mo:02d}-{day:02d}")
        elif year:
            out.add(f"{year:04d}-{mo:02d}")
    return out


def ent_key(word, lang):
    """1-to-1 cross-script key: trim Gujarati enclitics, map Gujarati→Devanagari by the 0x180 block
    offset, romanize, lowercase ASCII, collapse vowels (keep the first character)."""
    from .translit import romanize
    w = strip_punct(word)[1]
    if not w:
        return ""
    if INDIC_RE.search(w):
        if re.search(r"[઀-૿]", w):
            w = gu_to_deva(trim_gu(w))
        r = romanize(w, "hi")
    else:
        r = w.lower()
    r = re.sub(r"[^a-z0-9]", "", r)
    return (r[0] + re.sub(r"[aeiouy]", "", r[1:])) if r else ""


def entity_keys(entity_text, lang):
    return [k for k in (ent_key(w, lang) for w in entity_text.split()) if k]


def _close(k, keys):
    if k in keys:
        return True
    if len(k) <= 2:
        return False
    from rapidfuzz.distance import Levenshtein
    return any(abs(len(k) - len(s)) <= 1 and Levenshtein.distance(k, s, score_cutoff=1) <= 1 for s in keys)


def anchors(text, lang, entities=None, english_terms=()):
    """entities: list of entity surface strings (from ner.py). Returns a dict for JSONL storage."""
    ents = []
    for e in entities or []:
        ks = entity_keys(e, lang)
        if ks:
            ents.append(dict(text=e, key=" ".join(ks)))
    return dict(numbers=sorted(numbers(text)), dates=sorted(dates(text)), entities=ents,
                english=sorted(set(english_terms)))


def anchor_set(a):
    s = {f"num:{n}" for n in a.get("numbers", [])} | {f"date:{d}" for d in a.get("dates", [])}
    s |= {f"ent:{e['key']}" for e in a.get("entities", [])}
    return s


def match_anchors(a_s, a_d, d_text=None, lang=None):
    """Which anchors of summary S are supported by D. Entities match if every word key is within
    edit distance ≤ 1 of an entity word key of D or of any word key of D's text (NER misses)."""
    d_nums, d_dates = set(a_d.get("numbers", [])), set(a_d.get("dates", []))
    d_keys = {k for e in a_d.get("entities", []) for k in e["key"].split()}
    if d_text is not None:
        d_keys |= {k for k in (ent_key(w, lang) for w in d_text.split()) if k}
    out = []
    for n in a_s.get("numbers", []):
        out.append(("num:" + n, n in d_nums))
    for d in a_s.get("dates", []):
        out.append(("date:" + d, d in d_dates or any(x.endswith(d) or d.endswith(x) for x in d_dates)))
    for e in a_s.get("entities", []):
        out.append(("ent:" + e["key"], all(_close(k, d_keys) for k in e["key"].split())))
    return out


def anchor_prf(a_s, a_d, a_r, d_text=None, lang=None):
    """P = |A(S)∩A(D)|/|A(S)|; R = |A(S)∩A*(R)|/|A*(R)| with A*(R) = A(R)∩A(D)."""
    sup = match_anchors(a_s, a_d, d_text, lang)
    p = sum(ok for _, ok in sup) / len(sup) if sup else float("nan")
    r_in_d = [k for k, ok in match_anchors(a_r, a_d, d_text, lang) if ok]
    if not r_in_d:
        r = float("nan")
    else:
        s_keys = [k for k, ok in sup if ok]
        def hit(rk):
            if rk in s_keys:
                return True
            if rk.startswith("ent:"):
                ks = {x for sk in s_keys if sk.startswith("ent:") for x in sk[4:].split()}
                return all(_close(x, ks) for x in rk[4:].split())
            return False
        r = sum(hit(rk) for rk in r_in_d) / len(r_in_d)
    f = 2 * p * r / (p + r) if p == p and r == r and (p + r) > 0 else float("nan")
    return dict(P_anchor=p, R_anchor=r, F1_anchor=f, n_anchor_S=len(sup), n_anchor_R=len(r_in_d),
                n_unsup_num=sum(1 for k, ok in sup if k.startswith("num:") and not ok),
                n_unsup_ent=sum(1 for k, ok in sup if k.startswith("ent:") and not ok))


def salient_anchors(a_d, d_text, lang, top=6):
    """Most frequent source anchors (ties → earlier first mention), used by the Stage-4 J score."""
    keys = sorted(anchor_set(a_d))           # a fixed order for ties, whatever the hash seed
    if not keys:
        return []
    toks = d_text.split()
    def first_pos_and_count(k):
        kind, val = k.split(":", 1)
        if kind == "num":
            pos = [i for i, t in enumerate(toks) if val in numbers(t)]
        elif kind == "ent":
            first = val.split()[0]
            pos = [i for i, t in enumerate(toks) if ent_key(t, lang) == first]
        else:
            pos = []
        return (-len(pos), pos[0] if pos else 10 ** 6)
    return sorted(keys, key=first_pos_and_count)[:top]


def anchor_recall_salient(a_s, a_d, salient, d_text, lang):
    if not salient:
        return 1.0
    sup = {k for k, ok in match_anchors(a_s, a_d, d_text, lang) if ok}
    s_ent = {x for k in sup if k.startswith("ent:") for x in k[4:].split()}
    hit = 0
    for k in salient:
        if k in sup:
            hit += 1
        elif k.startswith("ent:") and all(_close(x, s_ent) for x in k[4:].split()):
            hit += 1
    return hit / len(salient)


def anchor_density(text):
    return len(numbers(text)) + len(dates(text))

### `factindic/ner.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/ner.py
"""Named entities in native script (IndicNER) and Latin script (wikineural), offset-safe.

The IndicNER tokenizer strips Indic vowel signs from `entity["word"]` (वेस्टइंडीज → सटइडीज), which
silently broke anchor matching in H1 run 1. Spans are therefore cut from the original text by
character offsets, widened to whole words, and stripped of outer punctuation.
"""
from __future__ import annotations

from .text import INDIC_RE, LATIN_RE, split_sentences, strip_punct

KEEP = {"PER", "LOC", "ORG"}


def _span(S, a, b):
    while a > 0 and not S[a - 1].isspace():
        a -= 1
    while b < len(S) and not S[b].isspace():
        b += 1
    return strip_punct(S[a:b].strip())[1]


class Tagger:
    def __init__(self, device=0, latin=True):
        from transformers import pipeline
        from .config import MODELS
        self.native = pipeline("token-classification", model=MODELS["indicner"],
                               aggregation_strategy="first", device=device)
        self.latin = (pipeline("token-classification", model=MODELS["latin_ner"],
                               aggregation_strategy="first", device=device) if latin else None)

    def _run(self, pipe, text):
        out = []
        for sent in split_sentences(text) or [text]:
            if not sent.strip():
                continue
            for e in pipe(sent[:2000]):
                grp = e.get("entity_group", "").replace("B-", "").replace("I-", "")
                if grp in KEEP:
                    s = _span(sent, e["start"], e["end"])
                    if s:
                        out.append(s)
        return out

    def entities(self, text):
        ents = []
        if INDIC_RE.search(text):
            ents += self._run(self.native, text)
        if self.latin is not None and LATIN_RE.search(text):
            ents += [e for e in self._run(self.latin, text) if LATIN_RE.search(e)]
        return list(dict.fromkeys(ents))

## Datasets (Stage 0)

XL-Sum and ILSUM, the 5-rule sanitation, the parallel testbed, the dev and natural code-mixed sets, and the training pool.

### `factindic/data.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/data.py
"""Datasets: XL-Sum, ILSUM (+ 5-rule sanitation), the controlled parallel testbed, dev split,
natural code-mixed subset, and the training pool for faithfulness weighting."""
from __future__ import annotations

import hashlib
import math
import random
from collections import Counter
from pathlib import Path

import numpy as np

from .config import CFG, DATA_VERSION, DIRS, LANGS, VERSION_FILE, data_version, h1_pilot_ids
from .io import jl_read, jl_write, json_dump
from .lid import cmi, mixed_cmi, tag_word
from .normalizer import anchor_density, anchors
from .text import LATIN_RE, W, clean, strip_boilerplate, strip_punct, strip_web, truncate_words
from .translit import romanize, script_mix


# --------------------------------------------------------------------------- loaders
def load_xlsum(lang, split):
    """Parquet conversion branch first (works without dataset scripts), then main branch."""
    from datasets import Dataset, concatenate_datasets, load_dataset
    from huggingface_hub import hf_hub_download, list_repo_files
    name = LANGS[lang]["xlsum"]
    split = "validation" if split in ("val", "valid", "dev") else split
    for rev in ("refs/convert/parquet", "main"):
        try:
            files = list_repo_files("csebuetnlp/xlsum", repo_type="dataset", revision=rev)
            shards = sorted(f for f in files if f.endswith(".parquet") and f.startswith(f"{name}/")
                            and (f"/{split}/" in f or f"-{split}" in f))
            if not shards:
                continue
            paths = [hf_hub_download("csebuetnlp/xlsum", f, repo_type="dataset", revision=rev)
                     for f in shards]
            return concatenate_datasets([Dataset.from_parquet(p) for p in paths])
        except Exception as e:
            print(f"xlsum {lang}/{split} @ {rev}: {repr(e)[:150]}")
    return load_dataset("csebuetnlp/xlsum", name, split=split)


def load_ilsum(lang, split, version="ILSUM/ILSUM-1.0"):
    """Returns list of dicts (id, article, summary) or [] if unavailable."""
    from datasets import load_dataset
    cfgname = LANGS[lang]["ilsum"]
    try:
        ds = load_dataset(version, cfgname, split=split)
    except Exception:
        try:
            from huggingface_hub import hf_hub_download
            fn = {"validation": "val"}.get(split, split)
            p = hf_hub_download(version, f"{cfgname}/{fn}.csv", repo_type="dataset")
            ds = load_dataset("csv", data_files=p, split="train")
        except Exception as e:
            print(f"ILSUM {lang}/{split} unavailable: {repr(e)[:150]}")
            return []
    cols = {c.lower(): c for c in ds.column_names}
    a, s = cols.get("article"), cols.get("summary")
    i = cols.get("id")
    return [dict(id=str(r[i]) if i else str(k), article=strip_web(r[a]), summary=strip_web(r[s]) if s else "")
            for k, r in enumerate(ds)]


# --------------------------------------------------------------------------- 5-rule sanitation
def prefix_copy(article, summary, max_offset=6):
    """Summary = the article's opening words, allowing a dateline or byline before it
    ("नई दिल्लीः …"). Compared word by word without punctuation; 90% of the words must match."""
    core = lambda t: [c for c in (strip_punct(w)[1] for w in t.split()) if c]
    a, s = core(article), core(summary)
    k = len(s)
    for o in range(max_offset + 1):
        head = a[o:o + k]
        if len(head) < k:
            return False
        if head == s or (k >= 8 and sum(x == y for x, y in zip(head, s)) / k >= 0.9):
            return True
    return False


def sanitize(rows, lang):
    """Rules: (1) empty / <5-word article or summary, (2) duplicate pairs, (3) duplicate summaries,
    (4) article-prefix copies (also after a dateline), (5) summary > 50% of article."""
    counts, kept = Counter(), []
    seen_pairs, seen_summ = set(), set()
    for r in rows:
        a, s = clean(r["article"], lang), clean(r["summary"], lang)
        wa, ws = a.split(), s.split()
        if len(wa) < 5 or len(ws) < 5:
            counts["empty_or_short"] += 1
            continue
        if (a, s) in seen_pairs:
            counts["duplicate_pair"] += 1
            continue
        seen_pairs.add((a, s))
        if s in seen_summ:
            counts["duplicate_summary"] += 1
            continue
        seen_summ.add(s)
        if prefix_copy(a, s):
            counts["article_prefix_copy"] += 1
            continue
        if len(ws) > 0.5 * len(wa):
            counts["summary_gt_50pct"] += 1
            continue
        kept.append(dict(id=r["id"], lang=lang, text=a, summary=s))
    report = dict(lang=lang, raw=len(rows), kept=len(kept), removed=dict(counts),
                  invalid_pct=round(100 * (len(rows) - len(kept)) / max(1, len(rows)), 2))
    return kept, report


# --------------------------------------------------------------------------- article records
def make_record(lang, source, split, doc_id, text, reference, tagger=None, tertile=None):
    D = truncate_words(strip_boilerplate(clean(text, lang)), CFG["max_article_words"])
    R = clean(reference, lang)
    ents = tagger.entities(D) if tagger else []
    mixed, emb = script_mix(D, lang, ents)
    variants = {"native": D, "script_mixed": mixed, "roman": romanize(D, lang)}
    ref_ents = tagger.entities(R) if tagger else []
    rec = dict(uid=f"{lang}:{source}:{doc_id}", lang=lang, source=source, split=split, doc_id=doc_id,
               text=D, reference=R, L_target=int(np.clip(W(R), 25, 45)), tertile=tertile,
               variants=variants, english_terms=emb,
               cmi={v: round(cmi(t, lang), 2) for v, t in variants.items()},   # Romanization must not change it
               anchors=anchors(D, lang, ents, emb), ref_anchors=anchors(R, lang, ref_ents))
    return rec


def _pool(lang, split, exclude):
    ds = load_xlsum(lang, split)
    rows = []
    for ex in ds:
        if ex["id"] in exclude:
            continue
        t, s = strip_boilerplate(clean(ex["text"], lang)), clean(ex["summary"], lang)
        if 150 <= W(t) <= 1500 and 10 <= W(s) <= 60:
            rows.append(dict(id=ex["id"], text=t, summary=s))
    return rows


def _stratified(rows, n, tagger, lang, rng, probe=600):
    """Verifier-independent stratified sampling over entity-density tertiles (seeded)."""
    cand = rng.sample(rows, min(probe, len(rows)))
    dens = []
    for r in cand:
        D = truncate_words(r["text"], CFG["max_article_words"])
        n_ent = len(tagger.entities(D)) if tagger else 0
        dens.append((n_ent + anchor_density(D)) / max(1, W(D)))
    dens = np.array(dens)
    cuts = np.quantile(dens, [1 / 3, 2 / 3])
    tert = np.digitize(dens, cuts)
    per = [n - 2 * (n // 3), n // 3, n // 3]
    out = []
    for t, k in enumerate(per):
        idx = [i for i in range(len(cand)) if tert[i] == t]
        out += [(cand[i], t) for i in rng.sample(idx, min(k, len(idx)))]
    return out


def upgrade():
    """Move files built by an older data version aside so they are rebuilt. The training pool is kept
    and repaired (build_train_pool), so its translations stay cached. From version 2 on, the testbed and
    dev articles are right: only their derived fields are computed again."""
    old = data_version()
    if old == DATA_VERSION:
        return
    if not any((DIRS["data"] / f).exists() for f in ("train_pool.jsonl", "testbed.jsonl", "natural_cm.jsonl")):
        VERSION_FILE.write_text(str(DATA_VERSION))       # fresh start: a retry must continue, not start over
        return
    redo = ["natural_cm.jsonl", "sanitation_report.json", "score_subset.json"]
    if old >= 2:
        for name in ("testbed", "dev"):
            path = DIRS["data"] / f"{name}.jsonl"
            if path.exists():
                jl_write(path, [_refresh(r) for r in jl_read(path)])
    else:
        redo += ["testbed.jsonl", "dev.jsonl", "weights.jsonl", "weights.shard0.jsonl", "weights.shard1.jsonl"]
    for f in redo:
        p = DIRS["data"] / f
        if p.exists():
            p.rename(p.with_name(f"old_v{old}_{f}"))
    pool = DIRS["data"] / "train_pool.jsonl"
    if pool.exists():
        pool.rename(pool.with_name("old_train_pool.jsonl"))


def _refresh(r):
    """Romanized variant, its CMI, and anchor keys of a stored record, without running NER again."""
    lang, names = r["lang"], lambda a: [e["text"] for e in a["entities"]]
    r["variants"]["roman"] = romanize(r["text"], lang)
    r["cmi"]["roman"] = round(cmi(r["variants"]["roman"], lang), 2)
    r["anchors"] = anchors(r["text"], lang, names(r["anchors"]), r["english_terms"])
    r["ref_anchors"] = anchors(r["reference"], lang, names(r["ref_anchors"]))
    return r


def stamp():
    for p in DIRS["data"].glob("old_*"):
        p.unlink()
    VERSION_FILE.write_text(str(DATA_VERSION))


def build_roman_vocab():
    """Roman form → most frequent native spelling, from the training pool (never the test articles)."""
    path = DIRS["data"] / "roman_vocab.json"
    if path.exists():
        return
    out = {}
    for lang in LANGS:
        lo, hi = LANGS[lang]["block"]
        native = Counter(w for r in jl_read(DIRS["data"] / "train_pool.jsonl") if r["lang"] == lang
                         for w in (strip_punct(t)[1] for t in r["text"].split())
                         if w and all(lo <= ord(c) <= hi for c in w))
        total, forms = sum(native.values()), {}
        for w, n in native.most_common():
            if n < 2:
                break
            f = forms.setdefault(romanize(w, lang), [w, 0])
            f[1] += n
        out[lang] = {k: [w, round(math.log10(n / total * 1e9), 2)] for k, (w, n) in forms.items() if k}
        print(f"roman_vocab/{lang}: {len(out[lang])} forms")
    json_dump(out, path)


def build_eval_sets(tagger):
    """testbed.jsonl (n_test_per_lang per language, XL-Sum test), dev.jsonl (50+50 XL-Sum validation)."""
    h1_ids = h1_pilot_ids()
    sets = {"testbed": ("test", CFG["n_test_per_lang"]), "dev": ("validation", CFG["n_dev_per_lang"])}
    for name, (split, n) in sets.items():
        path = DIRS["data"] / f"{name}.jsonl"
        if path.exists():
            continue
        rng = random.Random(f"{CFG['seed']}:{name}")
        recs = []
        for lang in LANGS:
            rows = _pool(lang, split, exclude=h1_ids)
            for r, t in _stratified(rows, n, tagger, lang, rng):
                recs.append(make_record(lang, "xlsum", split, r["id"], r["text"], r["summary"], tagger, int(t)))
            print(f"{name}/{lang}: {sum(x['lang'] == lang for x in recs)}")
        jl_write(path, recs)


def build_natural_cm(tagger):
    """Code-mixed ILSUM articles that pass the sanitation rules. Almost every Hindi article with CMI ≥ 8
    (the spec's rule) had page code in it, so whole English passages are not counted and the threshold
    steps down until there are enough articles. A random sample, since the top ranks are mostly job ads."""
    path = DIRS["data"] / "natural_cm.jsonl"
    if path.exists():
        return
    rng, steps = random.Random(CFG["seed"]), CFG["cmi_natural"]
    pool = jl_read(DIRS["data"] / "train_pool.jsonl")
    recs = []
    for lang in LANGS:
        used = {r["id"] for r in pool if r["lang"] == lang}
        seen = {r["text"][:200] for r in pool if r["lang"] == lang}
        scored = []
        for sp in ("test", "validation", "train"):
            rows = [r for r in load_ilsum(lang, sp) if r["id"] not in used]
            for r in sanitize(rows, lang)[0]:
                D = truncate_words(r["text"], CFG["max_article_words"])
                if W(D) < 100 or D[:200] in seen:
                    continue
                seen.add(D[:200])
                c = mixed_cmi(D, lang, CFG["cmi_max_run"])
                if c >= steps[-1]:
                    scored.append((c, r, sp))
        n = CFG["n_natural_cm_per_lang"]
        cut = next((s for s in steps if sum(x[0] >= s for x in scored) >= n), steps[-1])
        cand = [x for x in scored if x[0] >= cut]
        top = rng.sample(cand, min(n, len(cand)))
        for c, r, sp in top:
            rec = make_record(lang, "ilsum", sp, r["id"], r["text"], r["summary"], tagger)
            toks = rec["text"].split()
            rec["english_terms"] = sorted({strip_punct(t)[1].lower() for t in toks
                                           if LATIN_RE.search(t) and tag_word(t, lang) == "eng_Latn"})
            rec["anchors"]["english"] = rec["english_terms"]
            rec["variants"] = {"natural_cm": rec["text"]}       # already naturally mixed
            rec["cmi"] = {"natural_cm": round(c, 2)}
            recs.append(rec)
        cs = [c for c, _, _ in top]
        print(f"natural_cm/{lang}: {len(top)} of {len(cand)} with CMI≥{cut:g}; mean {np.mean(cs):.1f}, "
              f"{sum(c >= steps[0] for c in cs)} with CMI≥{steps[0]:g}"
              if cs else f"natural_cm/{lang}: none")
    jl_write(path, recs)


def build_train_pool():
    """Sanitized ILSUM-1.0 train (+ XL-Sum train to fill) → pool for weighting and SFT."""
    path = DIRS["data"] / "train_pool.jsonl"
    rep_path = DIRS["data"] / "sanitation_report.json"
    if path.exists():
        return
    rng = random.Random(CFG["seed"])
    old = jl_read(DIRS["data"] / "old_train_pool.jsonl")      # keep the valid pairs of an earlier pool
    seed = [] if old else pool_seed()
    pool, reports = [], []
    for lang in LANGS:
        raw = load_ilsum(lang, "train")
        clean_rows, rep = sanitize(raw, lang)
        reports.append(rep)
        print("ILSUM sanitation:", rep)
        n_target = CFG["n_weighted_per_lang"]
        had = {r["id"] for r in old or seed if r["lang"] == lang and r["source"] == "ilsum"}
        il = [r for r in clean_rows if r["id"] in had][: n_target // 2]
        rest = [r for r in clean_rows if r["id"] not in had]
        il += rng.sample(rest, min(n_target // 2 - len(il), len(rest)))
        for r in il:
            pool.append(dict(r, source="ilsum", text=truncate_words(r["text"], CFG["max_article_words"])))
        kept_xl = [r for r in old if r["lang"] == lang and r["source"] == "xlsum"]
        if kept_xl:
            pool += kept_xl
            continue
        xs = load_xlsum(lang, "train")
        idx = rng.sample(range(len(xs)), min(len(xs), 3 * n_target))
        want = {r["id"] for r in seed if r["lang"] == lang and r["source"] == "xlsum"}
        if want:
            idx = [j for j, i in enumerate(xs["id"]) if i in want] + idx
        xl, got = [], set()
        for j in idx:
            ex = xs[j]
            if ex["id"] in got:
                continue
            got.add(ex["id"])
            t, s = strip_boilerplate(clean(ex["text"], lang)), clean(ex["summary"], lang)
            if W(t) >= 100 and 10 <= W(s) <= 60:
                xl.append(dict(id=ex["id"], lang=lang, text=t, summary=s))
            if len(xl) >= n_target - len(il):
                break
        for r in xl:
            pool.append(dict(r, source="xlsum", text=truncate_words(r["text"], CFG["max_article_words"])))
    if old:
        _drop_stale_scores(old, pool)
    else:
        _seed_scores(seed, pool)
    jl_write(path, pool)
    json_dump(reports, rep_path)


def pool_seed():
    """Ids of the training pool of the finished notebook-01 run and its faithfulness scores (pool_seed.txt),
    so the pool is rebuilt identically and unchanged pairs are not scored again."""
    path = Path(__file__).with_name("pool_seed.txt")
    if not path.exists():
        return []
    cols = ("lang", "source", "id", "sig", "s_mc")
    return [dict(zip(cols, line.split("\t"))) for line in path.read_text(encoding="utf-8").splitlines()]


def _signature(r):
    return hashlib.md5("".join((r["text"] + r["summary"]).split()).encode()).hexdigest()[:8]


def _seed_scores(seed, pool):
    path = DIRS["data"] / "weights.jsonl"
    if path.exists() or list(DIRS["data"].glob("weights.shard*.jsonl")):
        return
    known = {(r["lang"], r["id"]): r for r in seed if r["s_mc"]}
    rows = []
    for r in pool:
        k = known.get((r["lang"], r["id"]))
        if k and k["sig"] == _signature(r):
            s = float(k["s_mc"])
            rows.append(dict(key=f"{r['lang']}:{r['id']}", lang=r["lang"], source=r["source"], s_mc=s,
                             w=float(np.clip(s, CFG["w_min"], 1.0)), keep=s >= CFG["w_min"], mt="it2"))
    jl_write(path, rows)
    print(f"{len(rows)} faithfulness scores reused from the finished run")


def _drop_stale_scores(old, pool):
    """Remove faithfulness scores of pairs whose text or summary is no longer the same."""
    sig = lambda r: (f"{r['lang']}:{r['id']}", r["text"], r["summary"])
    same = {s[0] for s in {*map(sig, old)} & {*map(sig, pool)}}
    for p in DIRS["data"].glob("weights*.jsonl"):
        rows = jl_read(p)
        keep = [w for w in rows if w["key"] in same]
        if len(keep) < len(rows):
            jl_write(p, keep)


def load_set(name):
    """The articles in use: the first n_use per language, taken evenly from the tertiles."""
    rows = jl_read(DIRS["data"] / f"{name}.jsonl")
    n = (CFG.get("n_use") or {}).get(name)
    if not n:
        return rows
    out = []
    for lang in LANGS:
        groups = {}
        for r in rows:
            if r["lang"] == lang:
                groups.setdefault(r["tertile"], []).append(r)
        longest = max(map(len, groups.values()), default=0)
        order = [g[i] for i in range(longest) for g in groups.values() if i < len(g)]
        out += order[:n]
    return out


def items(records, variants=None):
    """Expand article records into (item_id, record, variant) generation inputs."""
    out = []
    for r in records:
        for v, _ in r["variants"].items():
            if variants is None or v in variants:
                out.append((f"{r['uid']}|{v}", r, v))
    return out

## Indic to English translation

IndicTrans2 in its own environment, with a translation cache.

### `factindic/it2_env.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/it2_env.py
"""Builds the isolated environment IndicTrans2 needs.

IndicTrans2 cannot run on transformers 5 (needed by Gemma-4): its toolkit and tokenizer import
`transformers.tokenization_utils`, its config imports `transformers.onnx` (both removed), and its
model code uses the legacy tuple cache. A small venv with upstream's own pin (4.53.2) sidesteps all
three, and the rest of the project stays on one environment.
"""
from __future__ import annotations

import hashlib
import importlib.util
import os
import shutil
import subprocess
import sys
from contextlib import contextmanager
from pathlib import Path

from .config import IS_LOCAL, REPO

LOCK = Path(__file__).with_name("requirements-it2.txt")
SERVER = Path(__file__).with_name("it2_server.py")
ENV_DIR = Path(os.environ.get("FACTINDIC_IT2_ENV") or (REPO / ".it2_env" if IS_LOCAL else "/tmp/factindic_it2_env"))
TORCH_LOCAL, INDEX_LOCAL = "torch==2.8.0", "https://download.pytorch.org/whl/cu128"   # the locked 2.6 has no RTX 50 kernels


def clean_env():
    """Parent environment without the settings that would leak the notebook's packages into the venv."""
    drop = {"PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "PIP_CONSTRAINT", "PIP_TARGET"}
    env = {k: v for k, v in os.environ.items() if k not in drop and not k.startswith("UV_")}
    env.update(PYTHONNOUSERSITE="1", PYTHONUTF8="1", HF_HUB_ENABLE_HF_TRANSFER="0", TOKENIZERS_PARALLELISM="false",
               HF_MODULES_CACHE=str(ENV_DIR) + "_modules")
    return env


def _run(cmd, what):
    p = subprocess.run([str(c) for c in cmd], env=clean_env(), capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f"IndicTrans2 environment: {what} failed\n{(p.stdout + p.stderr)[-3000:]}")
    return p.stdout


def _uv():
    if shutil.which("uv"):
        return ["uv"]
    if importlib.util.find_spec("uv") is None:
        _run([sys.executable, "-m", "pip", "install", "-q", "uv"], "installing uv")
    return [sys.executable, "-m", "uv"]


@contextmanager
def _lock(path):
    with open(path, "w") as lk:
        try:
            import fcntl
            fcntl.flock(lk, fcntl.LOCK_EX)
        except ImportError:              # Windows: one job at a time there, so no lock is needed
            pass
        yield


def ensure():
    """Create the venv once (safe to call from several workers); returns its python path."""
    py = ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    stamp = ENV_DIR / ".ready"
    digest = hashlib.md5(LOCK.read_bytes() + (TORCH_LOCAL if IS_LOCAL else "").encode()).hexdigest()
    with _lock(str(ENV_DIR) + ".lock"):
        if py.exists() and stamp.exists() and stamp.read_text() == digest:
            return py
        if ENV_DIR.exists():
            shutil.rmtree(ENV_DIR)
        uv = _uv()
        print("building the IndicTrans2 environment (2-4 minutes, once per session)...", flush=True)
        try:
            _run([*uv, "venv", "--python", "3.11", ENV_DIR], "creating a Python 3.11 venv")
        except RuntimeError as e:
            print("Python 3.11 unavailable, using the notebook's Python:", str(e)[-200:])
            shutil.rmtree(ENV_DIR, ignore_errors=True)
            _run([*uv, "venv", "--python", sys.executable, ENV_DIR], "creating a venv")
        lock = LOCK
        if IS_LOCAL:                     # the GPU packages come with their own CUDA 12.8 build of torch
            _run([*uv, "pip", "install", "--python", py, TORCH_LOCAL, "--index-url", INDEX_LOCAL], "installing torch")
            lock = ENV_DIR / "requirements-local.txt"
            lock.write_text("\n".join(x for x in LOCK.read_text(encoding="utf-8").splitlines()
                                      if not x.startswith(("torch==", "triton==", "sympy==", "nvidia-", "#"))), encoding="utf-8")
        _run([*uv, "pip", "install", "--python", py, "--no-deps", "-r", lock], "installing packages")
        _run([py, "-c", "import torch; from transformers import AutoModelForSeq2SeqLM; "
                        "from IndicTransToolkit.processor import IndicProcessor; IndicProcessor(inference=True); "
                        "assert torch.cuda.is_available(), 'no CUDA in the IndicTrans2 environment'; "
                        "torch.ones(8, device='cuda').sum().item()"], "import check")
        stamp.write_text(digest)
    return py

### `factindic/it2_server.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/it2_server.py
"""IndicTrans2 worker. Runs inside its own pinned environment (see it2_env.py), never imported by the
package. Protocol: one JSON object per line on stdin/stdout; everything libraries print goes to stderr."""
import json
import os
import sys

_out = os.fdopen(os.dup(1), "w", encoding="utf-8", buffering=1)
os.dup2(2, 1)                       # library prints must not corrupt the protocol stream


def send(obj):
    _out.write(json.dumps(obj, ensure_ascii=False) + "\n")
    _out.flush()


def main(model_id):
    import torch
    from IndicTransToolkit.processor import IndicProcessor
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_id, trust_remote_code=True, torch_dtype=torch.float16, attn_implementation="eager").cuda().eval()
    send(dict(ready=True, torch=torch.__version__, model=model_id))

    @torch.no_grad()
    def run(sents, src, retry):
        ip = IndicProcessor(inference=True)      # fresh per batch: it queues placeholders between pre and post
        batch = ip.preprocess_batch(sents, src_lang=src, tgt_lang="eng_Latn")
        enc = tok(batch, truncation=True, padding="longest", max_length=256, return_tensors="pt",
                  return_attention_mask=True).to("cuda")
        out = model.generate(**enc, use_cache=True, min_length=0, max_length=256, num_beams=4,
                             num_return_sequences=1, **(dict(no_repeat_ngram_size=4) if retry else {}))
        dec = tok.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        return ip.postprocess_batch(dec, lang="eng_Latn")

    def safe(sents, src, retry):
        try:
            return run(sents, src, retry)
        except torch.cuda.OutOfMemoryError:      # not stored: its traceback would keep the failed batch on the GPU
            if len(sents) == 1:
                raise
        torch.cuda.empty_cache()
        mid = len(sents) // 2
        return safe(sents[:mid], src, retry) + safe(sents[mid:], src, retry)

    for line in open(0, encoding="utf-8", closefd=False):
        if not line.strip():
            continue
        try:
            req = json.loads(line)
            send(dict(out=safe(req["sents"], req["src"], req.get("retry", False))))
        except Exception as e:
            send(dict(error=repr(e)[:1500]))


if __name__ == "__main__":
    try:
        main(sys.argv[1])
    except Exception as e:
        import traceback
        traceback.print_exc()
        send(dict(fatal=repr(e)[:1500]))
        sys.exit(1)

### `factindic/mt.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/mt.py
"""Indic→English projection with IndicTrans2-dist-200M, which runs as a helper process in its own
environment (it2_env.py). Native digits are mapped to ASCII before translation so numbers are copied
rather than re-generated; all translations are cached on disk.

There is no silent fallback: mixing translators inside one experiment would confound H3. To run the
whole project on NLLB-200 instead, set CFG["mt_backend"]="nllb" or the env var FACTINDIC_MT=nllb.
"""
from __future__ import annotations

import atexit
import json
import os
import re
import subprocess

from .config import CFG, DIRS, LANGS, MODELS
from .io import JsonlAppender, chunked, jl_read_glob, md5
from .normalizer import DIGITS
from .text import INITIALS

_INIT = "|".join(sorted(INITIALS, key=len, reverse=True))
_CHAIN = re.compile(rf"(?<![\u0900-\u0AFF])(?:(?:{_INIT})\.)+(?:{_INIT})(?![\u0900-\u0AFF])\.?")
_GLUE = re.compile(r"([\u0900-\u0AFF][.।!?])(?=[^\s.,।!?\"'”’)\]])")
_LOOP = re.compile(r"(?:^|\s)(\S(?:.{0,40}?\S)?)(?:\s+\1){3,}(?=\s|$)")


def prepare(s):
    """What the translator sees: ASCII digits; initials without dots and a space after every full stop,
    because IndicTrans2's preprocessor reads 'છે.પાણી' and 'એલ.સી.બી' as web addresses and copies them."""
    s = s.translate(DIGITS)
    s = _CHAIN.sub(lambda m: m.group(0).replace(".", ""), s)
    return _GLUE.sub(r"\1 ", s)


_CHARS = re.compile(r"([^\w\s])\1{7,}")


def runaway(en):
    """Decoder loop: the same phrase four or more times in a row ('"" "" "" ""') or a run of one mark."""
    return _LOOP.search(en) is not None or _CHARS.search(en) is not None


def trim(en):
    """The translation before the loop started."""
    en = _CHARS.sub(r"\1", en)
    m = _LOOP.search(en)
    if m:
        en = en[:m.start()].strip() or m.group(1)
    return en


def lost(en):
    """A loop that left almost no translation before it: worth a second attempt."""
    return runaway(en) and len(trim(en).split()) < 4


class _IT2:
    """Client side of it2_server.py."""

    def __init__(self):
        from . import it2_env
        py = it2_env.ensure()
        DIRS["logs"].mkdir(parents=True, exist_ok=True)
        self.log = DIRS["logs"] / f"it2_{os.getpid()}.log"
        with open(self.log, "w") as err:
            self.p = subprocess.Popen([str(py), "-u", str(it2_env.SERVER), MODELS["it2"]], env=it2_env.clean_env(),
                                      stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=err,
                                      text=True, encoding="utf-8", bufsize=1)
        atexit.register(self.close)
        self._read()

    def _read(self):
        line = self.p.stdout.readline()
        msg = json.loads(line) if line.strip() else {"fatal": "the process exited"}
        if "fatal" in msg or "error" in msg:
            tail = "\n".join(self.log.read_text(errors="ignore").splitlines()[-15:])
            hint = ""
            if any(s in (tail + str(msg)).lower() for s in ("gated", "401", "403")):
                hint = f"\n→ Accept the terms at https://huggingface.co/{MODELS['it2']} with the account that owns HF_TOKEN."
            if "fatal" in msg:
                self.close()
            raise RuntimeError(f"IndicTrans2: {msg.get('fatal') or msg.get('error')}{hint}\n--- {self.log} ---\n{tail}")
        return msg

    def run(self, sents, lang, retry=False):
        self.p.stdin.write(json.dumps(dict(src=LANGS[lang]["it2"], sents=sents, retry=retry), ensure_ascii=False) + "\n")
        self.p.stdin.flush()
        out = self._read()["out"]
        assert len(out) == len(sents), "IndicTrans2 returned a different number of sentences"
        return out

    def close(self):
        if self.p.poll() is None:
            try:
                self.p.stdin.close()
                self.p.wait(timeout=20)
            except Exception:
                self.p.kill()


class _NLLB:
    def __init__(self):
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(MODELS["nllb"])
        self.model = AutoModelForSeq2SeqLM.from_pretrained(MODELS["nllb"], dtype=torch.float16).cuda().eval()

    def run(self, sents, lang, retry=False):
        import torch
        self.tok.src_lang = LANGS[lang]["it2"]
        enc = self.tok(sents, truncation=True, padding="longest", return_tensors="pt", max_length=256).to("cuda")
        with torch.no_grad():
            out = self.model.generate(**enc, num_beams=4, max_length=256, **(dict(no_repeat_ngram_size=4) if retry else {}),
                                      forced_bos_token_id=self.tok.convert_tokens_to_ids("eng_Latn"))
        return self.tok.batch_decode(out, skip_special_tokens=True)

    def close(self):
        self.model = None


class Translator:
    def __init__(self, backend=None, cache_name="mt_cache.jsonl"):
        self.backend = backend or os.environ.get("FACTINDIC_MT") or CFG["mt_backend"]
        if self.backend not in ("it2", "nllb"):
            raise ValueError(f"unknown MT backend {self.backend!r}")
        self._engine = None                          # started on the first uncached sentence
        self.cache_file = JsonlAppender(DIRS["cache"] / cache_name, key_fn=lambda r: r["k"])
        self.cache, self.trimmed = {}, set()
        for r in jl_read_glob(DIRS["cache"] / cache_name):
            if r.get("b") == self.backend and (r.get("r") or not lost(r["en"])):
                self.cache[r["k"]] = trim(r["en"])
                if self.cache[r["k"]] != r["en"]:
                    self.trimmed.add(r["k"])

    def _key(self, lang, s):
        return md5(self.backend, lang, s)

    def has(self, sents, lang):
        """True if every sentence has a cached translation that needed no repair."""
        ks = [self._key(lang, p) for p in map(prepare, sents) if p.strip()]
        return all(k in self.cache and k not in self.trimmed for k in ks)

    def _run(self, sents, lang, retry=False):
        if self._engine is None:
            self._engine = _IT2() if self.backend == "it2" else _NLLB()
        return self._engine.run(sents, lang, retry)

    def translate(self, sents, lang, bs=32):
        sents = [prepare(s) for s in sents]
        todo = sorted({s for s in sents if s.strip() and self._key(lang, s) not in self.cache}, key=len)
        for batch in chunked(todo, bs):
            res = list(zip(batch, self._run(batch, lang), [False] * len(batch)))
            loops = [s for s, e, _ in res if lost(e)]
            if loops:
                again = dict(zip(loops, self._run(loops, lang, retry=True)))
                res = [(s, again[s], True) if s in again else (s, e, r) for s, e, r in res]
            for s, e, r in res:
                e = trim(e)
                k = self._key(lang, s)
                self.cache[k] = e
                self.cache_file.add(dict(k=k, en=e, b=self.backend, **(dict(r=1) if r else {})))
        return [self.cache.get(self._key(lang, s), "") if s.strip() else "" for s in sents]

    def close(self):
        self.cache_file.close()
        if self._engine is not None:
            self._engine.close()
            self._engine = None

## Verifiers and metrics

MiniCheck, mDeBERTa-XNLI, FactCG (held out for evaluation), the round-trip router, reranking, and the P/R/F1 metrics.

### `factindic/verify.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/verify.py
"""Grounded verifiers and the Stage-4 Round-Trip Anchor Verification Router (spec §8.4).

MiniCheck-FT5   English, used inside Stage 4 (router route 1) and for mFACT weights.
mDeBERTa-XNLI   native multilingual NLI, router route 2.
FactCG-DeBERTa  English, HELD OUT: only for final evaluation (never weights or reranking).
"""
from __future__ import annotations

import torch

from .config import IS_LOCAL, MODELS
from .normalizer import ent_key, numbers, _close
from .text import chunk_sentences, split_sentences


def _split_on_oom(f):
    """score_pairs with the batch size halved until it fits."""
    def run(self, pairs, bs=None):
        bs = bs or (4 if IS_LOCAL else 8)
        while True:
            try:
                return f(self, pairs, bs)
            except torch.cuda.OutOfMemoryError:
                if bs == 1:
                    raise
            torch.cuda.empty_cache()
            bs //= 2
    return run


class MiniCheck:
    """First decoder step over token 3 ('▁', which begins "0") vs 209 ('1'); sanity-checked."""
    name = "minicheck"

    def __init__(self):
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(MODELS["minicheck"])
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            MODELS["minicheck"], dtype=torch.bfloat16 if IS_LOCAL else torch.float32).cuda().eval()
        self.ids = [3, 209]
        hi = self.score_pairs([("The district hospital in Surat admitted 42 patients on Monday.",
                                "42 patients were admitted to a hospital in Surat.")])[0]
        lo = self.score_pairs([("The district hospital in Surat admitted 42 patients on Monday.",
                                "The gas leak killed 300 people in Delhi.")])[0]
        assert hi > 0.5 > lo, f"MiniCheck sanity failed ({hi:.2f}, {lo:.2f})"

    @_split_on_oom
    @torch.no_grad()
    def score_pairs(self, pairs, bs):
        out = []
        for i in range(0, len(pairs), bs):
            texts = [f"premise: {d} hypothesis: {c}" for d, c in pairs[i:i + bs]]
            enc = self.tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to("cuda")
            dec = torch.full((enc["input_ids"].shape[0], 1), self.model.config.decoder_start_token_id, device="cuda")
            lg = self.model(**enc, decoder_input_ids=dec).logits[:, 0, self.ids]
            out += torch.softmax(lg.float(), -1)[:, 1].tolist()
        return out


class NativeNLI:
    name = "mdeberta"

    def __init__(self):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(MODELS["mdeberta"])
        self.model = AutoModelForSequenceClassification.from_pretrained(MODELS["mdeberta"], dtype=torch.float32).cuda().eval()
        self.ent = [i for i, l in self.model.config.id2label.items() if l.lower().startswith("entail")][0]
        src = "सूरत के ज़िला अस्पताल में सोमवार को 42 मरीज़ भर्ती हुए।"
        same, hi, lo = self.score_pairs([(src, src), (src, "सूरत के अस्पताल में 42 मरीज़ भर्ती हुए।"),
                                         (src, "दिल्ली में गैस रिसाव से 300 लोगों की मौत हुई।")])
        print(f"mDeBERTa check: same {same:.2f}, paraphrase {hi:.2f}, unrelated {lo:.2f}", flush=True)
        assert min(same, hi) > lo + 0.2, f"mDeBERTa sanity failed ({same:.2f}, {hi:.2f}, {lo:.2f})"

    @_split_on_oom
    @torch.no_grad()
    def score_pairs(self, pairs, bs):
        out = []
        for i in range(0, len(pairs), bs):
            p = pairs[i:i + bs]
            enc = self.tok([d for d, _ in p], [c for _, c in p], return_tensors="pt", padding=True,
                           truncation="only_first", max_length=512).to("cuda")
            out += torch.softmax(self.model(**enc).logits.float(), -1)[:, self.ent].tolist()
        return out


FACTCG_TEMPLATE = ("{text_a}\n\nChoose your answer: based on the paragraph above can we conclude that "
                   "\"{text_b}\"?\n\nOPTIONS:\n- Yes\n- No\nI think the answer is ")


class FactCG:
    """Template, label index 1 = supported, and max over sentence-packed chunks follow
    github.com/derenlei/FactCG (factcg/inference.py)."""
    name = "factcg"

    def __init__(self):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(MODELS["factcg"], use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            MODELS["factcg"], num_labels=2, dtype=torch.float32).cuda().eval()
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        src = "The district hospital in Surat admitted 42 patients on Monday."
        hi, lo = self.score_pairs([(src, "42 patients were admitted to a hospital in Surat."),
                                   (src, "The gas leak killed 300 people in Delhi.")])
        assert hi > 0.5 > lo, f"FactCG sanity failed ({hi:.2f}, {lo:.2f})"

    @_split_on_oom
    @torch.no_grad()
    def score_pairs(self, pairs, bs):
        out = []
        for i in range(0, len(pairs), bs):
            texts = [FACTCG_TEMPLATE.format(text_a=d, text_b=c) for d, c in pairs[i:i + bs]]
            enc = self.tok(texts, return_tensors="pt", padding="longest", truncation=True, max_length=2048).to("cuda")
            out += torch.softmax(self.model(**enc).logits.float(), -1)[:, 1].tolist()
        return out


def max_over_chunks(scorer, chunks, claims, cache=None):
    """p(D, c) = max_j P(Supported | D_j, c)."""
    pairs, owners = [], []
    res = [0.0] * len(claims)
    for ci, c in enumerate(claims):
        if not c:
            continue
        for ch in chunks:
            key = (scorer.name, ch, c)
            if cache is not None and key in cache:
                res[ci] = max(res[ci], cache[key])
            else:
                pairs.append((ch, c)); owners.append((ci, key))
    if pairs:
        for (ci, key), p in zip(owners, scorer.score_pairs(pairs)):
            if cache is not None:
                cache[key] = p
            res[ci] = max(res[ci], p)
    return res


def source_for_variant(rec, variant, mode="deployment"):
    """Text the verifier treats as D. Deployment: the input variant (Romanized → back-transliterated
    to native). Oracle: the aligned native article (analysis only, H3 proxy)."""
    from .translit import back_transliterate
    if mode == "oracle" or variant in ("native", "natural_cm"):
        return rec["variants"].get("native", rec["text"])
    if variant == "roman":
        return back_transliterate(rec["variants"]["roman"], rec["lang"])
    return rec["variants"][variant]


class Router:
    """Route 1 (translate-then-verify, MiniCheck) when every number and entity key of the native
    claim survives MT; otherwise Route 2 (native mDeBERTa against native-script source chunks)."""

    def __init__(self, translator, minicheck, nli):
        self.mt, self.mc, self.nli = translator, minicheck, nli
        self.src_cache, self.score_cache = {}, {}

    def prepare_source(self, rec, variant, mode="deployment"):
        key = (rec["uid"], variant, mode)
        if key not in self.src_cache:
            D = source_for_variant(rec, variant, mode)
            sents = split_sentences(D)
            en = self.mt.translate(sents, rec["lang"])
            self.src_cache[key] = dict(native_chunks=chunk_sentences(sents, 180),
                                       en_chunks=chunk_sentences(en, 250), text=D)
        return self.src_cache[key]

    @staticmethod
    def survived(claim, claim_en, rec):
        if not numbers(claim) <= numbers(claim_en):
            return False
        src_keys = {k for e in rec["anchors"].get("entities", []) for k in e["key"].split()}
        claim_ent = {ent_key(w, rec["lang"]) for w in claim.split()}
        claim_ent = {k for k in claim_ent if k and _close(k, src_keys)}
        en_keys = {ent_key(w, "en") for w in claim_en.split()} - {""}
        return all(_close(k, en_keys) for k in claim_ent)

    def verify(self, rec, variant, claims, mode="deployment", routes="router"):
        src = self.prepare_source(rec, variant, mode)
        claims_en = self.mt.translate(claims, rec["lang"]) if claims else []
        p_mc = max_over_chunks(self.mc, src["en_chunks"], claims_en, self.score_cache)
        p_nli = max_over_chunks(self.nli, src["native_chunks"], claims, self.score_cache)
        out = []
        for c, ce, a, b in zip(claims, claims_en, p_mc, p_nli):
            ok = self.survived(c, ce, rec)
            out.append(dict(claim=c, claim_en=ce, p_minicheck=a, p_nli=b, survived=ok,
                            route="minicheck" if ok else "nli", p=a if ok else b))
        return out

### `factindic/claims.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/claims.py
"""Claim units. Default: native summary sentences (MiniCheck/FactCG are sentence-level verifiers).
Optional: Qwen2.5-1.5B molecular decomposition of the English projection (Gunjal & Durrett, 2024)."""
from __future__ import annotations

import re

from .text import W, split_sentences


def sentence_claims(summary, min_words=3):
    return [c for c in split_sentences(summary) if W(c) >= min_words]


MOLECULAR_PROMPT = (
    "Split the sentence into the smallest set of self-contained factual claims. Each claim must be "
    "understandable on its own (replace pronouns with the names they refer to), keep multi-word "
    "predicates intact, and add nothing that is not in the sentence. One claim per line, no "
    "numbering.\n\nSentence: {s}\nClaims:")


class MolecularDecomposer:
    def __init__(self):
        from .config import MODELS
        from .llm import load_causal
        self.model, self.tok, _ = load_causal(MODELS["qwen"])

    def __call__(self, sentences_en, bs=8):
        from .llm import chat_text, generate
        out = []
        for i in range(0, len(sentences_en), bs):
            batch = sentences_en[i:i + bs]
            res, _, _ = generate(self.model, self.tok,
                                 [chat_text(self.tok, MOLECULAR_PROMPT.format(s=s)) for s in batch],
                                 max_new_tokens=128)
            for s, r in zip(batch, res):
                raw = r[0] if r else ""
                lines = [re.sub(r"^[\-\*\d\.\)\s]+", "", l).strip() for l in raw.split("\n")]
                lines = [l for l in lines if len(l.split()) >= 3]
                out.append(lines or [s])
        return out

### `factindic/rerank.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/rerank.py
"""Stage 4 single-round candidate reranking and capped pruning (spec §8.4.3–4), plus the
drop-only repair baseline (Exp 5a)."""
from __future__ import annotations

from .normalizer import anchor_recall_salient, anchors, salient_anchors
from .text import W


def J_score(cand_summary, verdicts, rec, tau, lam, ner=None):
    """J(S_k) = (frac claims with p ≥ τ) · R_anchor(S_k, D) − λ_len·|W(S_k) − L|/L."""
    frac = (sum(v["p"] >= tau for v in verdicts) / len(verdicts)) if verdicts else 0.0
    a_s = anchors(cand_summary, rec["lang"], ner.entities(cand_summary) if ner else None)
    sal = salient_anchors(rec["anchors"], rec["text"], rec["lang"])
    r_anchor = anchor_recall_salient(a_s, rec["anchors"], sal, rec["text"], rec["lang"])
    L = rec["L_target"]
    j = frac * r_anchor - lam * abs(W(cand_summary) - L) / L
    return dict(J=j, frac_supported=frac, R_anchor=r_anchor, words=W(cand_summary))


def prune(verdicts, tau):
    """Drop unsupported sentences only if at least one verified sentence remains (single pass)."""
    keep = [v["claim"] for v in verdicts if v["p"] >= tau]
    if keep and len(keep) < len(verdicts):
        return " ".join(keep), True
    return " ".join(v["claim"] for v in verdicts), False


def drop_only(verdicts, tau):
    """Delete every sentence the verifier rejects, even if nothing is left (recall-collapse baseline)."""
    return " ".join(v["claim"] for v in verdicts if v["p"] >= tau)

### `factindic/metrics.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/metrics.py
"""Evaluation metrics (spec §10.2): claim-level and anchor-level P/R/F1, Indic-tokenized ROUGE,
rescaled IndicBERTv2 BERTScore (semantic similarity only), length / compression."""
from __future__ import annotations

import random

import numpy as np
import torch

from .text import PUNCT, W


class IndicTok:
    def tokenize(self, text):
        from indicnlp.tokenize import indic_tokenize
        return [t.lower() for t in indic_tokenize.trivial_tokenize(text or "")
                if not all(ch in PUNCT for ch in t)]


_ROUGE = None


def rouge(ref, hyp):
    """Default rouge-score strips non-ASCII; an Indic tokenizer is injected (0–100 scale)."""
    global _ROUGE
    if _ROUGE is None:
        from rouge_score import rouge_scorer
        _ROUGE = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=IndicTok())
    if not hyp:
        return dict(rouge1=0.0, rouge2=0.0, rougeL=0.0)
    s = _ROUGE.score(ref, hyp)
    return {k: 100 * v.fmeasure for k, v in s.items()}


class BertScorer:
    """Greedy cosine matching on a middle-upper IndicBERTv2 layer, rescaled with a baseline b
    estimated from random unrelated (candidate, reference) pairs: F' = (F − b)/(1 − b)."""

    def __init__(self, layer=9):
        from transformers import AutoModel, AutoTokenizer
        from .config import MODELS
        self.tok = AutoTokenizer.from_pretrained(MODELS["indicbert"])
        self.model = AutoModel.from_pretrained(MODELS["indicbert"], output_hidden_states=True, dtype=torch.float32).cuda().eval()
        self.layer = layer
        self.baseline = {}

    @torch.no_grad()
    def _emb(self, texts):
        enc = self.tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        hs = self.model(**enc).hidden_states[self.layer]
        hs = torch.nn.functional.normalize(hs.float(), dim=-1)
        special = torch.zeros_like(enc["attention_mask"], dtype=torch.bool)
        for sid in self.tok.all_special_ids:
            special |= enc["input_ids"] == sid
        mask = enc["attention_mask"].bool() & ~special
        return [hs[i][mask[i]] for i in range(len(texts))]

    def _f(self, cands, refs, bs=32):
        out = []
        for i in range(0, len(cands), bs):
            ec, er = self._emb(cands[i:i + bs]), self._emb(refs[i:i + bs])
            for c, r in zip(ec, er):
                if len(c) == 0 or len(r) == 0:
                    out.append(0.0)
                    continue
                sim = c @ r.T
                p = sim.max(1).values.mean().item()
                rr = sim.max(0).values.mean().item()
                out.append(2 * p * rr / (p + rr) if p + rr > 0 else 0.0)
        return out

    def fit_baseline(self, lang, texts, n=500, seed=42):
        rng = random.Random(seed)
        a = [rng.choice(texts) for _ in range(n)]
        b = [rng.choice(texts) for _ in range(n)]
        self.baseline[lang] = float(np.mean(self._f(a, b)))
        return self.baseline[lang]

    def score(self, cands, refs, lang):
        f = self._f([c or "" for c in cands], refs)
        b = self.baseline.get(lang, 0.0)
        return [(x - b) / (1 - b) for x in f]


def fact_prf(p_claims, r_grounded_hits):
    """P_fact = mean I_eval(D, c_i); R_fact = mean I_eval(S, r_j) over C_grounded(R)."""
    P = float(np.mean(p_claims)) if len(p_claims) else float("nan")
    R = float(np.mean(r_grounded_hits)) if len(r_grounded_hits) else float("nan")
    F = 2 * P * R / (P + R) if (P == P and R == R and P + R > 0) else (0.0 if P == 0 or R == 0 else float("nan"))
    return P, R, F


def compression_ratio(summary, source):
    return 100.0 * W(summary) / max(1, W(source))

## Generation: prompts, LLM loading, LoRA SFT

Prompt conditions A, B1, B2, B3 and faithfulness-weighted LoRA fine-tuning.

### `factindic/llm.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/llm.py
"""Causal LM loading with T4-safe precision fallbacks, chat formatting, batched generation."""
from __future__ import annotations

import gc
import re
import time

import torch

from .config import IS_LOCAL, safe_attention

DEV = "cuda"


def load_tokenizer(model_id, padding_side="left"):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = padding_side
    return tok


def chat_text(tok, user, assistant=None):
    msgs = [{"role": "user", "content": user}]
    if assistant is not None:
        msgs.append({"role": "assistant", "content": assistant})
    kw = dict(tokenize=False, add_generation_prompt=assistant is None)
    try:
        return tok.apply_chat_template(msgs, enable_thinking=False, **kw)
    except TypeError:
        return tok.apply_chat_template(msgs, **kw)


def _bnb4(compute):
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute)


def _trials(model_id):
    if IS_LOCAL and torch.cuda.is_bf16_supported(including_emulation=False):     # RTX 40/50, 8 GB
        if "gemma" in model_id.lower():
            return [("bf16, per-layer embeddings on the CPU", "gemma_split")]
        return [("4bit-nf4/bf16", dict(quantization_config=_bnb4(torch.bfloat16), dtype=torch.bfloat16))]
    if "gemma" in model_id.lower():   # bf16-trained; fp16 compute overflows → fp32 compute (H1-proven)
        return [("4bit-nf4/fp32, per-layer embeddings fp16", dict(quantization_config=_bnb4(torch.float32), dtype=torch.float16))]
    if "llama" in model_id.lower() or "qwen" in model_id.lower():
        return [("fp16", dict(dtype=torch.float16)),
                ("4bit-nf4/fp32-compute", dict(quantization_config=_bnb4(torch.float32), dtype=torch.float16))]
    return [("fp32", dict(dtype=torch.float32))]


def _fp32_stream(model):
    """bitsandbytes layers compute in fp32 but hand back the dtype they were given. In fp16 Gemma's residual
    stream overflows in padded batches (NaN), so everything except the per-layer embedding table becomes fp32."""
    for n, p in model.named_parameters():
        if p.dtype == torch.float16 and p.__class__.__name__ != "Params4bit" and \
                not any(x in n for x in ("embed_tokens_per_layer", "vision", "audio")):
            p.data = p.data.to(torch.float32)
    torch.cuda.empty_cache()


def _gemma_split(model_id, **kw):
    """Gemma-4 E2B in bf16 on an 8 GB card. Its per-layer embedding table (4.7 GB) stays in RAM, where looking
    up a few rows is cheap; the unused vision and audio towers are dropped. About 5 GB of GPU memory."""
    from transformers import AutoModelForCausalLM
    lm = "model.language_model."
    ple_name = lm + "embed_tokens_per_layer"
    dm = {lm + x: 0 for x in ("embed_tokens", "layers", "norm", "rotary_emb", "per_layer_model_projection", "per_layer_projection_norm")}
    dm.update({"lm_head": 0, ple_name: "cpu"}, **{f"model.{x}": "cpu" for x in ("vision_tower", "audio_tower", "embed_vision", "embed_audio")})
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map=dm, **kw)
        from accelerate.hooks import remove_hook_from_module
        remove_hook_from_module(model, recurse=True)      # plain modules again: accelerate would copy the table to the GPU
    except Exception as e:
        print("split loading failed, loading through RAM (needs ~11 GB):", repr(e)[:300], flush=True)
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, **kw)
    inner = model.model
    for x in ("vision_tower", "audio_tower", "embed_vision", "embed_audio"):
        if hasattr(inner, x):
            setattr(inner, x, None)
    text = inner.language_model
    for name, child in text.named_children():
        child.to("cpu" if name == "embed_tokens_per_layer" else DEV)
    model.lm_head.to(DEV)
    if hasattr(model, "hf_device_map"):
        del model.hf_device_map                            # PEFT would otherwise dispatch the model again
    ple = text.embed_tokens_per_layer
    ple.register_forward_pre_hook(lambda m, a: (a[0].cpu(),))
    ple.register_forward_hook(lambda m, a, out: out.to(DEV))
    gc.collect(); torch.cuda.empty_cache()
    meta = [n for n, p in model.named_parameters() if p.device.type == "meta"]
    assert not meta, f"weights not loaded: {meta[:5]}"
    return model


def load_causal(model_id, adapter=None, attn_implementation=None, trials=None):
    """Returns (model, tok, precision_desc). Keeps the first config whose logits are finite."""
    import transformers as tf
    safe_attention()
    tok = load_tokenizer(model_id)
    classes = [getattr(tf, c) for c in ("AutoModelForCausalLM", "AutoModelForImageTextToText",
                                        "AutoModelForMultimodalLM") if hasattr(tf, c)]
    probe = tok([chat_text(tok, "Language: Gujarati\nArticle: ગુજરાતમાં 23 એપ્રિલે મતદાન થશે.\nSummary:")],
                return_tensors="pt", add_special_tokens=False)
    last = None
    for desc, kw in (trials or _trials(model_id)):
        split = kw == "gemma_split"
        kw = {} if split else kw
        if attn_implementation:
            kw = dict(kw, attn_implementation=attn_implementation)
        for cls in (classes[:1] if split else classes):
            try:
                try:
                    if split:
                        model = _gemma_split(model_id, **kw)
                    else:
                        model = cls.from_pretrained(model_id, device_map={"": 0}, **kw)
                except TypeError:   # transformers 4.x uses torch_dtype
                    if split:
                        raise
                    kw2 = {("torch_dtype" if k == "dtype" else k): v for k, v in kw.items()}
                    model = cls.from_pretrained(model_id, device_map={"": 0}, **kw2)
            except Exception as e:
                last = e
                print(f"{cls.__name__} [{desc}] failed: {repr(e)[:300]}", flush=True)
                continue
            if "gemma" in model_id.lower() and not split:
                _fp32_stream(model)
            if adapter:
                from peft import PeftModel
                model = PeftModel.from_pretrained(model, adapter)
            model.eval()
            with torch.no_grad():
                lg = model(**{k: v.to(DEV) for k, v in probe.items()}).logits
            if torch.isfinite(lg).all():
                return model, tok, desc
            del model
            torch.cuda.empty_cache()
            break
    raise RuntimeError(f"could not load {model_id} with finite logits: {repr(last)[:300]}")


def clean_output(text):
    text = re.sub(r"(?s)^.*?</think>", "", text)
    text = re.sub(r"(?im)^\s*(summary|सारांश|સારાંશ)\s*[:：]\s*", "", text.strip())
    text = text.replace("**", "").replace("##", "")
    text = re.split(r"\n\s*\n", text.strip())[0]
    return re.sub(r"\s+", " ", text).strip()


def _eos(model, tok):
    from .decoding import eos_ids
    return eos_ids(model, tok)


@torch.no_grad()
def generate(model, tok, texts, max_new_tokens=160, do_sample=False, num_return_sequences=1,
             temperature=0.7, top_p=0.9, seed=None):
    """texts are already chat-templated. Returns (outputs[list[list[str]]], seconds, n_new_tokens).
    A batch that does not fit in memory is split in two."""
    args = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, num_return_sequences=num_return_sequences,
                temperature=temperature, top_p=top_p, seed=seed)
    try:
        return _generate(model, tok, texts, **args)
    except torch.cuda.OutOfMemoryError:
        if len(texts) == 1:
            raise
    torch.cuda.empty_cache()
    mid = len(texts) // 2
    a, b = generate(model, tok, texts[:mid], **args), generate(model, tok, texts[mid:], **args)
    return a[0] + b[0], a[1] + b[1], a[2] + b[2]


def _guard(stop, pad):
    """Logits processor: a finished row may hold anything (it only emits padding), but a non-finite score in a
    row that is still writing must not pass silently (greedy search would turn it into an empty summary)."""
    from transformers import LogitsProcessor

    class Guard(LogitsProcessor):
        n0 = None

        def __call__(self, ids, scores):
            self.n0 = self.n0 or ids.shape[1]
            bad = torch.isnan(scores).any(-1) | torch.isposinf(scores).any(-1)
            if bool(bad.any()):
                done = torch.isin(ids[:, self.n0:], torch.tensor([*stop, pad], device=ids.device)).any(-1)
                if bool((bad & ~done).any()):
                    raise RuntimeError("non-finite logits in an unfinished row")
                scores = scores.clone()
                scores[bad] = float("-inf")
                scores[bad, pad] = 0.0
            return scores
    return Guard()


def _generate(model, tok, texts, max_new_tokens, do_sample, num_return_sequences, temperature, top_p, seed):
    from transformers import LogitsProcessorList
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
    if seed is not None:
        torch.manual_seed(seed)
    kw = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tok.pad_token_id,
              num_return_sequences=num_return_sequences, eos_token_id=_eos(model, tok),
              logits_processor=LogitsProcessorList([_guard(_eos(model, tok), tok.pad_token_id)]))
    if do_sample:
        kw.update(temperature=temperature, top_p=top_p, top_k=0)
    else:
        kw.update(temperature=None, top_p=None, top_k=None)
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = model.generate(**enc, **kw)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t
    new = out[:, enc["input_ids"].shape[1]:]
    dec = tok.batch_decode(new, skip_special_tokens=True)
    n_new = int((new != tok.pad_token_id).sum())
    res = [[clean_output(dec[i * num_return_sequences + j]) for j in range(num_return_sequences)]
           for i in range(len(texts))]
    return res, dt, n_new


def n_tokens(tok, text):
    return len(tok(text, add_special_tokens=False)["input_ids"])


def lm_parts(model):
    """(decoder layers count, final norm, lm_head, softcap) for early-exit (DoLa) projections."""
    lm_head = model.get_output_embeddings()
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    norm = None
    for path in ("model.norm", "model.language_model.norm", "language_model.model.norm",
                 "model.text_model.norm", "language_model.norm"):
        obj = base
        try:
            for p in path.split("."):
                obj = getattr(obj, p)
            norm = obj
            break
        except AttributeError:
            continue
    cfg = getattr(base.config, "text_config", base.config)
    n_layers = cfg.num_hidden_layers
    softcap = getattr(cfg, "final_logit_softcapping", None)
    if norm is None:
        raise RuntimeError("final norm not found; DoLa unavailable for this architecture")
    return n_layers, norm, lm_head, softcap

### `factindic/prompts.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/prompts.py
"""Stage 2 prompt conditions (spec §8.2): A standard; B1 +anchor checklist; B2 +2-step Chain of
Density; B3 +matrix-verb hint. All conditions output the matrix language in native script."""
from __future__ import annotations

from .config import LANGS

VARIANT_NOTE = {
    "native": "",
    "script_mixed": "The article mixes {L} with English words in Latin script.",
    "roman": "The article is {L} written in Latin script (Romanized).",
    "natural_cm": "The article may mix {L} with English words.",
}


def _head(lang, variant, article):
    L = LANGS[lang]["name"]
    note = VARIANT_NOTE.get(variant, "").format(L=L)
    return f"Language: {L}\n" + (f"Note: {note}\n" if note else "") + f"Article: {article}\n"


def cond_A(lang, variant, article, n_words, **_):
    L, S = LANGS[lang]["name"], LANGS[lang]["script"]
    return (_head(lang, variant, article) +
            f"\nWrite a summary of the article in {L} ({S} script) in about {n_words} words. "
            f"Output only the summary.\nSummary:")


def _rules(lang):
    L, S = LANGS[lang]["name"], LANGS[lang]["script"]
    return (f"Rules: (1) use only information stated in the article and add nothing else; "
            f"(2) keep names, numbers and dates exactly as the article gives them; "
            f"(3) do not change who did what, causes, or negations; "
            f"(4) the article may mix {L} and English or be in Latin script — understand it, but write "
            f"the summary in {L} using {S} script.")


def checklist(rec, max_items=12):
    a = rec["anchors"]
    ents = [e["text"] for e in a.get("entities", [])][:8]
    nums = a.get("numbers", [])[:6]
    dts = a.get("dates", [])[:3]
    parts = []
    if ents:
        parts.append("entities: " + "; ".join(ents))
    if nums:
        parts.append("numbers: " + "; ".join(nums))
    if dts:
        parts.append("dates: " + "; ".join(dts))
    return " | ".join(parts)[:600] or "(none detected)"


def cond_B1(lang, variant, article, n_words, rec=None, **_):
    L, S = LANGS[lang]["name"], LANGS[lang]["script"]
    return (_head(lang, variant, article) + "\n" + _rules(lang) + "\n" +
            f"Source entities, numbers & dates (keep the most important ones; any quantity you derive "
            f"must follow from the article): {checklist(rec)}\n"
            f"Write a faithful summary in {L} ({S} script) in about {n_words} words. "
            f"Output only the summary.\nSummary:")


def cond_B2_step2(lang, variant, article, n_words, rec, draft, missing):
    L, S = LANGS[lang]["name"], LANGS[lang]["script"]
    miss = "; ".join(missing) if missing else "(none)"
    return (_head(lang, variant, article) + "\n" + _rules(lang) + "\n" +
            f"Draft summary: {draft}\n"
            f"Important source anchors missing from the draft: {miss}\n"
            f"Rewrite the draft as a denser summary of about {n_words} words in {L} ({S} script) that "
            f"also covers the missing anchors where they matter, without adding anything not in the "
            f"article. Output only the summary.\nSummary:")


def cond_B3(lang, variant, article, n_words, rec=None, **_):
    L = LANGS[lang]["name"]
    ex = ("'increase ho raha hai' = 'बढ़ रहा है'" if lang == "hi" else "'increase thai rahyu chhe' = 'વધી રહ્યું છે'")
    hint = (f"Grammar hint: the matrix language is {L} (Subject–Object–Verb, postpositions"
            f"{', attached case endings -માં -થી -ને -નો/-ની/-નું/-ના' if lang == 'gu' else ''}). English words "
            f"inside {L} clauses are content words of the {L} clause ({ex}); the {L} verb carries tense "
            f"and negation — keep them as in the article.\n")
    base = cond_B1(lang, variant, article, n_words, rec)
    return base.replace("Write a faithful summary", hint + "Write a faithful summary")


CONDITIONS = {"A": cond_A, "B1": cond_B1, "B3": cond_B3}   # B2 = B1 draft + step 2


def bare_prior_prompt(lang, n_words):
    """Uncontaminated prior for CAD: no document, no anchors (spec §8.3 step 1)."""
    L, S = LANGS[lang]["name"], LANGS[lang]["script"]
    return (f"Language: {L}\n\nWrite a summary of the article in {L} ({S} script) in about {n_words} "
            f"words. Output only the summary.\nSummary:")

### `factindic/sft.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/sft.py
"""Faithfulness-weighted QLoRA SFT (mFACT-style loss weighting w_i ∈ [0.3, 1.0]), resumable."""
from __future__ import annotations

import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch


def build_examples(tok, rows, prompt_fn, max_len):
    """rows: dicts with prompt args + 'summary' + 'w'. Labels mask the prompt; article shrinks to fit."""
    from .llm import chat_text
    ex = []
    for r in rows:
        art = r["article"].split()
        while True:
            prompt = chat_text(tok, prompt_fn(r, " ".join(art)))
            full = chat_text(tok, prompt_fn(r, " ".join(art)), assistant=r["summary"])
            p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
            f_ids = tok(full, add_special_tokens=False)["input_ids"]
            if len(f_ids) <= max_len or len(art) < 60:
                break
            art = art[: int(len(art) * 0.85)]
        if f_ids[:len(p_ids)] != p_ids:      # template adds something after the turn start: append the answer instead
            tail = full[full.rfind(r["summary"].strip()):]
            f_ids = p_ids + tok(tail, add_special_tokens=False)["input_ids"]
        f_ids = f_ids[:max_len]
        labels = [-100] * min(len(p_ids), len(f_ids)) + f_ids[len(p_ids):]
        ex.append((f_ids, labels[: len(f_ids)], float(r.get("w", 1.0))))
    return ex


def train_lora(model_id, examples, out_dir, cfg, seed=42):
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
    from transformers import get_cosine_schedule_with_warmup
    from .llm import load_causal

    out_dir = Path(out_dir)
    final, ckpt = out_dir / "final", out_dir / "ckpt"
    if (final / "adapter_config.json").exists():
        return final
    model, tok, desc = load_causal(model_id)
    # Like prepare_model_for_kbit_training, but Gemma-4's per-layer embedding (2.3 B values) stays in half
    # precision: in fp32 it alone would need 9 GB. The vision and audio towers are not used.
    for n, p in model.named_parameters():
        p.requires_grad = False
        if not desc.startswith("bf16") and p.dtype in (torch.float16, torch.bfloat16) and p.__class__.__name__ != "Params4bit" and \
                not any(x in n for x in ("embed_tokens_per_layer", "vision", "audio")):
            p.data = p.data.to(torch.float32)
    torch.cuda.empty_cache()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    # text layers only: the towers also have layers called q_proj
    targets = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)" if "gemma-4" in model_id.lower() else \
        ["q_proj", "k_proj", "v_proj", "o_proj"]
    model = get_peft_model(model, LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=cfg["dropout"],
                                             target_modules=targets, task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    order = list(range(len(examples)))
    random.Random(seed).shuffle(order)
    mbs, acc = cfg["micro_bs"], cfg["accum"]
    spe = math.ceil(len(examples) / (mbs * acc))
    total = spe * cfg["epochs"]
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.03 * total)), total)
    step = 0
    if (ckpt / "state.pt").exists():
        from safetensors.torch import load_file
        set_peft_model_state_dict(model, load_file(str(ckpt / "adapter_model.safetensors")))
        st = torch.load(ckpt / "state.pt", weights_only=False)
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); step = st["step"]
        print("resumed SFT at step", step)
    pad = tok.pad_token_id

    def batch(sel):
        # left padding: every summary ends at the last position, so logits are only needed for the
        # final K positions (Gemma-4's 262k vocab makes full-sequence fp32 logits ~2 GB per batch)
        L = max(len(examples[i][0]) for i in sel)
        inp = torch.full((len(sel), L), pad); lab = torch.full((len(sel), L), -100)
        att = torch.zeros((len(sel), L), dtype=torch.long); w = torch.zeros(len(sel))
        for k, i in enumerate(sel):
            a, b, wi = examples[i]
            inp[k, L - len(a):] = torch.tensor(a); lab[k, L - len(b):] = torch.tensor(b)
            att[k, L - len(a):] = 1; w[k] = wi
        K = int((lab != -100).sum(1).max()) + 1
        return inp.cuda(), att.cuda(), lab.cuda(), w.cuda(), min(K, L)

    model.train()
    t0, hist, bad = time.time(), [], 0
    while step < total:
        base = (step % spe) * mbs * acc
        opt.zero_grad(set_to_none=True)
        step_loss, ok = 0.0, True
        for a in range(acc):
            sel = order[base + a * mbs: base + (a + 1) * mbs]
            if not sel:
                continue
            inp, att, lab, w, K = batch(sel)
            try:
                logits = model(input_ids=inp, attention_mask=att, logits_to_keep=K, use_cache=False).logits
            except TypeError:
                logits = model(input_ids=inp, attention_mask=att, use_cache=False).logits[:, -K:]
            logits = logits[:, :-1].float()
            tgt = lab[:, lab.shape[1] - K + 1:]
            ce = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                                   ignore_index=-100, reduction="none").view(tgt.shape)
            mask = (tgt != -100).float()
            per_ex = (ce * mask).sum(1) / mask.sum(1).clamp(min=1)
            loss = (per_ex * w).sum() / w.sum().clamp(min=1e-6) / acc   # faithfulness-weighted CE
            if not torch.isfinite(loss):
                ok = False
                break
            loss.backward()
            step_loss += loss.item()
        if not ok:
            bad += 1
            print(f"non-finite loss at step {step} (skipped); consecutive={bad}")
            if bad >= 5:
                raise RuntimeError("SFT diverged (5 consecutive non-finite steps)")
            step += 1
            sched.step()
            continue
        bad = 0
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sched.step(); step += 1
        hist.append(step_loss)
        if step % 10 == 0:
            print(f"step {step}/{total} loss {np.mean(hist[-10:]):.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)
        if step % 50 == 0 or step == total:
            model.save_pretrained(ckpt)
            torch.save(dict(opt=opt.state_dict(), sched=sched.state_dict(), step=step), ckpt / "state.pt")
            with open(out_dir / "loss_history.jsonl", "a") as f:
                f.write(json.dumps(dict(step=step, loss=float(np.mean(hist[-10:])))) + "\n")
    model.save_pretrained(final)
    with open(out_dir / "train_info.json", "w") as f:
        json.dump(dict(model=model_id, precision=desc, steps=total, n_examples=len(examples),
                       final_loss=float(np.mean(hist[-20:])) if hist else None), f)
    del model, opt
    return final

## Decoding: anchor-aware CAD + DoLa, CoVe, Lookback Lens

The contrastive decoding methods of Stage 3 and the self-verification baseline.

### `factindic/decoding.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/decoding.py
"""Stage 3: Anchor-Aware Context-Contrasted Decoding (spec §8.3), batched dual-KV-cache loop.

score_t(w) = (1+α_t)·log P_L(w|D,A,y<t) − α_t·log P_L(w|∅,y<t) − β·log P_l*(w|D,A,y<t)
restricted to V_head = {w : P_L(w|D,A,y<t) ≥ η·max P_L}.

methods
  static_cad       α_t = α (fixed), β = 0                         (Shi et al., 2024)
  adacad           α_t = JSD₂(P(·|D) ‖ P(·|∅)) per token, β = 0     (Wang et al., 2025)
  dola             α_t = 0, β > 0, l* = argmax_l JSD(P_L ‖ P_l)    (Chuang et al., 2024)
  anchor_cad       α_t = α_anchor if argmax P_L ∈ V_anchor else α_base, β = 0
  anchor_cad_dola  anchor_cad + DoLa term                          (FactIndic, H4)
"""
from __future__ import annotations

import math

import torch
import torch.nn.functional as F

from .llm import DEV, lm_parts

METHODS = ("static_cad", "adacad", "dola", "anchor_cad", "anchor_cad_dola")


def jsd(p, q, base2=True):
    m = 0.5 * (p + q)
    lm = m.clamp_min(1e-12).log()
    kl_pm = (p * (p.clamp_min(1e-12).log() - lm)).sum(-1)
    kl_qm = (q * (q.clamp_min(1e-12).log() - lm)).sum(-1)
    d = 0.5 * (kl_pm + kl_qm)
    return d / math.log(2) if base2 else d


def eos_ids(model, tok):
    ids = set()
    ge = getattr(model, "generation_config", None)
    e = getattr(ge, "eos_token_id", None) if ge is not None else None
    if isinstance(e, int):
        ids.add(e)
    elif isinstance(e, (list, tuple)):
        ids |= set(e)
    if tok.eos_token_id is not None:
        ids.add(tok.eos_token_id)
    for t in ("<end_of_turn>", "<|eot_id|>", "<|im_end|>", "<turn|>"):
        tid = tok.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid >= 0 and tid != tok.unk_token_id:
            ids.add(tid)
    return sorted(ids)


def anchor_token_ids(tok, rec, variant_text=None):
    """V_anchor(D): subword ids of normalized anchors' surface forms, digits, and embedded English."""
    surf = [e["text"] for e in rec["anchors"].get("entities", [])]
    surf += rec["anchors"].get("numbers", []) + list(rec.get("english_terms", []))
    ids = set()
    for s in surf:
        for form in (s, " " + s):
            ids |= set(tok(form, add_special_tokens=False)["input_ids"])
    for d in "0123456789":
        ids |= set(tok(d, add_special_tokens=False)["input_ids"])
    # drop whitespace/punctuation-only pieces that would fire on every step
    junk = {i for i in ids if not tok.decode([i]).strip() or tok.decode([i]).strip() in ",.।:;-–()"}
    return ids - junk


def _cache(out):
    """transformers 5.24 renamed past_key_values to cache."""
    return out["cache"] if "cache" in out else out.past_key_values


def _cache_arg():
    import transformers
    from packaging.version import parse
    return "cache" if parse(transformers.__version__) >= parse("5.24.0") else "past_key_values"


CACHE_ARG = _cache_arg()


def _prefill(model, ids, mask, want_hidden):
    """Prefill all but the last prompt token, then step the last one (hidden states only for it)."""
    pos = (mask.cumsum(-1) - 1).clamp(min=0)
    past = None
    if ids.shape[1] > 1:                 # logits_to_keep: full-prompt logits over a 262k vocabulary need gigabytes
        out = model(input_ids=ids[:, :-1], attention_mask=mask[:, :-1], position_ids=pos[:, :-1],
                    use_cache=True, logits_to_keep=1)
        past = _cache(out)
    out = model(input_ids=ids[:, -1:], attention_mask=mask, position_ids=pos[:, -1:],
                **{CACHE_ARG: past}, use_cache=True, output_hidden_states=want_hidden)
    return out, pos[:, -1:]


def contrastive_generate(model, tok, ctx_texts, unc_texts, anchor_sets, method, **kw):
    """Returns (texts, info with per-row mean alpha and the premature layers). A batch that does not fit
    in memory is split in two."""
    try:
        return _contrastive_generate(model, tok, ctx_texts, unc_texts, anchor_sets, method, **kw)
    except torch.cuda.OutOfMemoryError:
        if len(ctx_texts) == 1:
            raise
    torch.cuda.empty_cache()
    m = len(ctx_texts) // 2
    a, b = (contrastive_generate(model, tok, ctx_texts[s], unc_texts[s], anchor_sets[s], method, **kw)
            for s in (slice(None, m), slice(m, None)))
    return a[0] + b[0], dict(mean_alpha=a[1]["mean_alpha"] + b[1]["mean_alpha"], layers=a[1]["layers"])


@torch.no_grad()
def _contrastive_generate(model, tok, ctx_texts, unc_texts, anchor_sets, method,
                         alpha_base=0.4, alpha_anchor=0.9, beta=0.3, eta=0.1, static_alpha=0.5,
                         layers=None, max_new_tokens=160):
    assert method in METHODS, method
    use_unc = method in ("static_cad", "adacad", "anchor_cad", "anchor_cad_dola")
    use_dola = method in ("dola", "anchor_cad_dola")
    if method == "dola":
        alpha_base = alpha_anchor = static_alpha = 0.0
    if not use_dola:
        beta = 0.0
    B = len(ctx_texts)
    tok.padding_side = "left"
    ec = tok(ctx_texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
    m_c = ec["attention_mask"]
    out_c, pos_c = _prefill(model, ec["input_ids"], m_c, use_dola)
    if use_unc:
        eu = tok(unc_texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEV)
        m_u = eu["attention_mask"]
        out_u, pos_u = _prefill(model, eu["input_ids"], m_u, False)
    if use_dola:
        n_layers, norm, lm_head, softcap = lm_parts(model)
        layers = layers or sorted({max(1, int(n_layers * f)) for f in (0.35, 0.45, 0.55, 0.65)})
    V = out_c.logits.shape[-1]
    amask = torch.zeros((B, V), dtype=torch.bool, device=DEV)
    for b, s in enumerate(anchor_sets or [set()] * B):
        idx = [i for i in s if i < V]
        if idx:
            amask[b, idx] = True
    stop = torch.tensor(eos_ids(model, tok), device=DEV)
    finished = torch.zeros(B, dtype=torch.bool, device=DEV)
    gen = [[] for _ in range(B)]
    alpha_sum = torch.zeros(B, device=DEV)
    n_steps = torch.zeros(B, device=DEV)
    pad = tok.pad_token_id
    for t in range(max_new_tokens):
        if bool((~torch.isfinite(out_c.logits[:, -1]).all(-1) & ~finished).any()):
            raise RuntimeError("non-finite logits in an unfinished row")
        logp_c = F.log_softmax(out_c.logits[:, -1].float(), -1)
        p_c = logp_c.exp()
        score = logp_c.clone()
        a = torch.zeros(B, device=DEV)
        if use_unc:
            logp_u = F.log_softmax(out_u.logits[:, -1].float(), -1)
            if method == "static_cad":
                a = torch.full((B,), static_alpha, device=DEV)
            elif method == "adacad":
                a = jsd(p_c, logp_u.exp())
            else:
                top = p_c.argmax(-1)
                is_anchor = amask[torch.arange(B, device=DEV), top]
                a = torch.where(is_anchor, torch.tensor(alpha_anchor, device=DEV),
                                torch.tensor(alpha_base, device=DEV))
            score = (1 + a)[:, None] * logp_c - a[:, None] * logp_u
        if use_dola and beta > 0:
            hs = out_c.hidden_states
            best_j = torch.full((B,), -1.0, device=DEV)
            early = torch.zeros_like(logp_c)
            for l in layers:
                lg = lm_head(norm(hs[l][:, -1])).float()
                if softcap:
                    lg = torch.tanh(lg / softcap) * softcap
                lp = F.log_softmax(lg, -1)
                j = jsd(p_c, lp.exp())
                upd = j > best_j
                best_j = torch.where(upd, j, best_j)
                early = torch.where(upd[:, None], lp, early)
            score = score - beta * early
        plaus = p_c >= eta * p_c.max(-1, keepdim=True).values
        score = score.masked_fill(~plaus, float("-inf"))
        nxt = score.argmax(-1)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        alive = ~finished
        alpha_sum += a * alive
        n_steps += alive.float()
        for b in torch.nonzero(alive).flatten().tolist():
            gen[b].append(int(nxt[b]))
        finished = finished | torch.isin(nxt, stop)
        if bool(finished.all()):
            break
        ones = torch.ones((B, 1), dtype=m_c.dtype, device=DEV)
        m_c = torch.cat([m_c, ones], 1)
        pos_c = pos_c + 1
        out_c = model(input_ids=nxt[:, None], attention_mask=m_c, position_ids=pos_c,
                      **{CACHE_ARG: _cache(out_c)}, use_cache=True, output_hidden_states=use_dola)
        if use_unc:
            m_u = torch.cat([m_u, ones], 1)
            pos_u = pos_u + 1
            out_u = model(input_ids=nxt[:, None], attention_mask=m_u, position_ids=pos_u,
                          **{CACHE_ARG: _cache(out_u)}, use_cache=True)
    stop_set = set(stop.tolist())
    texts = []
    from .llm import clean_output
    for g in gen:
        g = [x for x in g if x not in stop_set and x != pad]
        texts.append(clean_output(tok.decode(g, skip_special_tokens=True)))
    info = dict(mean_alpha=(alpha_sum / n_steps.clamp(min=1)).tolist(), layers=layers if use_dola else None)
    return texts, info


@torch.no_grad()
def conditional_ppl(model, tok, prompt_texts, outputs):
    """Perplexity of each output under the plain model given its prompt (fluency/drift proxy)."""
    res = []
    for p, o in zip(prompt_texts, outputs):
        if not o:
            res.append(float("nan"))
            continue
        p_ids = tok(p, add_special_tokens=False, return_tensors="pt")["input_ids"]
        o_ids = tok(o, add_special_tokens=False, return_tensors="pt")["input_ids"]
        ids = torch.cat([p_ids, o_ids], 1).to(DEV)
        K = o_ids.shape[1] + 1
        try:
            lg = model(input_ids=ids, logits_to_keep=K).logits
        except TypeError:
            lg = model(input_ids=ids).logits[:, -K:]
        lp = F.log_softmax(lg[:, :-1].float(), -1)
        nll = -lp.gather(-1, ids[:, -o_ids.shape[1]:, None]).squeeze(-1).mean()
        res.append(float(nll.exp()))
    return res

### `factindic/cove.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/cove.py
"""Chain-of-Verification (factored) self-rewrite baseline (Exp 5b) — the 'self-verification
fallacy' comparator: the same 2B generator plans, answers, and rewrites its own summary."""
from __future__ import annotations

import re

from .config import LANGS
from .llm import chat_text, generate


def _plan(lang, summary):
    return (f"Here is a {LANGS[lang]['name']} news summary:\n{summary}\n\n"
            "Write up to 4 short verification questions (in English) that check its facts: names, "
            "numbers, dates, and who did what. One question per line, no numbering.")


def _answer(lang, variant_text, q):
    return (f"Article:\n{variant_text}\n\nAnswer the question using only the article, in one short "
            f"sentence. If the article does not say, answer 'not stated'.\nQuestion: {q}\nAnswer:")


def _revise(lang, summary, qa, n_words):
    L, S = LANGS[lang]["name"], LANGS[lang]["script"]
    qa_txt = "\n".join(f"Q: {q}\nA: {a}" for q, a in qa)
    return (f"Original summary ({L}):\n{summary}\n\nVerification results:\n{qa_txt}\n\n"
            f"Rewrite the summary in {L} ({S} script), about {n_words} words, correcting anything the "
            f"verification contradicts and removing anything marked 'not stated'. Output only the summary.")


def cove_batch(model, tok, recs_variants, drafts, max_new_tokens=160):
    """recs_variants: list of (rec, variant). Returns revised summaries and the Q/A traces."""
    plans, _, _ = generate(model, tok, [chat_text(tok, _plan(r["lang"], d)) for (r, _), d in zip(recs_variants, drafts)],
                           max_new_tokens=120)
    qs = []
    for p in plans:
        lines = [re.sub(r"^[\-\*\d\.\)\s]+", "", l).strip() for l in p[0].split("?")]
        qs.append([l + "?" for l in lines if len(l) > 5][:4])
    flat = [(i, q) for i, ql in enumerate(qs) for q in ql]
    answers = {}
    for k in range(0, len(flat), 8):
        batch = flat[k:k + 8]
        texts = [chat_text(tok, _answer(recs_variants[i][0]["lang"], recs_variants[i][0]["variants"][recs_variants[i][1]], q))
                 for i, q in batch]
        out, _, _ = generate(model, tok, texts, max_new_tokens=48)
        for (i, q), o in zip(batch, out):
            answers.setdefault(i, []).append((q, o[0]))
    rev_prompts = [chat_text(tok, _revise(r["lang"], d, answers.get(i, []), r["L_target"]))
                   for i, ((r, _), d) in enumerate(zip(recs_variants, drafts))]
    revised, _, _ = generate(model, tok, rev_prompts, max_new_tokens=max_new_tokens)
    return [x[0] for x in revised], [answers.get(i, []) for i in range(len(drafts))]

### `factindic/lookback.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/lookback.py
"""Lookback Lens (Chuang et al., 2024) on Gemma-4: per-head lookback ratios per summary sentence.

LR(l,h)_t = Σ_ctx Attn(t,i) / (Σ_ctx Attn(t,i) + Σ_new Attn(t,j)); averaged over each sentence's
tokens, concatenated with the sentence's relative position. An L2 logistic regression detects
unsupported sentences (labels: human dev labels when present, else held-out verifier labels).
"""
from __future__ import annotations

import numpy as np
import torch

from .llm import DEV
from .text import split_sentences


@torch.no_grad()
def sentence_features(model, tok, prompt_text, summary):
    sents = split_sentences(summary)
    if not sents:
        return []
    p_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    pieces, spans, cur = [], [], len(p_ids)
    for i, s in enumerate(sents):
        ids = tok((" " if i else "") + s, add_special_tokens=False)["input_ids"]
        pieces += ids
        spans.append((cur, cur + len(ids)))
        cur += len(ids)
    ids = torch.tensor([p_ids + pieces], device=DEV)
    out = model(input_ids=ids, output_attentions=True, logits_to_keep=1)
    P = len(p_ids)
    feats = []
    att = [a[0].float() for a in out.attentions if a is not None]   # each [H, T, T]
    for k, (a, b) in enumerate(spans):
        per_layer = []
        for A in att:
            rows = A[:, a:b, :]                          # [H, n, T]
            ctx = rows[:, :, :P].sum(-1)
            new = torch.stack([rows[:, i, P:a + i].sum(-1) for i in range(b - a)], 1)
            lr = (ctx / (ctx + new).clamp_min(1e-9)).mean(1)   # [H]
            per_layer.append(lr)
        f = torch.cat(per_layer).cpu().numpy()
        feats.append(np.concatenate([f, [k / max(1, len(spans) - 1)]]))
    return feats


def train_detector(X, y, C=1.0, seed=42):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, penalty="l2", max_iter=2000,
                                                             class_weight="balanced", random_state=seed))
    clf.fit(X, y)
    return clf

## Figures, annotation sheets, checks

### `factindic/viz.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/viz.py
"""Figures that test H1–H4 and justify each pipeline stage. Every figure is saved as PNG + SVG
with its data as CSV (the table view). Style: validated categorical palette in fixed slot order,
colour follows the entity (never its rank), one y-axis per chart, recessive grid, direct labels."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .config import DIRS

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
NEUTRAL = "#9a9890"
# fixed entity → colour (never re-assigned when a filter drops entities)
ENTITY = {"gemma": SLOTS[0], "sarvam": SLOTS[1], "llama": SLOTS[2], "indicbartss": SLOTS[3], "mt5": SLOTS[4],
          "native": SLOTS[0], "script_mixed": SLOTS[1], "roman": SLOTS[2], "natural_cm": SLOTS[3],
          "hi": SLOTS[0], "gu": SLOTS[1]}
VARIANT_LABEL = {"native": "Native", "script_mixed": "Script-mixed", "roman": "Romanized", "natural_cm": "Natural CM"}
LANG_LABEL = {"hi": "Hindi", "gu": "Gujarati"}


def color_of(name):
    for k, c in ENTITY.items():
        if name == k or name.startswith(k):
            return c
    return NEUTRAL


def style():
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
        "axes.titleweight": "semibold", "axes.labelsize": 9, "xtick.color": INK2, "ytick.color": INK2,
        "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "axes.grid": True, "grid.color": GRID,
        "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "legend.fontsize": 8.5, "font.size": 9, "lines.linewidth": 2,
        "font.family": ["DejaVu Sans"],
    })


def save(fig, name, data: pd.DataFrame | None = None, caption=""):
    DIRS["figures"].mkdir(parents=True, exist_ok=True)
    if caption:
        fig.text(0.01, 0.005, caption, fontsize=7.5, color=INK2, ha="left", va="bottom", wrap=True)
    fig.savefig(DIRS["figures"] / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(DIRS["figures"] / f"{name}.svg", bbox_inches="tight")
    if data is not None:
        data.to_csv(DIRS["figures"] / f"{name}.csv", index=False)
    import matplotlib.pyplot as plt
    plt.close(fig)
    return DIRS["figures"] / f"{name}.png"


def _bar(ax, x, h, color, width, label=None):
    """Thin bars with a 2px surface gap between neighbours."""
    return ax.bar(x, h, width=width, color=color, label=label, edgecolor=SURFACE, linewidth=1.5, zorder=2)


# ----------------------------------------------------------------------------------------- H1
def h1_fertility(fert: pd.DataFrame):
    """Grouped bars: tokens/word by tokenizer for each (language, variant)."""
    import matplotlib.pyplot as plt
    style()
    toks = [t for t in ["sarvam1_sft", "gemma4_e2b", "llama32_3b", "indicbartss"] if t in set(fert.tokenizer)]
    variants = [v for v in ["native", "script_mixed", "roman"] if v in set(fert.variant)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    w = 0.8 / max(1, len(toks))
    for ax, lang in zip(axes, ["hi", "gu"]):
        for i, t in enumerate(toks):
            vals = [fert[(fert.tokenizer == t) & (fert.lang == lang) & (fert.variant == v)].fertility_corpus.mean()
                    for v in variants]
            xs = np.arange(len(variants)) + (i - (len(toks) - 1) / 2) * w
            _bar(ax, xs, vals, color_of(t), w, label=t)
            for x, v in zip(xs, vals):
                ax.text(x, v + 0.04, f"{v:.2f}", ha="center", va="bottom", fontsize=7, color=INK2)
        ax.set_xticks(range(len(variants)), [VARIANT_LABEL[v] for v in variants])
        ax.set_title(LANG_LABEL[lang])
        ax.grid(axis="x", visible=False)
    axes[0].set_ylabel("Tokenizer fertility (tokens / word)")
    axes[1].legend(loc="upper left", ncols=2)
    fig.suptitle("Romanization raises tokens per word for Sarvam-1 and IndicBARTSS but lowers it for Llama-3.2", x=0.01, ha="left",
                 fontsize=12, fontweight="semibold", color=INK)
    fig.tight_layout()
    return save(fig, "h1_fertility", fert)


def h1_shift(per_summary: pd.DataFrame):
    """Arrow per (model, language): native → Romanized in (fertility, hallucination) space.
    H1 predicts arrows point up-and-right; a leftward-up arrow is a counterexample."""
    import matplotlib.pyplot as plt
    style()
    g = (per_summary.groupby(["model", "lang", "script"])
         .agg(F=("input_fertility", "mean"), H=("halluc", "mean")).reset_index())
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    rows = []
    for (m, lang), d in g.groupby(["model", "lang"]):
        d = d.set_index("script")
        if not {"native", "roman"} <= set(d.index):
            continue
        c = color_of(m)
        x0, y0, x1, y1 = d.loc["native", "F"], d.loc["native", "H"], d.loc["roman", "F"], d.loc["roman", "H"]
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="-|>", color=c, lw=2, shrinkA=4, shrinkB=4))
        ax.scatter([x0], [y0], s=64, color=c, edgecolor=SURFACE, linewidth=2, zorder=3)
        ax.scatter([x1], [y1], s=64, color=SURFACE, edgecolor=c, linewidth=2, zorder=3)
        ax.text(x1 + 0.03, y1, f"{m} · {LANG_LABEL[lang]}", fontsize=8, color=INK, va="center")
        rows.append(dict(model=m, lang=lang, F_native=x0, H_native=y0, F_roman=x1, H_roman=y1))
    ax.set_xlabel("Input fertility (tokens / word)")
    ax.set_ylabel("Claim hallucination rate (1 − P_fact)")
    ax.set_title("Native ● → Romanized ○: hallucination rises even where fertility falls", loc="left")
    fig.tight_layout()
    return save(fig, "h1_shift", pd.DataFrame(rows),
                caption="Each arrow joins the same 50 articles fed in native script and Romanized.")


def coef_plot(tests: dict, outcome_labels: dict, name, title, xlabel="β on input fertility (95% CI)"):
    """Dot + CI per (spec, outcome). tests[outcome][spec] = {beta, se, p_one_pos}."""
    import matplotlib.pyplot as plt
    style()
    rows = []
    for o, lab in outcome_labels.items():
        for spec, r in tests.get(o, {}).items():
            if "beta" in r:
                rows.append(dict(outcome=lab, spec=spec, beta=r["beta"], lo=r["beta"] - 1.96 * r["se"],
                                 hi=r["beta"] + 1.96 * r["se"], p=r.get("p_one_pos", r.get("p_one"))))
    df = pd.DataFrame(rows)
    if df.empty:
        return None
    fig, ax = plt.subplots(figsize=(7, 0.42 * len(df) + 1.2))
    specs = list(dict.fromkeys(df.spec))
    for i, r in enumerate(df.itertuples()):
        c = SLOTS[specs.index(r.spec) % len(SLOTS)]
        ax.plot([r.lo, r.hi], [i, i], color=c, lw=2, solid_capstyle="round")
        ax.scatter([r.beta], [i], s=40, color=c, edgecolor=SURFACE, linewidth=1.5, zorder=3)
        ax.text(max(r.hi, r.beta) + 0.01, i, f"p={r.p:.3f}" if r.p == r.p else "", va="center", fontsize=7.5, color=INK2)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(range(len(df)), [f"{r.outcome} · {r.spec.replace('_', ' ')}" for r in df.itertuples()])
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return save(fig, name, df)


# ----------------------------------------------------------------------------------------- generic
def dot_ci(df, name, title, xlabel, label_col="label", highlight=None, ref_line=None, caption=""):
    """Single-series horizontal dot plot with 95% CIs; one colour, optional highlighted row."""
    import matplotlib.pyplot as plt
    style()
    fig, ax = plt.subplots(figsize=(7.2, 0.38 * len(df) + 1.3))
    for i, r in enumerate(df.itertuples()):
        c = SLOTS[1] if highlight and getattr(r, label_col) in highlight else SLOTS[0]
        ax.plot([r.lo, r.hi], [i, i], color=c, lw=2, solid_capstyle="round", alpha=0.9)
        ax.scatter([r.mean], [i], s=44, color=c, edgecolor=SURFACE, linewidth=1.5, zorder=3)
        ax.text(r.hi, i, f"  {r.mean:.3f}", va="center", fontsize=7.5, color=INK2)
    if ref_line is not None:
        ax.axvline(ref_line, color=INK2, lw=1, ls=(0, (3, 3)))
    ax.set_yticks(range(len(df)), list(df[label_col]))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return save(fig, name, df, caption)


def lines_by_variant(df, name, title, ylabel, series_col="system", caption=""):
    """Small multiples per language: x = variant, y = metric (mean, lo, hi), one line per series (≤4)."""
    import matplotlib.pyplot as plt
    style()
    series = list(dict.fromkeys(df[series_col]))[:4]
    variants = [v for v in ["native", "script_mixed", "roman", "natural_cm"] if v in set(df.variant)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, lang in zip(axes, ["hi", "gu"]):
        for i, s in enumerate(series):
            d = df[(df[series_col] == s) & (df.lang == lang)].set_index("variant").reindex(variants)
            xs = np.arange(len(variants))
            c = SLOTS[i]
            ax.fill_between(xs, d["lo"], d["hi"], color=c, alpha=0.12, lw=0)
            ax.plot(xs, d["mean"], color=c, marker="o", ms=6, mec=SURFACE, mew=1.5, label=s)
            if len(series) <= 4 and not d["mean"].isna().all():
                ax.text(xs[-1] + 0.08, d["mean"].iloc[-1], s, fontsize=7.5, color=INK, va="center")
        ax.set_xticks(range(len(variants)), [VARIANT_LABEL[v] for v in variants])
        ax.set_title(LANG_LABEL[lang])
        ax.set_xlim(-0.3, len(variants) - 0.2 + 0.9)
    axes[0].set_ylabel(ylabel)
    axes[1].legend(loc="lower left")
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="semibold", color=INK)
    fig.tight_layout()
    return save(fig, name, df, caption)


def grouped_bars(df, name, title, ylabel, x_col, group_col, val_col, err=None, order_x=None, order_g=None,
                 color_fn=None, caption="", pct=False):
    import matplotlib.pyplot as plt
    style()
    xs_vals = order_x or list(dict.fromkeys(df[x_col]))
    groups = order_g or list(dict.fromkeys(df[group_col]))
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(xs_vals) * max(1, len(groups)) * 0.55 + 2), 3.8))
    w = 0.8 / max(1, len(groups))
    for i, g in enumerate(groups):
        d = df[df[group_col] == g].set_index(x_col).reindex(xs_vals)
        xs = np.arange(len(xs_vals)) + (i - (len(groups) - 1) / 2) * w
        c = color_fn(g) if color_fn else SLOTS[i % len(SLOTS)]
        _bar(ax, xs, d[val_col], c, w, label=str(g))
        if err:
            lo, hi = d[err[0]], d[err[1]]
            ax.errorbar(xs, d[val_col], yerr=[d[val_col] - lo, hi - d[val_col]], fmt="none",
                        ecolor=INK2, elinewidth=1, capsize=0, zorder=3)
        for x, v in zip(xs, d[val_col]):
            if v == v:
                ax.text(x, v, f"{v:.0%}" if pct else f"{v:.2f}", ha="center", va="bottom", fontsize=7, color=INK2)
    ax.set_xticks(range(len(xs_vals)), [VARIANT_LABEL.get(x, LANG_LABEL.get(x, x)) for x in xs_vals])
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.grid(axis="x", visible=False)
    if len(groups) > 1:
        ax.legend(loc="upper left", ncols=min(4, len(groups)))
    fig.tight_layout()
    return save(fig, name, df, caption)


def pr_scatter(df, name, title, caption=""):
    """Precision vs recall per system (≤3 colours: pipeline / baselines / repair)."""
    import matplotlib.pyplot as plt
    style()
    kind_color = {"FactIndic": SLOTS[1], "baseline": SLOTS[0], "repair": SLOTS[2]}
    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    for r in df.itertuples():
        c = kind_color.get(r.kind, NEUTRAL)
        ax.plot([r.R_lo, r.R_hi], [r.P, r.P], color=c, lw=1, alpha=0.6)
        ax.plot([r.R, r.R], [r.P_lo, r.P_hi], color=c, lw=1, alpha=0.6)
        ax.scatter([r.R], [r.P], s=56, color=c, edgecolor=SURFACE, linewidth=2, zorder=3)
        ax.text(r.R + 0.006, r.P + 0.006, r.system, fontsize=7.5, color=INK)
    for k, c in kind_color.items():
        ax.scatter([], [], color=c, s=40, label=k)
    ax.legend(loc="lower left")
    ax.set_xlabel("Claim-level factual recall R_fact")
    ax.set_ylabel("Claim-level factual precision P_fact")
    ax.set_title(title, loc="left")
    fig.tight_layout()
    return save(fig, name, df, caption)


def roc(curves: dict, name, title, caption=""):
    import matplotlib.pyplot as plt
    style()
    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    rows = []
    for i, (lab, (fpr, tpr, auc)) in enumerate(curves.items()):
        ax.plot(fpr, tpr, color=SLOTS[i], label=f"{lab} (AUROC {auc:.2f})")
        rows += [dict(curve=lab, fpr=a, tpr=b) for a, b in zip(fpr, tpr)]
    ax.plot([0, 1], [0, 1], color=GRID, lw=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title, loc="left")
    ax.legend(loc="lower right")
    fig.tight_layout()
    return save(fig, name, pd.DataFrame(rows), caption)


def histogram(values, name, title, xlabel, vline=None, vlabel="", caption=""):
    import matplotlib.pyplot as plt
    style()
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    for i, (lab, v) in enumerate(values.items()):
        ax.hist(v, bins=30, range=(0, 1), color=SLOTS[i], alpha=0.55, label=lab, edgecolor=SURFACE, linewidth=1)
    if vline is not None:
        ax.axvline(vline, color=INK, lw=1, ls=(0, (3, 3)))
        ax.text(vline, ax.get_ylim()[1] * 0.95, f" {vlabel}", fontsize=8, color=INK, va="top")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Training pairs")
    ax.set_title(title, loc="left")
    ax.legend()
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    data = pd.DataFrame([dict(group=k, value=x) for k, v in values.items() for x in v])
    return save(fig, name, data, caption)

### `factindic/annotate.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/annotate.py
"""Human evaluation: claim-level annotation sheets (300 summaries, 2 annotators per language),
variant-naturalness sheets (50 items per language), import + Cohen's κ.

Labels: 1 = Supported, 0 = Unsupported, -1 = Contradicted (scored as 0).
Error types (ILSUM 2024 Task 2): Misrepresentation, Inaccurate Quantities, False Attribution, Fabrication.
"""
from __future__ import annotations

import random

import pandas as pd

from .claims import sentence_claims
from .config import CFG, DIRS
from .stats import cohen_kappa

ERROR_TYPES = ["", "Misrepresentation", "Inaccurate Quantities", "False Attribution", "Fabrication"]


def export_claim_sheets(summaries, records, seed=42):
    """summaries: list of dicts (system, item, uid, lang, variant, summary). Systems are blinded."""
    rng = random.Random(seed)
    by_lang = {"hi": [], "gu": []}
    for s in summaries:
        if s.get("summary"):
            by_lang[s["lang"]].append(s)
    n_lang = CFG["n_annot_summaries"] // 2
    rows, key = [], []
    for lang, pool in by_lang.items():
        # stratify across (variant, system) cells
        cells = {}
        for s in pool:
            cells.setdefault((s["variant"], s["system"]), []).append(s)
        picked, cell_keys = [], sorted(cells)
        while len(picked) < n_lang and any(cells.values()):
            for ck in cell_keys:
                if cells[ck] and len(picked) < n_lang:
                    picked.append(cells[ck].pop(rng.randrange(len(cells[ck]))))
        rng.shuffle(picked)
        n_dev = CFG["n_annot_dev"] // 2
        for i, s in enumerate(picked):
            sid = f"{lang}-{i:03d}"
            split = "dev" if i < n_dev else "test"
            key.append(dict(sid=sid, split=split, system=s["system"], item=s["item"], uid=s["uid"],
                            lang=lang, variant=s["variant"]))
            src = records[s["uid"]]["variants"].get(s["variant"], records[s["uid"]]["text"])
            for ci, c in enumerate(sentence_claims(s["summary"])):
                rows.append(dict(sid=sid, claim_idx=ci, source=src if ci == 0 else "", summary=s["summary"] if ci == 0 else "",
                                 claim=c, label="", error_type="", note=""))
    df, kdf = pd.DataFrame(rows), pd.DataFrame(key)
    for lang in ("hi", "gu"):
        part = df[df.sid.str.startswith(lang)]
        for ann in ("A", "B"):
            part.to_csv(DIRS["annotation"] / f"claims_{lang}_annotator{ann}.csv", index=False, encoding="utf-8-sig")
    kdf.to_csv(DIRS["annotation"] / "claims_key_DO_NOT_SHARE.csv", index=False)
    return len(kdf), len(df)


def export_naturalness(records, seed=42):
    rng = random.Random(seed)
    rows = []
    for lang in ("hi", "gu"):
        pool = [r for r in records if r["lang"] == lang]
        for r in rng.sample(pool, min(CFG["n_naturalness_per_lang"], len(pool))):
            for v in ("script_mixed", "roman"):
                rows.append(dict(uid=r["uid"], lang=lang, variant=v, native=r["text"][:600],
                                 variant_text=r["variants"][v][:600], naturalness_1to5="",
                                 meaning_preserved_yes_no=""))
    df = pd.DataFrame(rows)
    for lang in ("hi", "gu"):
        path = DIRS["annotation"] / f"naturalness_{lang}.csv"
        if path.exists() and pd.read_csv(path).naturalness_1to5.notna().any():
            print(f"kept {path.name}: it already has ratings")
            continue
        df[df.lang == lang].to_csv(path, index=False, encoding="utf-8-sig")
    return len(df)


def _lab(x):
    try:
        v = int(float(x))
    except (TypeError, ValueError):
        return None
    return 1 if v == 1 else 0


def import_claim_labels():
    """Returns (merged claim table with gold = agreement or annotator A on ties, agreement report)."""
    key = pd.read_csv(DIRS["annotation"] / "claims_key_DO_NOT_SHARE.csv")
    frames, report = [], {}
    for lang in ("hi", "gu"):
        pa = DIRS["annotation"] / f"claims_{lang}_annotatorA.csv"
        pb = DIRS["annotation"] / f"claims_{lang}_annotatorB.csv"
        if not (pa.exists() and pb.exists()):
            continue
        a, b = pd.read_csv(pa), pd.read_csv(pb)
        m = a.merge(b, on=["sid", "claim_idx"], suffixes=("_a", "_b"))
        m["la"], m["lb"] = m.label_a.map(_lab), m.label_b.map(_lab)
        m = m.dropna(subset=["la", "lb"])
        if m.empty:
            continue
        report[lang] = dict(n_claims=len(m), kappa=cohen_kappa(m.la, m.lb),
                            raw_agreement=float((m.la == m.lb).mean()))
        m["gold"] = m.la  # adjudicate disagreements offline; default to annotator A
        m["error_type"] = m.error_type_a.fillna("")
        m["claim"] = m.claim_a
        frames.append(m[["sid", "claim_idx", "claim", "la", "lb", "gold", "error_type"]])
    if not frames:
        return None, report
    lab = pd.concat(frames).merge(key, on="sid")
    return lab, report

### `factindic/preflight.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/preflight.py
"""Run at the top of every notebook: fail in the first minutes, not hours in."""
from __future__ import annotations

import re
from importlib import metadata
from pathlib import Path

from .config import DATA_VERSION, MODELS, data_version

PINS = Path(__file__).with_name("requirements.txt")
PROBES = {"hi": "सूरत के अस्पताल में सोमवार को 42 मरीज भर्ती हुए।",
          "gu": "સુરતની હોસ્પિટલમાં સોમવારે 42 દર્દીઓ દાખલ થયા."}


def versions():
    bad = []
    for name, want in re.findall(r"^([A-Za-z0-9_.-]+)==([^\s#]+)", PINS.read_text(), re.M):
        try:
            got = metadata.version(name)
        except metadata.PackageNotFoundError:
            got = None
        if got != want:
            bad.append(f"{name}: need {want}, found {got}")
    if bad:
        raise RuntimeError("Wrong package versions. Re-run the install cell, then restart the kernel if a "
                           "package was already imported:\n  " + "\n  ".join(bad))
    print("versions ok")


def access(required=(), optional=()):
    """Gated-model access for the account behind HF_TOKEN."""
    from huggingface_hub import HfApi
    missing = []
    for key in (*required, *optional):
        try:
            HfApi().auth_check(MODELS[key])
            print("access ok:", MODELS[key])
        except Exception as e:
            msg = f"{MODELS[key]} ({type(e).__name__}): accept the terms at https://huggingface.co/{MODELS[key]}"
            if key in required:
                missing.append(msg)
            else:
                print("WARNING, optional model not accessible:", msg)
    if missing:
        raise RuntimeError("Hugging Face access missing:\n  " + "\n  ".join(missing))


def translation():
    """Build the IndicTrans2 environment and translate one Hindi and one Gujarati sentence."""
    from .mt import Translator
    mt = Translator(cache_name="mt_preflight.jsonl")
    try:
        for lang, s in PROBES.items():
            en = mt._run([s], lang)[0]          # bypass the cache: the point is to exercise the engine
            print(f"{mt.backend} {lang}: {en}")
            if "42" not in en or "hospital" not in en.lower():
                raise RuntimeError(f"translation sanity check failed for {lang}: {en!r}")
    finally:
        mt.close()


def data():
    v = data_version()
    if v != DATA_VERSION:
        raise RuntimeError(f"Notebook 01 output is missing or old (data version {v}, need {DATA_VERSION}). "
                           "Attach the output of the latest notebook 01 run (Add Input → Your Work).")
    print("data ok")


def check(mt=False, required=(), optional=(), need_data=True):
    versions()
    if need_data:
        data()
    access(required, optional)
    if mt:
        translation()
    print("preflight passed")

## Pipeline stages

Stage 1 to Stage 4, evaluation, hypothesis tests, and the job queue that runs everything on both GPUs.

### `factindic/pipeline/__init__.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/__init__.py
"""Stage drivers called by the Kaggle/Colab notebooks."""

### `factindic/pipeline/stage1.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/stage1.py
"""Stage 0/1 data preparation: sanitation, testbed/dev/natural-CM sets with normalized anchors,
mFACT-style faithfulness weights, naturalness sheets, and the data figures."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..config import CFG, DIRS
from ..io import JsonlAppender, free_gpu, jl_read, jl_read_glob, jl_write, json_dump, json_load
from ..text import chunk_sentences, split_sentences


def build_all():
    from ..data import build_eval_sets, build_natural_cm, build_roman_vocab, build_train_pool, stamp, upgrade
    from ..ner import Tagger
    upgrade()
    build_train_pool()
    build_roman_vocab()
    tagger = Tagger()
    build_eval_sets(tagger)
    build_natural_cm(tagger)
    stamp()
    del tagger
    free_gpu()


def _key(r):
    return f"{r['lang']}:{r['id']}"


def prepare_weights():
    """Fix which pool pairs get faithfulness scores: pairs scored earlier first (scores of changed pairs
    were dropped with the data upgrade), then random. Run once before weights_job."""
    import random
    path = DIRS["data"] / "score_subset.json"
    if path.exists():
        return
    pool = jl_read(DIRS["data"] / "train_pool.jsonl")
    base = DIRS["data"] / "weights.jsonl"
    scored = {w["key"]: w for w in jl_read_glob(base)}
    rng, subset = random.Random(CFG["seed"]), []
    for lang in ("hi", "gu"):
        rows = [r for r in pool if r["lang"] == lang]
        first = [r for r in rows if _key(r) in scored]
        rest = [r for r in rows if _key(r) not in scored]
        n = min(CFG["n_score_per_lang"], len(rows))
        subset += (first + rng.sample(rest, min(len(rest), max(0, n - len(first)))))[:n]
    keep = [scored[_key(r)] for r in subset if _key(r) in scored]
    for sh in base.parent.glob("weights.shard*.jsonl"):
        sh.unlink()
    jl_write(base, keep)
    json_dump([_key(r) for r in subset], path)
    print(f"{len(subset)} pairs to score, {len(keep)} earlier scores still valid")


def weights_job(shard, n_shards, **_):
    """s_MC(D_i, R_i) = mean over reference sentences of max_j MiniCheck(D̃_j, r̃). Discard < 0.30,
    weight w_i = clip(s, 0.3, 1.0)."""
    from ..mt import Translator
    from ..verify import MiniCheck, max_over_chunks
    by_key = {_key(r): r for r in jl_read(DIRS["data"] / "train_pool.jsonl")}
    pool = [by_key[k] for k in json_load(DIRS["data"] / "score_subset.json")]
    out = JsonlAppender(DIRS["data"] / "weights.jsonl", key_fn=lambda r: r["key"], shard=shard)
    todo = [r for i, r in enumerate(pool) if i % n_shards == shard and _key(r) not in out.done]
    if not todo:
        return
    mt = Translator()
    mc = MiniCheck()
    for k, r in enumerate(todo):
        src_en = mt.translate(split_sentences(r["text"]), r["lang"])
        ref_en = [x for x in mt.translate(split_sentences(r["summary"]), r["lang"]) if x]
        chunks = chunk_sentences(src_en, 250)
        ps = max_over_chunks(mc, chunks, ref_en) if ref_en else [0.0]
        s = float(np.mean(ps))
        out.add(dict(key=_key(r), lang=r["lang"], source=r["source"], s_mc=s,
                     w=float(np.clip(s, CFG["w_min"], 1.0)), keep=s >= CFG["w_min"], mt=mt.backend))
        if k % 200 == 0:
            print(f"[shard {shard}] {k}/{len(todo)} s_mc={s:.2f}", flush=True)
    out.close(); mt.close()


def weighted_pool():
    pool = {_key(r): r for r in jl_read(DIRS["data"] / "train_pool.jsonl")}
    rows = []
    for w in jl_read_glob(DIRS["data"] / "weights.jsonl"):
        if w["keep"] and w["key"] in pool:
            rows.append(dict(pool[w["key"]], w=w["w"], s_mc=w["s_mc"]))
    return rows


def figures():
    from .. import viz
    rep = json_load(DIRS["data"] / "sanitation_report.json", [])
    published = {"hi": dict(raw=7957, duplicate_pair=23, duplicate_summary=15, article_prefix_copy=2518,
                            summary_gt_50pct=11),
                 "gu": dict(raw=8457, duplicate_pair=0, duplicate_summary=113, article_prefix_copy=135,
                            summary_gt_50pct=37)}
    rows = []
    for r in rep:
        for rule in ["empty_or_short", "duplicate_pair", "duplicate_summary", "article_prefix_copy", "summary_gt_50pct"]:
            rows.append(dict(lang=r["lang"], rule=rule, source="this run", pct=100 * r["removed"].get(rule, 0) / max(1, r["raw"])))
            if rule in published[r["lang"]]:
                rows.append(dict(lang=r["lang"], rule=rule, source="CEUR Vol-3395 audit",
                                 pct=100 * published[r["lang"]][rule] / published[r["lang"]]["raw"]))
    if rows:
        df = pd.DataFrame(rows)
        for lang in ("hi", "gu"):
            viz.grouped_bars(df[df.lang == lang], f"data_sanitation_{lang}",
                             f"ILSUM {viz.LANG_LABEL[lang]} train: pairs removed by each sanitation rule",
                             "% of raw training pairs", "rule", "source", "pct",
                             caption="Prefix copies are counted even after a dateline. The published audit may define some rules differently.")
    w = pd.DataFrame(jl_read_glob(DIRS["data"] / "weights.jsonl"))
    if not w.empty:
        viz.histogram({f"{viz.LANG_LABEL[l]}": w[w.lang == l].s_mc.values for l in ("hi", "gu")},
                      "data_faithfulness_scores", "Reference faithfulness s_MC(D, R) in the training pool",
                      "MiniCheck support of reference sentences", vline=CFG["w_min"], vlabel="discard < 0.30",
                      caption="Soft weighting keeps abstractive pairs (0.3–0.75) instead of hard NLI filtering.")
    tb = jl_read(DIRS["data"] / "testbed.jsonl")
    if tb:
        rows = [dict(lang=r["lang"], variant=v, cmi=c) for r in tb for v, c in r["cmi"].items()]
        nc = jl_read(DIRS["data"] / "natural_cm.jsonl")
        rows += [dict(lang=r["lang"], variant="natural_cm", cmi=c) for r in nc for c in r["cmi"].values()]
        df = pd.DataFrame(rows).groupby(["lang", "variant"]).cmi.mean().reset_index()
        viz.grouped_bars(df, "data_cmi", "Code-mixing index of each input modality", "mean CMI",
                         "variant", "lang", "cmi", order_x=[v for v in ["native", "script_mixed", "roman", "natural_cm"] if v in set(df.variant)],
                         order_g=["hi", "gu"], color_fn=viz.color_of,
                         caption="Word-level language ID. Romanizing changes the script, not the language mix, so its CMI should stay near 0.")

### `factindic/pipeline/stage2.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/stage2.py
"""Stage 2: baselines and structured prompting.

notebook 02 : Seq2Seq baselines (IndicBARTSS, mT5-base) — train + beam-4 generation.
notebook 03 : Gemma-4 zero-shot conditions A/B1/B2/B3, faithfulness-weighted Gemma LoRA SFT,
              Llama-3.2-3B Cond A, latency probe, dev-set selection of the Stage-3 base.
"""
from __future__ import annotations

import math
import random
import time

import numpy as np
import torch

from ..config import CFG, DIRS, IS_LOCAL, MODELS, max_new_tokens
from ..data import items, load_set
from ..io import JsonlAppender, free_gpu, jl_read_glob, json_dump, json_load, peak_vram_gb
from ..text import W


def _eval_items(sets=("testbed", "natural_cm"), variants=None):
    out = []
    for s in sets:
        out += items(load_set(s), variants)
    return out


# ============================================================================ Seq2Seq
def _s2s_io(name, tok, lang, article, summary=None):
    if name == "indicbartss":
        tag = f"<2{lang}>"
        end = tok.convert_tokens_to_ids(["</s>", tag])          # kept when a long article is cut
        src = tok(article, add_special_tokens=False, truncation=True, max_length=510)["input_ids"] + end
        tgt = end[1:] + tok(summary, add_special_tokens=False, truncation=True, max_length=126)["input_ids"] + end[:1] if summary else None
    else:
        src = tok("summarize: " + article, truncation=True, max_length=512)["input_ids"]
        tgt = tok(summary, truncation=True, max_length=128)["input_ids"] if summary else None
    return src, tgt


def _s2s_load(name, path=None):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    mid = MODELS[name]
    if name == "indicbartss":
        tok = AutoTokenizer.from_pretrained(mid, do_lower_case=False, use_fast=False, keep_accents=True)
    else:
        tok = AutoTokenizer.from_pretrained(mid)
    if name == "indicbartss":            # wrong accents or language tags would corrupt the baseline silently
        for t in ("<2hi>", "<2gu>", "</s>"):
            i = tok.convert_tokens_to_ids(t)
            got = tok(t, add_special_tokens=False)["input_ids"]
            assert got == [i] and i != tok.unk_token_id, f"IndicBARTSS tokenizer splits {t}: {got}"
        for t in ("हिन्दी की किताब", "ગુજરાતી ભાષાની ચોપડી"):
            back = tok.decode(tok(t, add_special_tokens=False)["input_ids"], clean_up_tokenization_spaces=False)
            assert back.split() == t.split(), f"IndicBARTSS tokenizer changes the text: {t!r} -> {back!r}"
    # explicit fp32: transformers 5 otherwise loads the checkpoint's own dtype, which breaks GradScaler
    model = AutoModelForSeq2SeqLM.from_pretrained(path or mid, dtype=torch.float32).cuda()
    return model, tok


def train_seq2seq(name, epochs=None, lr=None, bs=8):
    """Standard (unweighted) CE on the sanitized pool — the baseline has no faithfulness weighting."""
    from transformers import get_linear_schedule_with_warmup
    from ..io import jl_read
    out_dir = DIRS["models"] / name
    if (out_dir / "config.json").exists():
        return out_dir
    epochs = epochs or CFG["s2s_epochs"]
    lr = lr or (5e-4 if name == "indicbartss" else 3e-4)
    model, tok = _s2s_load(name)
    pool = jl_read(DIRS["data"] / "train_pool.jsonl")
    if len(pool) > CFG["n_s2s_train"]:
        pool = random.Random(CFG["seed"]).sample(pool, CFG["n_s2s_train"])
    data = []
    for r in pool:
        s, t = _s2s_io(name, tok, r["lang"], r["text"], r["summary"])
        data.append((s, t))
    amp = name == "indicbartss" and not IS_LOCAL     # fp16 with loss scaling; mT5 overflows in fp16 → fp32
    dtype = torch.bfloat16 if IS_LOCAL else torch.float16
    if name == "mt5":                    # Adam's two fp32 copies of 580 M weights do not fit
        opt = torch.optim.Adafactor(model.parameters(), lr=lr, weight_decay=0.01)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    if IS_LOCAL or name == "mt5":
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
    steps = epochs * math.ceil(len(data) / bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    pad = tok.pad_token_id
    model.train()
    rng = random.Random(CFG["seed"])
    t0, step = time.time(), 0
    for ep in range(epochs):
        order = list(range(len(data)))
        rng.shuffle(order)
        for i in range(0, len(order), bs):
            batch = [data[j] for j in order[i:i + bs]]
            Ls, Lt = max(len(s) for s, _ in batch), max(len(t) for _, t in batch)
            src = torch.full((len(batch), Ls), pad); att = torch.zeros_like(src)
            tgt = torch.full((len(batch), Lt), pad)
            for k, (s, t) in enumerate(batch):
                src[k, :len(s)] = torch.tensor(s); att[k, :len(s)] = 1; tgt[k, :len(t)] = torch.tensor(t)
            src, att, tgt = src.cuda(), att.cuda(), tgt.cuda()
            micro, n_tok = (2 if IS_LOCAL else 4 if name == "mt5" else bs), (tgt != pad).sum()
            for m in range(0, len(batch), micro):            # weighted by tokens, so the size of micro changes nothing
                s, a, y = src[m:m + micro], att[m:m + micro], tgt[m:m + micro]
                with torch.autocast("cuda", dtype=dtype, enabled=amp or IS_LOCAL):
                    if name == "indicbartss":
                        dec_in, labels = y[:, :-1], y[:, 1:].clone()
                        labels[labels == pad] = -100
                        loss = model(input_ids=s, attention_mask=a, decoder_input_ids=dec_in, labels=labels).loss
                    else:
                        labels = y.clone(); labels[labels == pad] = -100
                        loss = model(input_ids=s, attention_mask=a, labels=labels).loss
                scaler.scale(loss * (y != pad).sum() / n_tok).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            if step % 100 == 0:
                print(f"{name} ep{ep} step {step}/{steps} loss {loss.item():.3f} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    model.config.use_cache = True
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    del model, opt
    free_gpu()
    return out_dir


def seq2seq_gen_job(shard, n_shards, name, **_):
    model, tok = _s2s_load(name, DIRS["models"] / name)
    model.eval()
    out = JsonlAppender(DIRS["outputs"] / f"{name}.jsonl", key_fn=lambda r: r["item"], shard=shard)
    todo = [x for i, x in enumerate(_eval_items()) if i % n_shards == shard and x[0] not in out.done]
    torch.cuda.reset_peak_memory_stats()
    for k in range(0, len(todo), 8):
        batch = todo[k:k + 8]
        enc = [_s2s_io(name, tok, r["lang"], r["variants"][v])[0] for _, r, v in batch]
        L = max(map(len, enc))
        ids = torch.full((len(enc), L), tok.pad_token_id); att = torch.zeros_like(ids)
        for i, e in enumerate(enc):
            ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
        kw = dict(num_beams=4, max_length=96, early_stopping=True)
        if name == "indicbartss":
            # one language per call because the decoder start tag is language-specific
            outs = [None] * len(batch)
            for lang in {r["lang"] for _, r, _ in batch}:
                idx = [i for i, (_, r, _) in enumerate(batch) if r["lang"] == lang]
                start = tok.convert_tokens_to_ids(f"<2{lang}>")
                torch.cuda.synchronize(); t = time.perf_counter()
                g = model.generate(input_ids=ids[idx].cuda(), attention_mask=att[idx].cuda(),
                                   decoder_start_token_id=start, pad_token_id=tok.pad_token_id,
                                   eos_token_id=tok.convert_tokens_to_ids("</s>"), **kw)
                torch.cuda.synchronize(); dt = (time.perf_counter() - t) / len(idx)
                for i, row in zip(idx, g):
                    outs[i] = (tok.decode(row, skip_special_tokens=True, clean_up_tokenization_spaces=False), dt)
        else:
            torch.cuda.synchronize(); t = time.perf_counter()
            g = model.generate(input_ids=ids.cuda(), attention_mask=att.cuda(), **kw)
            torch.cuda.synchronize(); dt = (time.perf_counter() - t) / len(batch)
            outs = [(tok.decode(row, skip_special_tokens=True), dt) for row in g]
        for (item, r, v), e, (s, dt) in zip(batch, enc, outs):
            s = s.split("</s>")[0].replace(f"<2{r['lang']}>", "").strip()     # IndicBARTSS's </s> is no special token here
            out.add(dict(item=item, uid=r["uid"], lang=r["lang"], variant=v, system=name, summary=s,
                         latency_s=dt, latency_mode="batch8", input_tokens=len(e),
                         input_fertility=len(e) / max(1, W(r["variants"][v])),
                         sum_tokens=len(tok(s, add_special_tokens=False)["input_ids"]), sum_words=W(s),
                         peak_vram_gb=peak_vram_gb()))
    out.close()


# ============================================================================ LLM prompting
def prompt_for(cond, rec, variant):
    from ..prompts import CONDITIONS
    fn = CONDITIONS["B1" if cond == "B2" else cond]
    return fn(rec["lang"], variant, rec["variants"][variant], rec["L_target"], rec=rec)


def _missing_anchors(rec, draft):
    from ..normalizer import anchors, match_anchors, salient_anchors
    sal = salient_anchors(rec["anchors"], rec["text"], rec["lang"])
    a_s = anchors(draft, rec["lang"])
    have = {k for k, ok in match_anchors(a_s, rec["anchors"], rec["text"], rec["lang"]) if ok}
    have_text = draft
    surf = []
    for k in sal:
        if k in have:
            continue
        kind, val = k.split(":", 1)
        if kind == "ent":
            ent = next((e["text"] for e in rec["anchors"]["entities"] if e["key"] == val), None)
            if ent and ent not in have_text:
                surf.append(ent)
        elif kind == "num" and val not in have_text:
            surf.append(val)
        elif kind == "date":
            surf.append(val)
    return surf[:4]


def llm_gen_job(shard, n_shards, system, model_key, cond, sets=("testbed", "natural_cm"), variants=None,
                adapter=None, **_):
    from ..llm import chat_text, generate, load_causal, n_tokens
    from ..prompts import cond_B2_step2
    path = DIRS["outputs"] / f"{system}.jsonl"
    out = JsonlAppender(path, key_fn=lambda r: r["item"], shard=shard)
    all_items = []
    for s in sets:
        all_items += items(load_set(s), variants)
    todo = [x for i, x in enumerate(all_items) if i % n_shards == shard and x[0] not in out.done]
    if not todo:
        return
    model, tok, desc = load_causal(MODELS[model_key], adapter=adapter)
    bs = CFG["gen"].get("batch_size_by_model", {}).get(model_key, CFG["gen"]["batch_size"])
    torch.cuda.reset_peak_memory_stats()
    for k in range(0, len(todo), bs):
        batch = todo[k:k + bs]
        prompts = [chat_text(tok, prompt_for(cond, r, v)) for _, r, v in batch]
        res, dt, _ = generate(model, tok, prompts, max_new_tokens=max_new_tokens(model_key))
        summ = [x[0] for x in res]
        drafts = None
        if cond == "B2":     # 2-step CoD: densify with deterministic missing-anchor list
            drafts = summ
            p2 = [chat_text(tok, cond_B2_step2(r["lang"], v, r["variants"][v], r["L_target"], r, d,
                                               _missing_anchors(r, d))) for (_, r, v), d in zip(batch, drafts)]
            res2, dt2, _ = generate(model, tok, p2, max_new_tokens=max_new_tokens(model_key))
            summ, dt = [x[0] for x in res2], dt + dt2
        for j, ((item, r, v), p, s) in enumerate(zip(batch, prompts, summ)):
            row = dict(item=item, uid=r["uid"], lang=r["lang"], variant=v, system=system, summary=s,
                       latency_s=dt / len(batch), latency_mode=f"batch{bs}", precision=desc,
                       input_tokens=n_tokens(tok, r["variants"][v]),
                       input_fertility=n_tokens(tok, r["variants"][v]) / max(1, W(r["variants"][v])),
                       prompt_tokens=n_tokens(tok, p), sum_tokens=n_tokens(tok, s), sum_words=W(s),
                       peak_vram_gb=peak_vram_gb())
            if drafts:
                row["draft"] = drafts[j]
            out.add(row)
        if (k // bs) % 10 == 0:
            print(f"[{system} shard {shard}] {k}/{len(todo)} :: {summ[0][:90]}", flush=True)
    out.close()
    del model
    free_gpu()


def latency_probe(system, model_key, cond, adapter=None, n=None):
    """Per-summary wall-clock latency at batch size 1 (the mandatory inference-time metric)."""
    from ..llm import chat_text, generate, load_causal
    path = DIRS["outputs"] / f"latency_{system}.json"
    if path.exists():
        return json_load(path)
    n = n or CFG["gen"]["latency_probe_n"]
    its = _eval_items(("testbed",))
    rng = random.Random(CFG["seed"])
    its = rng.sample(its, min(n, len(its)))
    model, tok, desc = load_causal(MODELS[model_key], adapter=adapter)
    generate(model, tok, [chat_text(tok, prompt_for(cond, its[0][1], its[0][2]))], max_new_tokens=8)  # warm-up
    times = []
    torch.cuda.reset_peak_memory_stats()
    for _, r, v in its:
        _, dt, _ = generate(model, tok, [chat_text(tok, prompt_for(cond, r, v))],
                            max_new_tokens=max_new_tokens(model_key))
        times.append(dt)
    res = dict(system=system, n=len(times), mean_s=float(np.mean(times)), p50_s=float(np.median(times)),
               p90_s=float(np.percentile(times, 90)), peak_vram_gb=peak_vram_gb(), precision=desc)
    json_dump(res, path)
    del model
    free_gpu()
    return res


def sft_examples(tok, n_per_lang=None):
    from ..prompts import cond_A
    from .stage1 import weighted_pool
    rows = weighted_pool()
    rng = random.Random(CFG["seed"])
    n = n_per_lang or CFG["n_sft_per_lang"]
    picked = []
    for lang in ("hi", "gu"):
        lr = [r for r in rows if r["lang"] == lang]
        if len(lr) < n:
            print(f"WARNING: only {len(lr)} kept weighted pairs for {lang} (< {n}); raise n_score_per_lang")
        picked += rng.sample(lr, min(n, len(lr)))
    for r in picked:
        r["article"] = r["text"]
        r["n_words"] = int(np.clip(W(r["summary"]), 25, 45))
    from ..sft import build_examples
    return build_examples(tok, picked, lambda r, art: cond_A(r["lang"], "native", art, r["n_words"]),
                          CFG["sft"]["max_len"])


# ============================================================================ dev selection
def dev_grade(system, tau=None):
    """Dev-only tuning signal (NOT the evaluation grader): router support on dev summaries, salient
    anchor recall, ROUGE-L. Score = mean(frac_supported × R_anchor_salient)."""
    from ..claims import sentence_claims
    from ..metrics import rouge
    from ..normalizer import anchor_recall_salient, anchors, salient_anchors
    path = DIRS["verify"] / f"dev_{system}.jsonl"
    tau = tau or CFG["tau_ver"]
    gens = jl_read_glob(DIRS["outputs"] / f"{system}.jsonl")
    recs = {r["uid"]: r for r in load_set("dev")}
    gens = [g for g in gens if g["uid"] in recs]
    out = JsonlAppender(path, key_fn=lambda r: r["item"])
    todo = [g for g in gens if g["item"] not in out.done]
    if todo:
        router, ner = _dev_router()
        for g in todo:
            rec = recs[g["uid"]]
            cl = sentence_claims(g["summary"])
            v = router.verify(rec, g["variant"], cl) if cl else []
            frac = sum(x["p"] >= tau for x in v) / len(v) if v else 0.0
            sal = salient_anchors(rec["anchors"], rec["text"], rec["lang"])
            ra = anchor_recall_salient(anchors(g["summary"], rec["lang"], ner.entities(g["summary"])), rec["anchors"], sal, rec["text"], rec["lang"])
            out.add(dict(item=g["item"], system=system, variant=g["variant"], lang=g["lang"], frac=frac,
                         R_anchor=ra, score=frac * ra, rougeL=rouge(rec["reference"], g["summary"])["rougeL"],
                         words=W(g["summary"])))
    out.close()
    rows = jl_read_glob(path)
    return dict(system=system, n=len(rows), score=float(np.mean([r["score"] for r in rows])) if rows else float("nan"),
                frac=float(np.mean([r["frac"] for r in rows])) if rows else float("nan"),
                R_anchor=float(np.mean([r["R_anchor"] for r in rows])) if rows else float("nan"),
                rougeL=float(np.mean([r["rougeL"] for r in rows])) if rows else float("nan"))


_ROUTER = None


def _dev_router():
    global _ROUTER
    if _ROUTER is None:
        from ..mt import Translator
        from ..ner import Tagger
        from ..verify import MiniCheck, NativeNLI, Router
        # the tagger as in stage 4: without it the summary has no entity anchors and R_anchor is always about 0
        _ROUTER = Router(Translator(cache_name="mt_cache_dev.jsonl"), MiniCheck(), NativeNLI()), Tagger(latin=False)
    return _ROUTER


def release_dev_router():
    global _ROUTER
    if _ROUTER is not None:
        _ROUTER[0].mt.close()
    _ROUTER = None
    free_gpu()

### `factindic/pipeline/stage3.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/stage3.py
"""Stage 3: anchor-aware context-contrasted decoding, K=5 candidates, CoVe baseline,
perplexity, and Lookback-Lens features."""
from __future__ import annotations

import random

import torch

from ..config import CFG, DIRS, MODELS
from ..data import items, load_set
from ..io import JsonlAppender, free_gpu, jl_read_glob, json_dump, json_load, peak_vram_gb
from ..text import W
from .stage2 import prompt_for

SELECTED = DIRS["outputs"] / "stage2_selected.json"
TUNED = DIRS["outputs"] / "stage3_params.json"

TUNE_GRID = [  # (alpha_base, alpha_anchor, beta) — small grid inside the spec's ranges
    (0.4, 0.9, 0.2), (0.2, 0.6, 0.2), (0.4, 0.9, 0.0), (0.4, 1.2, 0.4),
][: CFG["n_tune"]]


def selected():
    sel = json_load(SELECTED)
    return sel or dict(system="gemma_B1", model_key="gemma", cond="B1", adapter=None)   # if the dev selection failed


def ablation_uids():
    rng = random.Random(CFG["seed"])
    recs = load_set("testbed")
    out = []
    for lang in ("hi", "gu"):
        lr = sorted(r["uid"] for r in recs if r["lang"] == lang)
        out += rng.sample(lr, min(CFG["n_decoding_ablation_per_lang"], len(lr)))
    return sorted(out)


def _items(sets, variants=None, uids=None):
    its = []
    for s in sets:
        its += items(load_set(s), variants)
    if uids is not None:
        uids = set(uids)
        its = [x for x in its if x[1]["uid"] in uids]
    return its


def decode_job(shard, n_shards, method, system, params=None, sets=("testbed",), variants=None, uids=None,
               batch_size=6, **_):
    from ..decoding import anchor_token_ids, contrastive_generate
    from ..llm import chat_text, load_causal, n_tokens
    from ..prompts import bare_prior_prompt
    sel = selected()
    params = params or json_load(TUNED, {}) or {}
    out = JsonlAppender(DIRS["outputs"] / f"{system}.jsonl", key_fn=lambda r: r["item"], shard=shard)
    todo = [x for i, x in enumerate(_items(sets, variants, uids)) if i % n_shards == shard and x[0] not in out.done]
    if not todo:
        return
    model, tok, desc = load_causal(MODELS[sel["model_key"]], adapter=sel.get("adapter"))
    torch.cuda.reset_peak_memory_stats()
    kw = dict(alpha_base=params.get("alpha_base", 0.4), alpha_anchor=params.get("alpha_anchor", 0.9),
              beta=params.get("beta", 0.2), eta=CFG["cad"]["eta"], static_alpha=CFG["cad"]["static_alpha"],
              max_new_tokens=CFG["gen"]["max_new_tokens"])
    for k in range(0, len(todo), batch_size):
        batch = todo[k:k + batch_size]
        ctx = [chat_text(tok, prompt_for(sel["cond"], r, v)) for _, r, v in batch]
        unc = [chat_text(tok, bare_prior_prompt(r["lang"], r["L_target"])) for _, r, v in batch]
        anc = [anchor_token_ids(tok, r) for _, r, _ in batch]
        torch.cuda.synchronize()
        import time
        t = time.perf_counter()
        texts, info = contrastive_generate(model, tok, ctx, unc, anc, method, **kw)
        torch.cuda.synchronize()
        dt = (time.perf_counter() - t) / len(batch)
        for j, ((item, r, v), s) in enumerate(zip(batch, texts)):
            out.add(dict(item=item, uid=r["uid"], lang=r["lang"], variant=v, system=system, method=method,
                         params=kw, summary=s, mean_alpha=info["mean_alpha"][j], dola_layers=info["layers"],
                         latency_s=dt, latency_mode=f"batch{batch_size}", precision=desc,
                         input_fertility=n_tokens(tok, r["variants"][v]) / max(1, W(r["variants"][v])),
                         sum_tokens=n_tokens(tok, s), sum_words=W(s), peak_vram_gb=peak_vram_gb()))
        if (k // batch_size) % 10 == 0:
            print(f"[{system} shard {shard}] {k}/{len(todo)} :: {texts[0][:90]}", flush=True)
    out.close()
    del model
    free_gpu()


def tune_generate_job(shard, n_shards, **_):
    """Generate every grid config on the dev tuning subset (graded afterwards in a separate step)."""
    rng = random.Random(CFG["seed"])
    dev = load_set("dev")
    uids = []
    for lang in ("hi", "gu"):
        lr = sorted(r["uid"] for r in dev if r["lang"] == lang)
        uids += rng.sample(lr, min(CFG["cad"]["tune_docs"] // 2, len(lr)))
    for gi, (ab, aa, b) in enumerate(TUNE_GRID):
        decode_job(shard, n_shards, "anchor_cad_dola", f"tune_g{gi}", params=dict(alpha_base=ab, alpha_anchor=aa, beta=b),
                   sets=("dev",), variants=("native", "roman"), uids=uids)


def pick_params():
    from .stage2 import dev_grade, release_dev_router
    rows = []
    for gi, (ab, aa, b) in enumerate(TUNE_GRID):
        g = dev_grade(f"tune_g{gi}")
        rows.append(dict(grid=gi, alpha_base=ab, alpha_anchor=aa, beta=b, **{k: g[k] for k in ("score", "frac", "R_anchor", "rougeL")}))
    release_dev_router()
    best = max(rows, key=lambda r: (r["score"], r["rougeL"]))
    json_dump(dict(alpha_base=best["alpha_base"], alpha_anchor=best["alpha_anchor"], beta=best["beta"], grid=rows), TUNED)
    return best, rows


def samples_job(shard, n_shards, sets=("testbed", "natural_cm"), **_):
    """K−1 = 4 nucleus samples (T=0.7, top-p=0.9) per input with the selected Stage-2 setup."""
    from ..llm import chat_text, generate, load_causal
    sel = selected()
    K1 = CFG["K"] - 1
    out = JsonlAppender(DIRS["outputs"] / "stage3_samples.jsonl", key_fn=lambda r: r["item"], shard=shard)
    todo = [x for i, x in enumerate(_items(sets)) if i % n_shards == shard and x[0] not in out.done]
    if not todo:
        return
    model, tok, desc = load_causal(MODELS[sel["model_key"]], adapter=sel.get("adapter"))
    bs = max(1, CFG["gen"]["batch_size"] // 2)
    for k in range(0, len(todo), bs):
        batch = todo[k:k + bs]
        prompts = [chat_text(tok, prompt_for(sel["cond"], r, v)) for _, r, v in batch]
        res, dt, _ = generate(model, tok, prompts, max_new_tokens=CFG["gen"]["max_new_tokens"], do_sample=True,
                              num_return_sequences=K1, temperature=CFG["sample_T"], top_p=CFG["sample_top_p"],
                              seed=CFG["seed"] + k)
        for (item, r, v), cands in zip(batch, res):
            out.add(dict(item=item, uid=r["uid"], lang=r["lang"], variant=v, samples=cands,
                         latency_s=dt / len(batch)))
    out.close()
    del model
    free_gpu()


def cove_job(shard, n_shards, **_):
    from ..cove import cove_batch
    from ..llm import load_causal
    sel = selected()
    rng = random.Random(CFG["seed"])
    base = {g["item"]: g for g in jl_read_glob(DIRS["outputs"] / f"{sel['system']}.jsonl")}
    tb = _items(("testbed",), uids=ablation_uids())
    tb = rng.sample(tb, min(CFG["n_cove"], len(tb)))
    out = JsonlAppender(DIRS["outputs"] / "cove.jsonl", key_fn=lambda r: r["item"], shard=shard)
    todo = [x for i, x in enumerate(tb) if i % n_shards == shard and x[0] not in out.done and x[0] in base]
    if not todo:
        return
    model, tok, desc = load_causal(MODELS[sel["model_key"]], adapter=sel.get("adapter"))
    import time
    for k in range(0, len(todo), 4):
        batch = todo[k:k + 4]
        drafts = [base[i]["summary"] for i, _, _ in batch]
        torch.cuda.synchronize(); t = time.perf_counter()
        revised, traces = cove_batch(model, tok, [(r, v) for _, r, v in batch], drafts)
        torch.cuda.synchronize(); dt = (time.perf_counter() - t) / len(batch)
        for (item, r, v), d, s, tr in zip(batch, drafts, revised, traces):
            out.add(dict(item=item, uid=r["uid"], lang=r["lang"], variant=v, system="cove", summary=s,
                         draft=d, qa=tr, latency_s=dt + base[item].get("latency_s", 0), sum_words=W(s)))
    out.close()
    del model
    free_gpu()


def ppl_job(shard, n_shards, systems, **_):
    from ..decoding import conditional_ppl
    from ..llm import chat_text, load_causal
    sel = selected()
    recs = {r["uid"]: r for r in load_set("testbed")}
    out = JsonlAppender(DIRS["eval"] / "ppl.jsonl", key_fn=lambda r: (r["system"], r["item"]), shard=shard)
    jobs = []
    for s in systems:
        for g in jl_read_glob(DIRS["outputs"] / f"{s}.jsonl"):
            if g["uid"] in recs and (s, g["item"]) not in out.done:
                jobs.append((s, g))
    jobs = [j for i, j in enumerate(jobs) if i % n_shards == shard]
    if not jobs:
        return
    model, tok, _ = load_causal(MODELS[sel["model_key"]], adapter=sel.get("adapter"))
    for s, g in jobs:
        r = recs[g["uid"]]
        p = chat_text(tok, prompt_for(sel["cond"], r, g["variant"]))
        out.add(dict(system=s, item=g["item"], ppl=conditional_ppl(model, tok, [p], [g["summary"]])[0]))
    out.close()
    del model
    free_gpu()


def lookback_job(shard, n_shards, systems, **_):
    from ..llm import chat_text, load_causal
    from ..lookback import sentence_features
    sel = selected()
    recs = {r["uid"]: r for s in ("testbed", "dev") for r in load_set(s)}
    keep = set(ablation_uids()) | {r["uid"] for r in load_set("dev")}
    out = JsonlAppender(DIRS["verify"] / "lookback_feats.jsonl", key_fn=lambda r: (r["system"], r["item"]), shard=shard)
    jobs = [(s, g) for s in systems for g in jl_read_glob(DIRS["outputs"] / f"{s}.jsonl")
            if g["uid"] in keep and (s, g["item"]) not in out.done and g["summary"]]
    jobs = [j for i, j in enumerate(jobs) if i % n_shards == shard]
    if not jobs:
        return
    model, tok, _ = load_causal(MODELS[sel["model_key"]], adapter=sel.get("adapter"), attn_implementation="eager")
    for s, g in jobs:
        r = recs[g["uid"]]
        p = chat_text(tok, prompt_for(sel["cond"], r, g["variant"]))
        try:
            feats = sentence_features(model, tok, p, g["summary"])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            continue
        out.add(dict(system=s, item=g["item"], feats=[f.round(4).tolist() for f in feats]))
    out.close()
    del model
    free_gpu()

### `factindic/pipeline/stage4.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/stage4.py
"""Stage 4: Hybrid Verification Router + single-round candidate reranking (K=5),
capped pruning, drop-only baseline; also logs oracle-source verdicts for the H3 analysis."""
from __future__ import annotations

import time

from ..claims import sentence_claims
from ..config import CFG, DIRS
from ..data import items, load_set
from ..io import JsonlAppender, free_gpu, jl_read_glob, jl_write
from ..text import W


def _load_candidates():
    from .stage3 import selected
    sel = selected()
    greedy = {g["item"]: g for g in jl_read_glob(DIRS["outputs"] / f"{sel['system']}.jsonl")}
    acd = {g["item"]: g for g in jl_read_glob(DIRS["outputs"] / "dec_anchor_cad_dola.jsonl")}
    smp = {g["item"]: g for g in jl_read_glob(DIRS["outputs"] / "stage3_samples.jsonl")}
    return sel, greedy, acd, smp


def verify_job(shard, n_shards, **_):
    from ..mt import Translator
    from ..ner import Tagger
    from ..rerank import J_score, drop_only, prune
    from ..verify import MiniCheck, NativeNLI, Router
    sel, greedy, acd, smp = _load_candidates()
    tau, lam = CFG["tau_ver"], CFG["lambda_len"]
    out = JsonlAppender(DIRS["verify"] / "stage4.jsonl", key_fn=lambda r: r["item"], shard=shard)
    its = items(load_set("testbed")) + items(load_set("natural_cm"))
    todo = [x for i, x in enumerate(its) if i % n_shards == shard and x[0] not in out.done and x[0] in acd]
    if not todo:
        return
    router = Router(Translator(), MiniCheck(), NativeNLI())
    ner = Tagger(latin=False)
    for n, (item, rec, v) in enumerate(todo):
        t0 = time.perf_counter()
        cands = [dict(k=0, source="anchor_cad_dola", summary=acd[item]["summary"])]
        cands += [dict(k=i + 1, source="sample", summary=s) for i, s in enumerate(smp.get(item, {}).get("samples", []))]
        seen = {}
        for c in cands:
            cl = sentence_claims(c["summary"])
            key = " ".join(cl)
            if key not in seen:
                seen[key] = router.verify(rec, v, cl) if cl else []
            c["verdicts"] = seen[key]
            c.update(J_score(c["summary"], c["verdicts"], rec, tau, lam, ner))
        best = max(cands, key=lambda c: (c["J"], -c["k"]))
        final, pruned = prune(best["verdicts"], tau) if best["verdicts"] else (best["summary"], False)
        verify_s = time.perf_counter() - t0
        g = greedy.get(item)
        g_cl = sentence_claims(g["summary"]) if g else []
        g_ver = router.verify(rec, v, g_cl) if g_cl else []
        oracle = router.verify(rec, v, sentence_claims(cands[0]["summary"]), mode="oracle") if v in ("roman", "script_mixed") else None
        out.add(dict(item=item, uid=rec["uid"], lang=rec["lang"], variant=v,
                     candidates=[{k: c[k] for k in ("k", "source", "summary", "J", "frac_supported", "R_anchor", "words", "verdicts")} for c in cands],
                     selected_k=best["k"], final_summary=final, pruned=pruned, verify_seconds=verify_s,
                     greedy_summary=g["summary"] if g else "", greedy_verdicts=g_ver,
                     drop_only_summary=drop_only(g_ver, tau) if g else "",
                     oracle_verdicts_c0=oracle, mt=router.mt.backend))
        if n % 50 == 0:
            print(f"[stage4 shard {shard}] {n}/{len(todo)} J*={best['J']:.2f} k*={best['k']} pruned={pruned}", flush=True)
    out.close()
    router.mt.close()
    del router, ner
    free_gpu()


def materialize():
    """Write outputs/factindic.jsonl and outputs/drop_only.jsonl as ordinary system outputs."""
    sel, greedy, acd, smp = _load_candidates()
    rows = jl_read_glob(DIRS["verify"] / "stage4.jsonl")
    fi, do = [], []
    for r in rows:
        gen_lat = acd.get(r["item"], {}).get("latency_s", 0) + smp.get(r["item"], {}).get("latency_s", 0)
        base = dict(item=r["item"], uid=r["uid"], lang=r["lang"], variant=r["variant"])
        fi.append(dict(base, system="factindic", summary=r["final_summary"], sum_words=W(r["final_summary"]),
                       latency_s=gen_lat + r["verify_seconds"], selected_k=r["selected_k"], pruned=r["pruned"],
                       input_fertility=acd.get(r["item"], {}).get("input_fertility")))
        g = greedy.get(r["item"], {})
        do.append(dict(base, system="drop_only", summary=r["drop_only_summary"], sum_words=W(r["drop_only_summary"]),
                       latency_s=g.get("latency_s", 0) + r["verify_seconds"] / max(1, len(r["candidates"])),
                       input_fertility=g.get("input_fertility")))
    jl_write(DIRS["outputs"] / "factindic.jsonl", fi)
    jl_write(DIRS["outputs"] / "drop_only.jsonl", do)
    return len(fi)

### `factindic/pipeline/evaluate.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/evaluate.py
"""Non-circular evaluation. FactCG-DeBERTa-L is held out: it never weights training
data and never reranks candidates. Ground truth source = the aligned native article D_mono (facts are
invariant across variants by construction), so every system and modality is graded against the
same facts.

  P_fact = mean_i I[FactCG(D̃, c̃_i) ≥ τ]
  R_fact = mean_j I[FactCG(S̃, r̃_j) ≥ τ] over C_grounded(R) = {r_j : FactCG(D̃, r̃_j) ≥ τ}
"""
from __future__ import annotations

from ..claims import sentence_claims
from ..config import DIRS
from ..data import load_set
from ..io import JsonlAppender, free_gpu, jl_read_glob
from ..metrics import compression_ratio, fact_prf, rouge
from ..normalizer import anchor_prf, anchors
from ..text import W, chunk_sentences, split_sentences

TAU_EVAL = 0.5


def _records():
    return {r["uid"]: r for s in ("testbed", "natural_cm") for r in load_set(s)}


def _sub(a, kinds):
    return dict(numbers=a.get("numbers", []) if "num" in kinds else [],
                dates=a.get("dates", []) if "date" in kinds else [],
                entities=a.get("entities", []) if "ent" in kinds else [])


def ref_job(shard, n_shards, **_):
    """Translate sources + references once; mark source-grounded reference claims."""
    from ..mt import Translator
    from ..verify import FactCG, max_over_chunks
    recs = list(_records().values())
    out = JsonlAppender(DIRS["eval"] / "refs.jsonl", key_fn=lambda r: r["uid"], shard=shard)
    todo = [r for i, r in enumerate(recs) if i % n_shards == shard and r["uid"] not in out.done]
    if not todo:
        return
    mt, fc = Translator(), FactCG()
    for r in todo:
        d_en = mt.translate(split_sentences(r["text"]), r["lang"])
        rc = sentence_claims(r["reference"])
        rc_en = mt.translate(rc, r["lang"]) if rc else []
        p = max_over_chunks(fc, chunk_sentences(d_en, 400), rc_en) if rc_en else []
        out.add(dict(uid=r["uid"], d_en=d_en, ref_claims=rc, ref_en=rc_en, p_grounded=p,
                     grounded=[x >= TAU_EVAL for x in p], mt=mt.backend))
    out.close(); mt.close()
    del mt, fc
    free_gpu()


def eval_job(shard, n_shards, systems, **_):
    from ..mt import Translator
    from ..ner import Tagger
    from ..verify import FactCG, max_over_chunks
    recs = _records()
    refs = {r["uid"]: r for r in jl_read_glob(DIRS["eval"] / "refs.jsonl")}
    mt, fc, ner = Translator(), FactCG(), Tagger()
    cache = {}
    for system in systems:
        out = JsonlAppender(DIRS["eval"] / f"eval_{system}.jsonl", key_fn=lambda r: r["item"], shard=shard)
        gens = [g for g in jl_read_glob(DIRS["outputs"] / f"{system}.jsonl") if g["uid"] in recs and g["uid"] in refs]
        todo = [g for i, g in enumerate(gens) if i % n_shards == shard and g["item"] not in out.done]
        for n, g in enumerate(todo):
            rec, ref = recs[g["uid"]], refs[g["uid"]]
            S = g.get("summary") or ""
            cl = sentence_claims(S)
            cl_en = mt.translate(cl, rec["lang"]) if cl else []
            p_claims = max_over_chunks(fc, chunk_sentences(ref["d_en"], 400), cl_en, cache) if cl_en else []
            g_ref = [e for e, ok in zip(ref["ref_en"], ref["grounded"]) if ok]
            s_en = " ".join(cl_en)
            p_rec = max_over_chunks(fc, [s_en], g_ref, cache) if (g_ref and s_en) else ([0.0] * len(g_ref))
            P, R, F = fact_prf([p >= TAU_EVAL for p in p_claims], [p >= TAU_EVAL for p in p_rec])
            if not cl:
                P = float("nan")
            a_s = anchors(S, rec["lang"], ner.entities(S) if S else [])
            row = dict(item=g["item"], uid=g["uid"], lang=rec["lang"], variant=g["variant"], system=system,
                       n_claims=len(cl), p_claims=p_claims, P_fact=P, R_fact=R, F1_fact=F,
                       n_ref_grounded=len(g_ref), words=W(S), CR=compression_ratio(S, rec["text"]),
                       empty=not S.strip(), **rouge(rec["reference"], S))
            row.update(anchor_prf(a_s, rec["anchors"], rec["ref_anchors"], rec["text"], rec["lang"]))
            for kind, ks in (("num", ("num", "date")), ("ent", ("ent",))):
                sub = anchor_prf(_sub(a_s, ks), _sub(rec["anchors"], ks), _sub(rec["ref_anchors"], ks), rec["text"], rec["lang"])
                row.update({f"{k}_{kind}": v for k, v in sub.items() if k in ("P_anchor", "R_anchor", "F1_anchor")})
            out.add(row)
            if n % 200 == 0:
                print(f"[eval {system} shard {shard}] {n}/{len(todo)} P={P:.2f} R={R:.2f}", flush=True)
        out.close()
    mt.close()
    del mt, fc, ner
    free_gpu()


def bertscore_job(shard, n_shards, systems, **_):
    from ..metrics import BertScorer
    recs = _records()
    out = JsonlAppender(DIRS["eval"] / "bertscore.jsonl", key_fn=lambda r: (r["system"], r["item"]), shard=shard)
    bs = BertScorer()
    for lang in ("hi", "gu"):
        bs.fit_baseline(lang, [r["reference"] for r in recs.values() if r["lang"] == lang])
    for system in systems:
        gens = [g for g in jl_read_glob(DIRS["outputs"] / f"{system}.jsonl")
                if g["uid"] in recs and (system, g["item"]) not in out.done]
        gens = [g for i, g in enumerate(gens) if i % n_shards == shard]
        for lang in ("hi", "gu"):
            gl = [g for g in gens if g["lang"] == lang]
            for i in range(0, len(gl), 64):
                b = gl[i:i + 64]
                sc = bs.score([g["summary"] for g in b], [recs[g["uid"]]["reference"] for g in b], lang)
                for g, s in zip(b, sc):
                    out.add(dict(system=system, item=g["item"], bertscore=s, baseline=bs.baseline[lang]))
    out.close()
    del bs
    free_gpu()

### `factindic/pipeline/analysis.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/analysis.py
"""Aggregation, hypothesis tests, and figures (CPU only). Produces eval/results.json (consumed by
the results dashboard), eval/main_table.csv, and figures/*.png|svg|csv."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .. import viz
from ..config import CFG, DIRS, H1_DIR
from ..io import jl_read, jl_read_glob, json_dump, json_load
from ..stats import bootstrap_mean, cohen_kappa, fe_ols, paired_bootstrap

MAIN = ["indicbartss", "mt5", "llama_A", "gemma_A", "gemma_B1", "gemma_B2", "gemma_B3", "gemma_sft",
        "dec_anchor_cad_dola", "drop_only", "cove", "factindic"]
DECODING = ["dec_greedy", "dec_static_cad", "dec_adacad", "dec_dola", "dec_anchor_cad_dola"]
LABEL = {"indicbartss": "IndicBARTSS (Exp 1a)", "mt5": "mT5-base (Exp 1b)", "llama_A": "Llama-3.2-3B · A",
         "gemma_A": "Gemma-4 · A standard", "gemma_B1": "Gemma-4 · B1 anchors", "gemma_B2": "Gemma-4 · B2 CoD",
         "gemma_B3": "Gemma-4 · B3 verb hint", "gemma_sft": "Gemma-4 LoRA-SFT",
         "dec_greedy": "Greedy", "dec_static_cad": "Static CAD", "dec_adacad": "AdaCAD", "dec_dola": "DoLa",
         "dec_anchor_cad_dola": "Anchor-CAD + DoLa", "drop_only": "Drop-only repair", "cove": "CoVe self-rewrite",
         "factindic": "FactIndic (full)"}


def load_all():
    sel = json_load(DIRS["outputs"] / "stage2_selected.json", {}) or {}
    frames = []
    names = set(MAIN) | set(DECODING)
    for f in DIRS["eval"].glob("eval_*.jsonl"):
        if ".shard" in f.name:
            continue
        names.add(f.stem.replace("eval_", ""))
    for s in sorted(names):
        rows = jl_read_glob(DIRS["eval"] / f"eval_{s}.jsonl")
        if rows:
            frames.append(pd.DataFrame(rows))
    if not frames:
        return pd.DataFrame(), sel
    ev = pd.concat(frames, ignore_index=True)
    meta = []
    for s in ev.system.unique():
        for g in jl_read_glob(DIRS["outputs"] / f"{s}.jsonl"):
            meta.append(dict(system=s, item=g["item"], latency_s=g.get("latency_s"), sum_tokens=g.get("sum_tokens"),
                             input_fertility=g.get("input_fertility"), peak_vram_gb=g.get("peak_vram_gb"),
                             mean_alpha=g.get("mean_alpha")))
    ev = ev.merge(pd.DataFrame(meta), on=["system", "item"], how="left")
    bs = pd.DataFrame(jl_read_glob(DIRS["eval"] / "bertscore.jsonl"))
    if not bs.empty:
        ev = ev.merge(bs[["system", "item", "bertscore"]], on=["system", "item"], how="left")
    ppl = pd.DataFrame(jl_read_glob(DIRS["eval"] / "ppl.jsonl"))
    if not ppl.empty:
        ev = ev.merge(ppl, on=["system", "item"], how="left")
    # dec_greedy = the selected Stage-2 system, restricted to the decoding-ablation articles
    if sel.get("system") and sel["system"] in set(ev.system):
        from .stage3 import ablation_uids
        uids = set(ablation_uids())
        g = ev[(ev.system == sel["system"]) & ev.uid.isin(uids)].copy()
        g["system"] = "dec_greedy"
        ev = pd.concat([ev[ev.system != "dec_greedy"], g], ignore_index=True)
    return ev, sel


METRICS = ["P_fact", "R_fact", "F1_fact", "P_anchor", "R_anchor", "F1_anchor", "rouge1", "rouge2", "rougeL",
           "bertscore", "words", "sum_tokens", "CR", "latency_s", "peak_vram_gb", "input_fertility"]


def main_table(ev):
    rows = []
    for (s, lang, v), d in ev.groupby(["system", "lang", "variant"]):
        r = dict(system=s, label=LABEL.get(s, s), lang=lang, variant=v, n=len(d), empty_rate=float(d["empty"].mean()))
        for m in METRICS:
            if m in d:
                r[m] = float(d[m].astype(float).mean())
        for m in ("P_fact", "F1_fact"):
            b = bootstrap_mean(d[m], CFG["bootstrap"])
            r[f"{m}_lo"], r[f"{m}_hi"] = b["lo"], b["hi"]
        rows.append(r)
    t = pd.DataFrame(rows)
    t.to_csv(DIRS["eval"] / "main_table.csv", index=False)
    return t


def overall(ev, systems, metric, uids=None, variants=("native", "script_mixed", "roman")):
    rows = []
    for s in systems:
        d = ev[(ev.system == s) & ev.variant.isin(variants)]
        if uids is not None:
            d = d[d.uid.isin(uids)]
        if d.empty:
            continue
        b = bootstrap_mean(d[metric], CFG["bootstrap"])
        rows.append(dict(system=s, label=LABEL.get(s, s), mean=b["mean"], lo=b["lo"], hi=b["hi"], n=b["n"]))
    return pd.DataFrame(rows)


def paired(ev, a, b, metric, uids=None):
    x = ev[ev.system == a].set_index("item")[metric]
    y = ev[ev.system == b].set_index("item")[metric]
    common = x.index.intersection(y.index)
    if uids is not None:
        keep = set(ev[ev.uid.isin(uids)].item)
        common = [c for c in common if c in keep]
    return paired_bootstrap(x[common].values, y[common].values, CFG["bootstrap"])


# ------------------------------------------------------------------------------------------ H1
MODEL_LABEL = {"gemma4_e2b": "Gemma-4", "sarvam1_sft": "Sarvam-1", "llama32_3b": "Llama-3.2"}
MEASURES = {"halluc_mc": "MiniCheck", "halluc_nli": "Native NLI", "anchor_err": "Names & numbers"}


def h1_sensitivity(ps):
    """Does the gate verdict depend on Llama, and does fertility matter once Romanization is held fixed?"""
    ps = ps.assign(model_doc=ps.model + "_" + ps.doc_key)
    specs = {"within model (gate)": "halluc ~ input_fertility + log_len + C(doc_key) + C(model)",
             "across models (gate)": "halluc ~ input_fertility + log_len + C(cell)",
             "within model, script fixed": "halluc ~ input_fertility + log_len + C(model_doc) + C(script)"}
    subsets = {"all": ps, "no_llama": ps[ps.model != "llama32_3b"]}
    tests = {k: {n: fe_ols(d, f, "input_fertility", cluster="doc_key") for n, f in specs.items()}
             for k, d in subsets.items()}
    fig = viz.coef_plot(tests, {"all": "All models", "no_llama": "Without Llama"}, "h1_sensitivity",
                        "H1: the only positive result goes away once script is held fixed")
    return dict(tests=tests, fig=str(fig))


def h1_measures(ps):
    """Paired native → Romanized change per model × language under each error measure."""
    rows = []
    for (m, lang), d in ps.groupby(["model", "lang"]):
        for col, lab in MEASURES.items():
            w = d.pivot_table(index="doc_key", columns="script", values=col).dropna()
            if len(w) >= 10 and {"native", "roman"} <= set(w.columns):
                b = paired_bootstrap(w["roman"].values, w["native"].values, CFG["bootstrap"])
                rows.append(dict(cell=f"{MODEL_LABEL.get(m, m)}\n{viz.LANG_LABEL[lang]}", measure=lab, change=b["diff"],
                                 lo=b["lo"], hi=b["hi"], n=b["n"]))
    t = pd.DataFrame(rows)
    fig = viz.grouped_bars(t, "h1_shift_by_measure", "Effect of Romanized input depends on which error measure is used",
                           "change in error rate (Romanized − native)", "cell", "measure", "change", err=("lo", "hi"),
                           order_g=list(MEASURES.values()),
                           caption="Same articles in both scripts; bars are paired differences with 95% bootstrap CIs.")
    return dict(table=t.to_dict("records"), fig=str(fig))


def h1_verifier_agreement():
    c = pd.DataFrame(jl_read(H1_DIR / "claim_scores.jsonl"))
    if c.empty:
        return None
    tau, out = CFG["tau_ver"], {}
    for name, d in [("all", c), *c.groupby("script")]:
        mc, nli = (d.p_minicheck >= tau).astype(int), (d.p_nli >= tau).astype(int)
        out[name] = dict(n=len(d), kappa=cohen_kappa(mc, nli), agreement=float((mc == nli).mean()),
                         minicheck_supported=float(mc.mean()), nli_supported=float(nli.mean()),
                         routed_to_nli=int((d.route == "nli").sum()))
    return out


def h1_section():
    out = {}
    p_tab = H1_DIR / "fertility_table.csv"
    p_sum = H1_DIR / "h1_per_summary.csv"
    if p_tab.exists():
        out["fertility_fig"] = str(viz.h1_fertility(pd.read_csv(p_tab)))
    if p_sum.exists():
        ps = pd.read_csv(p_sum)
        ps = ps[ps.script_ok & ps.has_claims]
        out["shift_fig"] = str(viz.h1_shift(ps))
        out["sensitivity"] = h1_sensitivity(ps)
        out["by_measure"] = h1_measures(ps)
        out["verifier_agreement"] = h1_verifier_agreement()
    tests = json_load(H1_DIR / "h1_tests.json")
    if tests:
        out["coef_fig"] = str(viz.coef_plot(
            tests, {"halluc": "Router", "halluc_mc": "MiniCheck"}, "h1_coefficients",
            "H1 gate tests: no reliable positive effect of fertility"))
        out["tests"] = {k: tests[k] for k in ("decision", "model_contrasts_halluc") if k in tests}
        out["regressions"] = {o: tests[o] for o in ("halluc", "halluc_mc", "halluc_nli") if o in tests}
    return out


# ------------------------------------------------------------------------------------------ H2
def h2_section(ev, sel):
    systems = [s for s in ["gemma_A", sel.get("system"), "factindic", "indicbartss"] if s and s in set(ev.system)]
    systems = list(dict.fromkeys(systems))
    rows, tests = [], {}
    for s in systems:
        for lang in ("hi", "gu"):
            for v in ("native", "script_mixed", "roman", "natural_cm"):
                d = ev[(ev.system == s) & (ev.lang == lang) & (ev.variant == v)]
                if d.empty:
                    continue
                b = bootstrap_mean(d.P_fact, CFG["bootstrap"])
                rows.append(dict(system=LABEL.get(s, s), lang=lang, variant=v, mean=b["mean"], lo=b["lo"], hi=b["hi"], n=b["n"]))
        d = ev[(ev.system == s) & ev.variant.isin(["native", "script_mixed", "roman"])].copy()
        tests[s] = {}
        for lang in ("hi", "gu"):
            dl = d[d.lang == lang]
            tests[s][lang] = {f"native→{v}": paired_bootstrap(
                dl[dl.variant == v].set_index("uid").P_fact.reindex(dl[dl.variant == "native"].uid).values,
                dl[dl.variant == "native"].set_index("uid").P_fact.values, CFG["bootstrap"])
                for v in ("script_mixed", "roman")}
        d["gu"] = (d.lang == "gu").astype(int)
        tests[s]["regression"] = {
            "script_mixed": fe_ols(d, "P_fact ~ C(variant, Treatment('native')) + C(uid)",
                                   "C(variant, Treatment('native'))[T.script_mixed]"),
            "roman": fe_ols(d, "P_fact ~ C(variant, Treatment('native')) + C(uid)",
                            "C(variant, Treatment('native'))[T.roman]"),
            "gujarati_vs_hindi_native": fe_ols(d[d.variant == "native"], "P_fact ~ gu", "gu"),
        }
    df = pd.DataFrame(rows)
    figs = {}
    if not df.empty:
        figs["precision"] = str(viz.lines_by_variant(
            df, "h2_precision_by_variant", "H2: claim precision by input modality",
            "Claim-level factual precision (FactCG)", series_col="system",
            caption="Same articles and facts across variants; 95% bootstrap CIs."))
    err = []
    base = ev[ev.system.isin(systems) & ev.variant.isin(["native", "script_mixed", "roman"])]
    for (lang, v), d in base.groupby(["lang", "variant"]):
        n_num = d.n_unsup_num.sum(); n_ent = d.n_unsup_ent.sum(); n_anc = d.n_anchor_S.sum()
        err.append(dict(lang=lang, variant=v, kind="numeric", rate=n_num / max(1, n_anc)))
        err.append(dict(lang=lang, variant=v, kind="entity", rate=n_ent / max(1, n_anc)))
    if err:
        e = pd.DataFrame(err)
        e["group"] = e.lang.map(viz.LANG_LABEL) + " · " + e.kind
        figs["errors"] = str(viz.grouped_bars(
            e, "h2_anchor_errors", "H2: unsupported numbers and entities per summary anchor", "share of anchors unsupported",
            "variant", "group", "rate", order_x=["native", "script_mixed", "roman"], pct=True,
            caption="Gujarati/Gujlish vs Hindi/Hinglish baseline error rates (pooled over the H2 systems)."))
    return dict(tests=tests, figs=figs, table=df.to_dict("records"))


# ------------------------------------------------------------------------------------------ H4
def h4_section(ev):
    from .stage3 import ablation_uids
    uids = set(ablation_uids())
    present = [s for s in DECODING if s in set(ev.system)]
    figs, tests = {}, {}
    for metric, lab, name in (("F1_anchor_num", "Numeric/date anchor F1", "h4_numeric_f1"),
                              ("F1_anchor_ent", "Entity anchor F1", "h4_entity_f1"),
                              ("P_fact", "Claim precision (FactCG)", "h4_claim_precision")):
        df = overall(ev, present, metric, uids)
        if not df.empty:
            figs[name] = str(viz.dot_ci(df, name, f"H4: {lab} by decoding method", lab,
                                        highlight={LABEL["dec_anchor_cad_dola"]}))
    if "ppl" in ev:
        df = overall(ev, present, "ppl", uids)
        if not df.empty:
            figs["h4_ppl"] = str(viz.dot_ci(df, "h4_perplexity", "H4: fluency cost — conditional perplexity",
                                            "Perplexity under the plain model (lower = more fluent)",
                                            highlight={LABEL["dec_anchor_cad_dola"]}))
    for ref in ("dec_static_cad", "dec_adacad", "dec_greedy"):
        if ref in present and "dec_anchor_cad_dola" in present:
            tests[f"anchor_cad_dola−{ref}"] = {m: paired(ev, "dec_anchor_cad_dola", ref, m, uids)
                                               for m in ("F1_anchor_num", "F1_anchor_ent", "P_fact", "F1_fact")
                                               + (("ppl",) if "ppl" in ev else ())}
    return dict(tests=tests, figs=figs)


# ------------------------------------------------------------------------------------------ pipeline
def pipeline_section(ev, sel):
    order = [s for s in ["indicbartss", "mt5", "llama_A", "gemma_A", "gemma_B1", "gemma_B2", "gemma_B3",
                         "gemma_sft", "dec_anchor_cad_dola", "drop_only", "factindic"] if s in set(ev.system)]
    figs, tests = {}, {}
    test_uids = set(ev[ev.variant.isin(["native", "script_mixed", "roman"])].uid)
    for metric, lab in (("F1_fact", "Claim-level factual F1"), ("P_fact", "Claim-level factual precision")):
        df = overall(ev, order, metric, test_uids)
        if not df.empty:
            figs[f"ladder_{metric}"] = str(viz.dot_ci(
                df, f"pipeline_ladder_{metric}", f"Each stage's contribution — {lab}", lab,
                highlight={LABEL["factindic"]},
                caption="Testbed, all three input variants, FactCG held-out grader, 95% bootstrap CIs."))
    pr = []
    for s in order + (["cove"] if "cove" in set(ev.system) else []):
        d = ev[(ev.system == s) & ev.variant.isin(["native", "script_mixed", "roman"])]
        if d.empty:
            continue
        bp, br = bootstrap_mean(d.P_fact), bootstrap_mean(d.R_fact)
        kind = "FactIndic" if s == "factindic" else ("repair" if s in ("drop_only", "cove") else "baseline")
        pr.append(dict(system=LABEL.get(s, s), kind=kind, P=bp["mean"], P_lo=bp["lo"], P_hi=bp["hi"],
                       R=br["mean"], R_lo=br["lo"], R_hi=br["hi"]))
    if pr:
        figs["pr"] = str(viz.pr_scatter(pd.DataFrame(pr), "pipeline_precision_recall",
                                        "Claim precision and recall of every system"))
    lat = overall(ev, order, "latency_s", test_uids)
    if not lat.empty:
        figs["latency"] = str(viz.dot_ci(lat, "pipeline_latency", "Inference cost per summary",
                                         "seconds per summary (batched generation + verification)",
                                         highlight={LABEL["factindic"]}))
    base = sel.get("system")
    for ref in ("gemma_A", base, "drop_only", "cove", "indicbartss"):
        if ref and ref in set(ev.system) and "factindic" in set(ev.system):
            tests[f"factindic−{ref}"] = {m: paired(ev, "factindic", ref, m) for m in ("P_fact", "R_fact", "F1_fact", "F1_anchor")}
    if "gemma_A" in set(ev.system) and base in set(ev.system):
        tests[f"{base}−gemma_A (factuality-aware vs standard prompt)"] = {
            m: paired(ev, base, "gemma_A", m) for m in ("P_fact", "R_fact", "F1_fact", "F1_anchor")}
    # stage-4 diagnostics
    s4 = jl_read_glob(DIRS["verify"] / "stage4.jsonl")
    diag = {}
    if s4:
        ks = pd.Series([r["selected_k"] for r in s4]).value_counts(normalize=True).sort_index()
        diag["selected_k_share"] = ks.to_dict()
        diag["pruned_rate"] = float(np.mean([r["pruned"] for r in s4]))
        routes = [v["route"] for r in s4 for c in r["candidates"] for v in c["verdicts"]]
        var = [r["variant"] for r in s4 for c in r["candidates"] for v in c["verdicts"]]
        rt = pd.DataFrame(dict(route=routes, variant=var))
        share = rt.assign(nli=rt.route == "nli").groupby("variant").nli.mean().reset_index()
        share["lang"] = "all"
        diag["nli_route_share"] = share.set_index("variant").nli.to_dict()
        figs["route_share"] = str(viz.grouped_bars(
            share.rename(columns={"nli": "share"}), "stage4_route_share",
            "Round-trip router: share of claims whose anchors failed MT (→ native NLI)", "share of claims",
            "variant", "lang", "share", pct=True,
            order_x=[v for v in ["native", "script_mixed", "roman", "natural_cm"] if v in set(share.variant)]))
    return dict(tests=tests, figs=figs, stage4=diag)


def run_all():
    ev, sel = load_all()
    res = dict(selected=sel)
    parts = dict(h1=h1_section)
    if not ev.empty:
        res["main_table"] = main_table(ev).to_dict("records")
        parts.update(h2=lambda: h2_section(ev, sel), h4=lambda: h4_section(ev), pipeline=lambda: pipeline_section(ev, sel))
    for k, f in parts.items():               # one failing section must not lose the others
        try:
            res[k] = f()
        except Exception:
            import traceback
            traceback.print_exc()
            res[k] = dict(error=traceback.format_exc()[-600:])
    res["stage3_params"] = json_load(DIRS["outputs"] / "stage3_params.json")
    res["stage2_dev"] = json_load(DIRS["outputs"] / "stage2_dev_scores.json")
    res["sanitation"] = json_load(DIRS["data"] / "sanitation_report.json")
    human = json_load(DIRS["eval"] / "human_results.json")
    if human:
        res["h3"] = human
    json_dump(res, DIRS["eval"] / "results.json")
    return res

### `factindic/pipeline/human.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/human.py
"""H3 (verifier agreement by modality), inter-annotator κ, FactCG-vs-human validation, τ tuning,
and the Lookback-Lens hallucination detector. Writes eval/human_results.json."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .. import viz
from ..config import CFG, DIRS
from ..data import load_set
from ..io import free_gpu, jl_read_glob, json_dump
from ..stats import cohen_kappa, macro_f1
from ..text import W, split_sentences

VERIFIERS = {"tv_minicheck": "Translate→MiniCheck", "tv_factcg": "Translate→FactCG",
             "native_nli": "Native mDeBERTa-XNLI", "router": "Round-trip router"}


def export_sheets():
    from ..annotate import export_claim_sheets, export_naturalness
    recs = {r["uid"]: r for s in ("testbed", "natural_cm") for r in load_set(s)}
    pool = []
    for s in ("gemma_A", "gemma_B1", "gemma_B2", "gemma_B3", "gemma_sft", "factindic", "indicbartss", "llama_A"):
        for g in jl_read_glob(DIRS["outputs"] / f"{s}.jsonl"):
            if g["uid"] in recs:
                pool.append(dict(system=s, item=g["item"], uid=g["uid"], lang=g["lang"], variant=g["variant"],
                                 summary=g["summary"]))
    n_s, n_c = export_claim_sheets(pool, recs)
    n_nat = export_naturalness(load_set("testbed"))
    return dict(summaries=n_s, claims=n_c, naturalness_items=n_nat)


def score_annotated(lab):
    """Deployment-mode scores from every verifier for each human-labelled claim (GPU)."""
    from ..mt import Translator
    from ..verify import FactCG, MiniCheck, NativeNLI, Router, max_over_chunks
    recs = {r["uid"]: r for s in ("testbed", "natural_cm") for r in load_set(s)}
    router = Router(Translator(), MiniCheck(), NativeNLI())
    fc = FactCG()
    rows = []
    for (uid, variant), d in lab.groupby(["uid", "variant"]):
        rec = recs[uid]
        claims = list(d.claim)
        v = router.verify(rec, variant, claims)
        src = router.prepare_source(rec, variant)
        p_fc = max_over_chunks(fc, [c for c in src["en_chunks"]], [x["claim_en"] for x in v])
        for (_, r), x, pf in zip(d.iterrows(), v, p_fc):
            rows.append(dict(sid=r.sid, claim_idx=r.claim_idx, split=r.split, lang=r.lang, variant=variant,
                             system=r.system, gold=int(r.gold), error_type=r.error_type,
                             tv_minicheck=x["p_minicheck"], native_nli=x["p_nli"], tv_factcg=pf,
                             router=x["p"], route=x["route"]))
    router.mt.close()
    del router, fc
    free_gpu()
    return pd.DataFrame(rows)


def tune_tau(df, col):
    dev = df[df.split == "dev"]
    if dev.empty or dev.gold.nunique() < 2:
        return CFG["tau_ver"]
    grid = np.linspace(0.1, 0.9, 17)
    return float(max(grid, key=lambda t: macro_f1(dev.gold, (dev[col] >= t).astype(int))))


def h3_with_labels(scored):
    taus = {k: tune_tau(scored, k) for k in VERIFIERS}
    test = scored[scored.split == "test"]
    rows = []
    rng = np.random.default_rng(CFG["seed"])
    for v in sorted(test.variant.unique()):
        d = test[test.variant == v]
        for k, lab in VERIFIERS.items():
            pred = (d[k] >= taus[k]).astype(int)
            kap = cohen_kappa(d.gold, pred)
            bk = []
            for _ in range(CFG["bootstrap"]):
                idx = rng.integers(0, len(d), len(d))
                bk.append(cohen_kappa(d.gold.values[idx], pred.values[idx]))
            bk = np.array([x for x in bk if x == x])
            rows.append(dict(variant=v, verifier=lab, kappa=kap, lo=np.percentile(bk, 2.5) if len(bk) else np.nan,
                             hi=np.percentile(bk, 97.5) if len(bk) else np.nan, macro_f1=macro_f1(d.gold, pred), n=len(d)))
    res = pd.DataFrame(rows)
    # H3 test: on code-mixed variants, does the router close the translate→MiniCheck gap? (paired correctness)
    tests = {}
    for v in ("script_mixed", "roman", "natural_cm"):
        d = test[test.variant == v]
        if len(d) < 10:
            continue
        a = ((d.router >= taus["router"]).astype(int) == d.gold).astype(float).values
        b = ((d.tv_minicheck >= taus["tv_minicheck"]).astype(int) == d.gold).astype(float).values
        from ..stats import paired_bootstrap
        tests[f"router−translate_minicheck accuracy ({v})"] = paired_bootstrap(a, b, CFG["bootstrap"])
    fig = viz.grouped_bars(res, "h3_agreement", "H3: verifier agreement with human claim labels (Cohen's κ)",
                           "Cohen's κ vs. human gold", "variant", "verifier", "kappa", err=("lo", "hi"),
                           order_x=[v for v in ["native", "script_mixed", "roman", "natural_cm"] if v in set(res.variant)],
                           order_g=list(VERIFIERS.values()),
                           caption="Thresholds tuned on the 100-summary dev split; κ on the 200-summary test split.")
    return dict(taus=taus, table=res.to_dict("records"), tests=tests, fig=str(fig))


def h3_proxy():
    """Label-free proxy: does verification on the (back-transliterated) code-mixed source agree with
    verification on the aligned native source for the same claims?"""
    rows = []
    for r in jl_read_glob(DIRS["verify"] / "stage4.jsonl"):
        if not r.get("oracle_verdicts_c0"):
            continue
        dep = r["candidates"][0]["verdicts"]
        for a, b in zip(dep, r["oracle_verdicts_c0"]):
            for k, lab in (("p_minicheck", "Translate→MiniCheck"), ("p_nli", "Native mDeBERTa-XNLI"), ("p", "Round-trip router")):
                rows.append(dict(variant=r["variant"], lang=r["lang"], verifier=lab,
                                 agree=int((a[k] >= CFG["tau_ver"]) == (b["p"] >= CFG["tau_ver"]))))
    if not rows:
        return None
    df = pd.DataFrame(rows).groupby(["variant", "verifier"]).agree.mean().reset_index()
    fig = viz.grouped_bars(df, "h3_proxy_oracle_agreement",
                           "H3 proxy: agreement with verification on the aligned native source",
                           "agreement rate", "variant", "verifier", "agree", pct=True,
                           caption="No human labels needed; the native-source router decision is the reference.")
    return dict(table=df.to_dict("records"), fig=str(fig))


def lookback_detector(labels_from="factcg", scored=None):
    """Train on half of the ablation articles, test on the other half (split by article)."""
    from sklearn.metrics import roc_auc_score, roc_curve
    from ..lookback import train_detector
    feats = jl_read_glob(DIRS["verify"] / "lookback_feats.jsonl")
    if not feats:
        return None
    ev = {}
    for name in sorted({f.name.split(".")[0] for f in DIRS["eval"].glob("eval_*.jsonl")}):     # shard files included
        for r in jl_read_glob(DIRS["eval"] / f"{name}.jsonl"):
            ev[(r["system"], r["item"])] = r
    X, y, groups = [], [], []
    gen_cache = {}
    for fr in feats:
        e = ev.get((fr["system"], fr["item"]))
        if not e:
            continue
        if fr["system"] not in gen_cache:
            gen_cache[fr["system"]] = {g["item"]: g for g in jl_read_glob(DIRS["outputs"] / f"{fr['system']}.jsonl")}
        g = gen_cache[fr["system"]].get(fr["item"])
        if not g:
            continue
        sents = split_sentences(g["summary"])
        keep = [i for i, s in enumerate(sents) if W(s) >= 3]
        if len(keep) != len(e["p_claims"]) or len(sents) != len(fr["feats"]):
            continue
        for i, p in zip(keep, e["p_claims"]):
            X.append(fr["feats"][i]); y.append(int(p < 0.5)); groups.append(e["uid"])
    if len(set(y)) < 2 or len(X) < 50:
        return None
    X, y = np.array(X), np.array(y)
    uids = sorted(set(groups))
    rng = np.random.default_rng(CFG["seed"])
    tr_u = set(rng.choice(uids, len(uids) // 2, replace=False))
    tr = np.array([g in tr_u for g in groups])
    clf = train_detector(X[tr], y[tr])
    s = clf.predict_proba(X[~tr])[:, 1]
    auc = roc_auc_score(y[~tr], s)
    fpr, tpr, _ = roc_curve(y[~tr], s)
    base = X[~tr][:, :-1].mean(1)              # mean lookback ratio alone (lower → more hallucination)
    auc_b = roc_auc_score(y[~tr], -base)
    fpr_b, tpr_b, _ = roc_curve(y[~tr], -base)
    fig = viz.roc({"Lookback-Lens LR (per-head)": (fpr, tpr, auc), "Mean lookback ratio": (fpr_b, tpr_b, auc_b)},
                  "lookback_roc", "Detecting unsupported sentences from attention alone",
                  caption=f"Labels: held-out FactCG; train/test split by article; n_test={int((~tr).sum())}.")
    return dict(auroc=float(auc), auroc_mean_ratio=float(auc_b), n_train=int(tr.sum()), n_test=int((~tr).sum()),
                fig=str(fig), labels=labels_from)


def run(gpu=True):
    from ..annotate import import_claim_labels
    res = {}
    try:
        lab, kappa = import_claim_labels()
    except FileNotFoundError:
        lab, kappa = None, {}
    res["inter_annotator"] = kappa
    if lab is not None and gpu:
        scored = score_annotated(lab)
        scored.to_csv(DIRS["eval"] / "h3_scored_claims.csv", index=False)
        res["h3"] = h3_with_labels(scored)
        test = scored[scored.split == "test"]
        res["factcg_vs_human"] = dict(kappa=cohen_kappa(test.gold, (test.tv_factcg >= 0.5).astype(int)),
                                      n=len(test))
        res["error_types"] = test[test.gold == 0].error_type.value_counts().to_dict()
    res["h3_proxy"] = h3_proxy()
    res["lookback"] = lookback_detector()
    json_dump(res, DIRS["eval"] / "human_results.json")
    return res

### `factindic/pipeline/auto.py`

In [ ]:
%%writefile /tmp/factindic_src/factindic/pipeline/auto.py
"""Everything after notebook 01 in one unattended run.

A queue of small jobs runs on all GPUs. Each job is its own process, so an error or a crash stops only
that job; it is retried once, and later jobs use whatever exists. A short smoke test of the whole
pipeline runs first. Optional jobs are dropped at `soft_h`, everything stops at `hard_h`, and the
analysis always runs. Finished work is never repeated, so a rerun continues where the last one stopped.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path

from ..config import CFG, DIRS, IS_KAGGLE, IS_LOCAL, MODELS, ROOT
from ..io import free_gpu, jl_read_glob, json_dump, json_load

SELECTED = DIRS["outputs"] / "stage2_selected.json"
ADAPTER = DIRS["models"] / "gemma_sft" / "final"
SMOKE = Path("/tmp/factindic_smoke") if IS_KAGGLE else ROOT.parent / "smoke"


# --------------------------------------------------------------------------- jobs not found elsewhere
def prep(shard, n_shards, **_):
    from ..annotate import export_naturalness
    from ..data import load_set
    from . import stage1
    stage1.build_all()
    stage1.prepare_weights()
    export_naturalness(load_set("testbed"))


def sft(shard, n_shards, **_):
    from ..llm import load_tokenizer
    from ..sft import train_lora
    from .stage2 import sft_examples
    if (ADAPTER / "adapter_config.json").exists():
        return
    ex = sft_examples(load_tokenizer(MODELS["gemma"]))
    print(len(ex), "SFT examples", flush=True)
    train_lora(MODELS["gemma"], ex, ADAPTER.parent, CFG["sft"])
    shutil.rmtree(ADAPTER.parent / "ckpt", ignore_errors=True)


def sft_gen(shard, n_shards, **kw):
    from .stage2 import llm_gen_job
    if not (ADAPTER / "adapter_config.json").exists():
        raise RuntimeError("no SFT adapter (the sft job failed)")
    llm_gen_job(shard, n_shards, system="gemma_sft", model_key="gemma", cond="A", adapter=str(ADAPTER), **kw)


def s2s(shard, n_shards, name, **_):
    from .stage2 import _eval_items, seq2seq_gen_job, train_seq2seq
    done = {r["item"] for r in jl_read_glob(DIRS["outputs"] / f"{name}.jsonl")}
    if {x[0] for x in _eval_items()} <= done:
        return
    train_seq2seq(name)
    seq2seq_gen_job(0, 1, name=name)
    if not IS_LOCAL:                     # 1-2 GB, not needed again; a laptop has room to keep them
        shutil.rmtree(DIRS["models"] / name, ignore_errors=True)


def select(shard, n_shards, **_):
    """Dev-set choice of the Stage-3 base (B2 is two-pass, so it cannot be the single-pass decoding base)."""
    from . import stage2
    have = {f.name.split(".")[0] for f in DIRS["outputs"].glob("gemma_*.jsonl")}
    scores = {s: stage2.dev_grade(s) for s in ("gemma_A", "gemma_B1", "gemma_B3", "gemma_sft") if s in have}
    stage2.release_dev_router()
    json_dump(scores, DIRS["outputs"] / "stage2_dev_scores.json")
    cands = {"gemma_B1": ("B1", None), "gemma_B3": ("B3", None), "gemma_sft": ("A", str(ADAPTER))}
    cands = {s: c for s, c in cands.items() if scores.get(s, {}).get("n")}
    best = max(cands, key=lambda s: (scores[s]["score"], scores[s]["rougeL"]))
    json_dump(dict(system=best, model_key="gemma", cond=cands[best][0], adapter=cands[best][1]), SELECTED)
    print(json.dumps(scores, indent=1), "\nStage-3 base:", best, flush=True)


def pick(shard, n_shards, **_):
    from .stage3 import pick_params
    print(pick_params()[0], flush=True)


def ppl(shard, n_shards, **_):
    from .stage3 import ppl_job, selected
    ppl_job(shard, n_shards, systems=[selected()["system"], "dec_static_cad", "dec_adacad", "dec_dola", "dec_anchor_cad_dola"])


def lookback(shard, n_shards, **_):
    from .stage3 import lookback_job, selected
    lookback_job(shard, n_shards, systems=[selected()["system"], "dec_anchor_cad_dola"])


def ablation(shard, n_shards, method, **_):
    from .stage3 import ablation_uids, decode_job
    decode_job(shard, n_shards, method=method, system=f"dec_{method}", uids=ablation_uids())


def latency(shard, n_shards, **_):
    from .stage2 import latency_probe
    from .stage3 import selected
    sel = selected()
    print(latency_probe("gemma_A", "gemma", "A"), flush=True)
    print(latency_probe(sel["system"], "gemma", sel["cond"], sel["adapter"]), flush=True)


def materialize(shard, n_shards, **_):
    from .stage4 import materialize as run
    print(run(), "final summaries", flush=True)


def bertscore(shard, n_shards, **_):
    from .evaluate import bertscore_job
    bertscore_job(shard, n_shards, systems=[s for s in EVAL if list(DIRS["outputs"].glob(f"{s}.*jsonl"))])


def final(shard, n_shards, **_):
    """CPU only: sheets, label-free H3, Lookback detector, tests, tables, figures."""
    import traceback
    from . import analysis, human, stage1
    for name, f in (("annotation sheets", human.export_sheets), ("H3 proxy and Lookback", lambda: human.run(gpu=False)),
                    ("analysis", analysis.run_all), ("data figures", stage1.figures)):
        try:
            f()
            print("done:", name, flush=True)
        except Exception:
            print("FAILED:", name, flush=True)
            traceback.print_exc()


# --------------------------------------------------------------------------- the plan
P = "factindic.pipeline."
EVAL = ["gemma_A", "gemma_B1", "gemma_B3", "gemma_sft", "dec_anchor_cad_dola", "dec_static_cad", "dec_adacad", "dec_dola",
        "indicbartss", "mt5", "llama_A", "gemma_B2", "cove", "drop_only", "factindic"]
DEV = dict(sets=["dev"], variants=["native", "roman"])


@dataclass
class Job:
    name: str
    func: str
    kw: dict = field(default_factory=dict)
    shards: int = 1
    after: tuple = ()            # finished, failed or dropped: order only
    prio: int = 50               # lower runs first
    optional: bool = False       # dropped at soft_h
    core: bool = False           # a failure in the smoke test stops the run
    cpu: bool = False


def plan(smoke):
    g = lambda c: dict(system=f"gemma_{c}", model_key="gemma", cond=c)
    gen, dec = P + "stage2:llm_gen_job", P + "stage3:decode_job"
    J = [Job("weights", P + "stage1:weights_job", shards=2, prio=0),
         Job("refs", P + "evaluate:ref_job", shards=2, prio=20, core=True),
         Job("sft", P + "auto:sft", after=("weights",), prio=1),
         Job("sft_dev", P + "auto:sft_gen", DEV, after=("sft",), prio=2),
         Job("dev_A", gen, dict(g("A"), **DEV), prio=3), Job("dev_B1", gen, dict(g("B1"), **DEV), prio=3, core=True),
         Job("dev_B3", gen, dict(g("B3"), **DEV), prio=3),
         Job("select", P + "auto:select", after=("dev_A", "dev_B1", "dev_B3", "sft_dev"), prio=4),
         Job("tune", P + "stage3:tune_generate_job", shards=2, after=("select",), prio=5),
         Job("pick", P + "auto:pick", after=("tune",), prio=6),
         Job("anchor_cad_dola", dec, dict(method="anchor_cad_dola", system="dec_anchor_cad_dola", sets=["testbed", "natural_cm"]),
             shards=2, after=("pick",), prio=7, core=True),
         Job("samples", P + "stage3:samples_job", shards=2, after=("select",), prio=8, core=True),
         Job("gen_B1", gen, g("B1"), shards=2, prio=10, core=True), Job("gen_A", gen, g("A"), shards=2, prio=11),
         Job("gen_sft", P + "auto:sft_gen", shards=2, after=("sft",), prio=12), Job("gen_B3", gen, g("B3"), shards=2, prio=13),
         Job("stage4", P + "stage4:verify_job", shards=2, after=("anchor_cad_dola", "samples", "gen_B1", "gen_B3", "gen_sft"),
             prio=9, core=True),
         Job("materialize", P + "auto:materialize", after=("stage4",), prio=9, core=True, cpu=True),
         Job("static_cad", P + "auto:ablation", dict(method="static_cad"), after=("pick",), prio=14),
         Job("indicbartss", P + "auto:s2s", dict(name="indicbartss"), prio=15),
         Job("dola", P + "auto:ablation", dict(method="dola"), after=("pick",), prio=16),
         Job("adacad", P + "auto:ablation", dict(method="adacad"), after=("pick",), prio=17),
         Job("llama", gen, dict(system="llama_A", model_key="llama", cond="A"), shards=2, prio=24, optional=True),
         Job("mt5", P + "auto:s2s", dict(name="mt5"), prio=25, optional=True),
         Job("cove", P + "stage3:cove_job", after=("select", "gen_B1", "gen_B3", "gen_sft"), prio=26, optional=True),
         Job("gen_B2", gen, g("B2"), shards=2, prio=27, optional=True),
         Job("ppl", P + "auto:ppl", after=("anchor_cad_dola", "static_cad", "dola", "adacad", "gen_B1", "gen_B3", "gen_sft"),
             prio=28, optional=True),
         Job("lookback", P + "auto:lookback", after=("anchor_cad_dola", "gen_B1", "gen_B3", "gen_sft"), prio=29, optional=True),
         Job("latency", P + "auto:latency", after=("select", "sft"), prio=30, optional=True)]
    src = dict(gemma_A="gen_A", gemma_B1="gen_B1", gemma_B3="gen_B3", gemma_B2="gen_B2", gemma_sft="gen_sft", llama_A="llama",
               indicbartss="indicbartss", mt5="mt5", cove="cove", dec_anchor_cad_dola="anchor_cad_dola",
               dec_static_cad="static_cad", dec_adacad="adacad", dec_dola="dola", drop_only="materialize", factindic="materialize")
    for s in EVAL:
        J.append(Job(f"eval_{s}", P + "evaluate:eval_job", dict(systems=[s]), after=("refs", src[s]),
                     prio=9 if s == "factindic" else 21, core=s == "factindic"))
    J.append(Job("bertscore", P + "auto:bertscore", after=tuple(f"eval_{s}" for s in EVAL), prio=40))
    if smoke:                    # one of each kind; the others differ only in prompt or method name
        drop = {"dev_A", "dev_B3", "gen_A", "gen_B3", "static_cad", "dola", "eval_gemma_A", "eval_gemma_B3", "eval_dec_static_cad",
                "eval_dec_dola", "eval_gemma_sft", "eval_dec_adacad", "eval_indicbartss", "eval_mt5", "eval_llama_A",
                "eval_gemma_B2", "eval_cove", "eval_drop_only", "latency"}
        J = [j for j in J if j.name not in drop]
        for j in J:
            j.shards = 1
    return J


# --------------------------------------------------------------------------- the runner
def _launch(job, shard, n, gpu, root, cfg):
    logs = Path(root) / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    kwf = logs / f"auto_{job.name}.json"
    kwf.write_text(json.dumps(job.kw))
    log = logs / f"auto_{job.name}.{shard}.log"
    env = dict(os.environ, FACTINDIC_ROOT=str(root), FACTINDIC_CFG=json.dumps(cfg), CUDA_VISIBLE_DEVICES="" if gpu is None else str(gpu),
               PYTORCH_ALLOC_CONF="expandable_segments:True", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false", PYTHONUTF8="1",
               PYTHONPATH=str(Path(__file__).resolve().parents[2]) + os.pathsep + os.environ.get("PYTHONPATH", ""))
    p = subprocess.Popen([sys.executable, "-u", "-m", "factindic.parallel", job.func, str(kwf), str(shard), str(n)], env=env,
                         stdout=open(log, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    return p, log


def _kill(p):
    import signal
    if os.name == "nt":
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(p.pid)], capture_output=True)
        return p.wait()
    try:
        os.killpg(p.pid, signal.SIGKILL)         # with its IndicTrans2 helper process
    except ProcessLookupError:
        pass
    p.wait()


def _tail(log, n):
    return "\n".join(log.read_text(errors="ignore").replace("\r", "\n").strip().splitlines()[-n:])


def run_plan(jobs, root, cfg, t0, n_gpu, soft_h=None, hard_h=None, timeout_min=150, retries=1, label=""):
    """Returns {job: "ok" | "failed" | "dropped"}. t0: start of the whole session, for soft_h and hard_h."""
    names = {j.name for j in jobs}
    todo = {(j.name, s): j for j in jobs for s in range(j.shards)}
    left = {j.name: j.shards for j in jobs}
    status, tries, running, free = {}, {}, {}, list(range(n_gpu))
    last = time.time()
    hours = lambda: (time.time() - t0) / 3600

    def finish(name, state):
        left[name] -= 1
        status[name] = "failed" if "failed" in (status.get(name), state) else ("dropped" if "dropped" in (status.get(name), state) else "ok")
        if left[name] == 0:
            print(f"[{hours():.2f} h] {label}{name}: {status[name]}", flush=True)

    while todo or running:
        stop = hard_h is not None and hours() >= hard_h
        soft = soft_h is not None and hours() >= soft_h
        for key in list(todo):
            if stop or (soft and todo[key].optional):
                del todo[key]
                finish(key[0], "dropped")
        for key, (p, log, job, gpu, start) in list(running.items()):
            rc = p.poll()
            late = stop or (soft and job.optional and hours() >= soft_h + 0.4) or time.time() - start > 60 * timeout_min
            if rc is None and not late:
                continue
            if rc is None:
                _kill(p)
            del running[key]
            if gpu is not None:
                free.append(gpu)
            if rc == 0:
                finish(key[0], "ok")
            elif rc is not None and tries.get(key, 0) < retries:
                tries[key] = tries.get(key, 0) + 1
                todo[key] = job
                print(f"[{hours():.2f} h] {label}{key[0]}#{key[1]} failed, retrying: {_tail(log, 1)[:200]}", flush=True)
            elif rc is None:
                print(f"[{hours():.2f} h] {label}{key[0]}#{key[1]} stopped (time)", flush=True)
                finish(key[0], "dropped")
            else:
                print(f"----- {label}{key[0]}#{key[1]} FAILED -----\n{_tail(log, 25)}\n-----", flush=True)
                finish(key[0], "failed")
        ready = [k for k, j in todo.items() if all(left[a] == 0 for a in j.after if a in names)]
        for key in sorted(ready, key=lambda k: (todo[k].prio, k[1])):
            job = todo[key]
            if not job.cpu and not free:
                continue
            gpu = None if job.cpu else free.pop(0)
            p, log = _launch(job, key[1], job.shards, gpu, root, cfg)
            running[key] = (p, log, job, gpu, time.time())
            del todo[key]
        if time.time() - last > 600:
            last = time.time()
            done = sum(v == 0 for v in left.values())
            print(f"[{hours():.2f} h] {label}{done}/{len(left)} jobs finished; running: " +
                  " | ".join(f"{k[0]}#{k[1]}: {_tail(log, 1)[:70]}" for k, (_, log, *_r) in running.items()), flush=True)
        time.sleep(5)
    json_dump(status, Path(root) / "logs" / f"auto_status{'_smoke' if label else ''}.json")
    return status


SMOKE_CFG = dict(n_use=dict(testbed=1, dev=1, natural_cm=1), n_sft_per_lang=8, n_decoding_ablation_per_lang=1, n_cove=2, n_tune=1,
                 n_s2s_train=32, s2s_epochs=1, n_score_per_lang=2, bootstrap=50, n_annot_summaries=6, n_annot_dev=2)


def smoke(t0, n_gpu):
    """Every kind of job on one article per language, in a scratch folder."""
    root = SMOKE / "factindic"
    shutil.rmtree(root.parent, ignore_errors=True)
    for d in ("data", "cache"):
        shutil.copytree(DIRS[d], root / d)
    for d in DIRS:
        (root / d).mkdir(parents=True, exist_ok=True)
    keys = json_load(root / "data" / "score_subset.json")
    old = {w["key"] for w in jl_read_glob(root / "data" / "weights.jsonl")}
    json_dump([k for k in keys if k in old] + [k for k in keys if k not in old][:4], root / "data" / "score_subset.json")
    h1 = ROOT.parent / "factindic_h1"
    if h1.exists():
        shutil.copytree(h1, root.parent / "factindic_h1", ignore=shutil.ignore_patterns("*sft*", "mt_cache*"), dirs_exist_ok=True)
    jobs = plan(smoke=True) + [Job("final", P + "auto:final", after=("bertscore",), cpu=True)]
    st = run_plan(jobs, root, SMOKE_CFG, t0, n_gpu, timeout_min=25 if IS_KAGGLE else 60, retries=0, label="smoke ")
    bad = [k for k, v in st.items() if v != "ok"]
    for f in sorted((root / "outputs").glob("*.jsonl")):     # what the systems actually wrote
        rows = [r for r in jl_read_glob(f) if "summary" in r]
        if rows:
            print(f"  {f.name}: {sum(not r['summary'].strip() for r in rows)} of {len(rows)} empty :: {rows[-1]['summary'][:110]}")
    print("SMOKE TEST:", "all jobs passed" if not bad else f"problems in {bad}", flush=True)
    print(_tail(root / "logs" / "auto_final.0.log", 12))
    core = [j.name for j in jobs if j.core and st.get(j.name) != "ok"]
    if set(core) & {"dev_B1", "gen_B1", "anchor_cad_dola", "samples"}:     # later steps can be rerun on finished outputs
        raise RuntimeError(f"smoke test: the generator failed ({core}). Send the messages above before the long run.")
    return st


START = time.time()


def _gpus():
    import torch
    return max(1, torch.cuda.device_count())


def part1_data():
    """Notebook 01: sanitation, testbed, dev, natural code-mixed set, anchors, training pool, score subset."""
    st = run_plan([Job("prep", P + "auto:prep", core=True)], ROOT, {}, START, _gpus(), timeout_min=100 if IS_KAGGLE else 240)
    keep = ("ILSUM san", "testbed/", "dev/", "natural_cm/", "pairs to score", "scores reused", "roman_vocab")
    log = (DIRS["logs"] / "auto_prep.0.log").read_text(errors="ignore").replace("\r", "\n")
    print("\n".join(x for x in log.splitlines() if any(k in x for k in keep)))
    if st["prep"] != "ok":
        raise RuntimeError("data preparation failed")
    for name in ("testbed", "dev", "natural_cm", "train_pool"):
        n = {}
        for r in jl_read_glob(DIRS["data"] / f"{name}.jsonl"):
            n[r["lang"]] = n.get(r["lang"], 0) + 1
        print(f"{name}: {n}")


def part2_smoke():
    smoke(START, _gpus())
    shutil.rmtree(SMOKE, ignore_errors=True)


def part3_full(soft_h=3.25, hard_h=4.5):
    """Notebook 01 scores + 02-06 + analysis. Hours count from the import of this module."""
    t0, n = START, _gpus()
    st = run_plan(plan(smoke=False), ROOT, {}, t0, n, soft_h, hard_h)
    f = run_plan([Job("final", P + "auto:final", cpu=True)], ROOT, {}, t0, n, timeout_min=40)
    print(_tail(DIRS["logs"] / "auto_final.0.log", 12))
    free_gpu()
    return dict(st, **f)

# Run

In [ ]:
import glob, os, shutil, sys
sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

# factindic_inputs dataset: results of the finished H1 run, and the training-pool ids with their faithfulness scores
for f in glob.glob("/kaggle/input/**/pool_seed.txt", recursive=True)[:1]:
    shutil.copy(f, f"{SRC}/factindic/pool_seed.txt"); print("training-pool scores:", f)
for d in glob.glob("/kaggle/input/**/h1_results", recursive=True)[:1]:
    shutil.copytree(d, f"{SRC}/factindic/h1_results", dirs_exist_ok=True); print("H1 results:", d)
assert os.path.exists(f"{SRC}/factindic/pool_seed.txt"), "Add the factindic_inputs dataset (Add Input), then run again."

from factindic.config import setup, CFG, DIRS
info = setup()

In [ ]:
# Preflight: versions, Hugging Face access, translator. Stops here if anything is missing.
from factindic import preflight
preflight.check(mt=True, required=("it2", "gemma", "indicner"), optional=("llama", "indicbartss", "indicbert"), need_data=False)

## Part 1. Data: sanitation, testbed, anchors, training pool (notebook 01)

In [ ]:
from factindic.pipeline import auto
auto.part1_data()

## Part 2. Smoke test of every step

In [ ]:
auto.part2_smoke()

## Part 3. Faithfulness weights, baselines, prompting, SFT, decoding, verification, evaluation (01–06)

In [ ]:
# optional jobs are dropped after soft_h hours, everything stops after hard_h hours, then the analysis runs
status = auto.part3_full(soft_h=3.25, hard_h=4.5)
status

## Part 4. Results and figures

In [ ]:
import pandas as pd
f = DIRS['eval'] / 'main_table.csv'
if f.exists():
    t = pd.read_csv(f)
    display(t.pivot_table(index='label', columns='variant', values=['P_fact', 'F1_fact']).round(3))

In [ ]:
from IPython.display import Image, Markdown, display
for p in sorted(DIRS["figures"].glob("*.png")):
    display(Markdown(f"**{p.stem}**")); display(Image(str(p), width=760))

In [ ]:
# Package this notebook's outputs; attach this notebook's output as an input to the next notebook
!cd /kaggle/working 2>/dev/null && zip -qr factindic_outputs.zip factindic factindic_h1 -x "*/models/*/ckpt/*" || true
print("done:", DIRS["outputs"].parent)